In [2]:
import pandas as pd
import PyPDF2
import os
import seaborn as sns
import openai
import matplotlib.pyplot as plt
import numpy as np
import re
from textstat import flesch_reading_ease
import requests
import json
import pandas as pd
import prettytable
import zipfile
import difflib
from difflib import get_close_matches

In [3]:
### DataFrame and Graphs Settings
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colum', None)
pd.set_option("display.max_colwidth", None)

sns.set(font_scale=1.5)
sns.set(style='white', font_scale=1.5)

In [3]:
df= pd.read_table('/Users/mgor/Dropbox/Second YPP/Data/Data Aux/Main_2012-03.zip', encoding='latin')


/opt/anaconda3/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3331: DtypeWarning: Columns (1) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [ ]:
df['BestFitMSA']=pd.to_numeric(df['BestFitMSA'], errors='coerce').fillna(-999).astype(int)
df['FIPSCounty']=pd.to_numeric(df['FIPSCounty'], errors='coerce').fillna(-999).astype(int)
df['FIPS']=pd.to_numeric(df['FIPS'], errors='coerce').fillna(-999).astype(int)


In [ ]:
df['Year'] = df['JobDate'].str[:4].astype(int)  
df['Month'] = df['JobDate'].str[5:7].astype(int) 

In [ ]:
df['OccFam'] = pd.to_numeric(df['OccFam'], errors='coerce').fillna(-999).astype(int)
df = df.loc[df['OccFamName'] != -999]
df['SOC'] = df['SOC'].replace('na', '-999')

In [ ]:
#df = df.loc[df['OccFamName'] != 'na']
#df = df.loc[df['FIPSState'] != 'na']
#df = df.loc[df['SOC'] != 'na']
df['SOCName'] = df['SOCName'].str.strip()
df['Sector'] = df.apply(lambda row: str(row['NAICS3'])[:2] if row['NAICS3'] != -999 else str(row['Sector'])[:2], axis=1)
df['Sector'] = pd.to_numeric(df['Sector'], errors='coerce').fillna(-999).astype(int)

In [ ]:
df = df.rename(columns={"Employer": "CanonEmployer"})
df= df[(df['CanonEmployer'].notna()) & (df['CanonEmployer'] != None) & (df['CanonEmployer'] != "None")]
df.head()
df = df.loc[df['CanonEmployer'] != 'na']

In [ ]:
df.loc[df['NAICS6'] == 0, 'NAICS6'] = -999


### Crosswalks

In [483]:
df_occ=pd.read_excel('/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/Crosswalks/soc_2010_to_2018_crosswalk.xlsx', skiprows=8) 
df_occ['SOC_change'] = np.where(df_occ['2010 SOC Code'] == df_occ['2018 SOC Code'], 'No change', 'Changed')
df_occ['SOC_name'] = np.where(df_occ['2010 SOC Title'] == df_occ['2018 SOC Title'], 'No change', 'Changed')

cross_occ= df_occ[df_occ['SOC_change'] == 'Changed'][['2010 SOC Code', '2018 SOC Code']]
cross_occname= df_occ[df_occ['SOC_name'] == 'Changed'][['2010 SOC Title', '2018 SOC Title']]


In [484]:

inverse_mapping = cross_occ[['2018 SOC Code', '2010 SOC Code']].dropna().rename(
    columns={'2018 SOC Code': 'SOC', '2010 SOC Code': 'Mapped SOC'})

cross_occ=cross_occ.rename(
    columns={'2018 SOC Code': 'Mapped SOC', '2010 SOC Code': 'SOC'})

cross_occ=pd.concat([cross_occ, inverse_mapping], ignore_index=True).drop_duplicates()


In [ ]:
#df_naics=pd.read_excel('/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/Crosswalks/2012_to_2017_NAICS.xlsx', skiprows=2)--- No changes in NAICS 3 
df_naics=pd.read_excel('/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/Crosswalks/2007_to_2012_NAICS.xls', skiprows=2) 


In [ ]:

df_naics['2012NAICS3'] = df_naics['2012 NAICS Code'].astype(str).str[:3]
df_naics['2007NAICS3'] = df_naics['2007 NAICS Code'].astype(str).str[:3]

In [ ]:
df_naics['NAICS_change'] = np.where(df_naics['2012NAICS3'] == df_naics['2007NAICS3'], 'No change', 'Changed')



In [ ]:
# Finding unique values in column2 that are not in column1
unique_in_col2 = df_naics['2017NAICS3'][~df_naics['2017NAICS3'].isin(df_naics['2022NAICS3'])]

# Creating a DataFrame for better visualization
result_df = pd.DataFrame({'Unique in Column 1': unique_in_col1, 'Unique in Column 2': unique_in_col2})


In [ ]:

df_naics['NAICS_change'] = np.where(df_naics['2012 NAICS Code'] == df_naics['2017 NAICS Code'], 'No change', 'Changed')
df_naics2['NAICS_change'] = np.where(df_naics2['2022 NAICS Code'] == df_naics2['2017 NAICS Code'], 'No change', 'Changed')

cross_naics= df_naics[df_naics['NAICS_change'] == 'Changed'][['2012 NAICS Code', '2017 NAICS Code']]
cross_naics = cross_naics.rename(columns={'2012 NAICS Code': 'NAICS A', '2017 NAICS Code': 'NAICS B'})
cross_naics2= df_naics2[df_naics2['NAICS_change'] == 'Changed'][['2022 NAICS Code', '2017 NAICS Code']]
cross_naics2 = cross_naics2.rename(columns={'2022 NAICS Code': 'NAICS A', '2017 NAICS Code': 'NAICS B'})



In [ ]:
cross_naics = pd.concat([cross_naics, cross_naics2], axis=0)
del df_naics, df_naics2, df_occ

In [ ]:
cross_naics.dtypes

## JOLTS

In [370]:
# Define the JOLTS mapping
JOLTS_mapping = {
    "HIR": "Hires",
    "JOR": "Job openings",
    "LDR": "Layoffs and discharges",
    "OSR": "Other separations",
    "QUR": "Quits",
    "TSR": "Total separations",
    "UOR": "Unemployed persons per job opening ratio",
}

# Define the month mapping
month_mapping = {'Jan': '01', 'Feb': '02', 'Mar': '03', 'Apr': '04', 
                 'May': '05', 'Jun': '06', 'Jul': '07', 'Aug': '08', 
                 'Sep': '09', 'Oct': '10', 'Nov': '11', 'Dec': '12'}

### National

*** Currently keeping the code for reference but not using in final merge with Burning Glass

In [ ]:
df_aux=pd.read_excel('/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/JOLTS/National.xlsx', skiprows=3) 

In [ ]:
# Map Series ID to JOLTS names
df_aux['Name'] = df_aux['Series ID'].str[-3:].map(JOLTS_mapping)
df_aux['NAICS6'] = df_aux['Series ID'].str[3:9]

In [ ]:
# Melt the DataFrame
df_aux = pd.melt(
            df_aux,
            id_vars=["Name", "NAICS6"],
            var_name="Date",
            value_name="value")

In [ ]:
# Pivot the DataFrame
df_aux = df_aux.pivot_table(
            index=["NAICS6", "Date"],
            columns="Name",
            values="value",
            aggfunc="first"
        ).reset_index()

df_aux.columns.name = None  # Remove columns name
# Extract Year and Month
df_aux['Year'] = df_aux['Date'].str.split('\n').str[1]
df_aux['Month'] = df_aux['Date'].str.split('\n').str[0].map(month_mapping)
df_aux = df_aux.drop('Date', axis=1)

In [ ]:
df_aux.NAICS6.unique()

In [ ]:
# Sort the DataFrame
df_aux = df_aux.sort_values(by=["NAICS6", "Year", "Month"])

# Keep only rows where NAICS6 is '000000'
df_aux = df_aux[df_aux['NAICS6'] == '000000']

# Drop the NAICS6 column
df_aux = df_aux.drop(columns=['NAICS6'])

In [ ]:
# Convert columns to numeric
columns_to_convert = ['Year', 'Month']
for col in columns_to_convert:
    df_aux[col] = pd.to_numeric(df_aux[col], errors='coerce').fillna(-999).astype(int)

In [ ]:
df_aux.rename(columns={'Hires':'hires', 'Job openings':'jobOpenings', 'Layoffs and discharges':'layoffs',
       'Other separations':'otherSeparations', 'Quits':'quits', 'Total separations':'totalSeparations',
       'Unemployed persons per job opening ratio':'unemployedPerOpening'}, inplace=True)

In [ ]:
columns_to_rename = df_aux.columns.difference(['Year', 'Month'])
df_aux = df_aux.rename(columns={col: f"{col}_nt" for col in columns_to_rename})

In [ ]:
df_aux.to_excel('/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/JOLTS/Nationalb.xlsx', index=False)

### State

In [ ]:
# Directory containing the files
directory = '/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/JOLTS/'

# Collect all DataFrames
dataframes = [
    pd.read_excel(os.path.join(directory, file_name), skiprows=3)
    for file_name in os.listdir(directory)
    if file_name.startswith('state') and file_name.endswith('.xlsx')
]

# Concatenate all DataFrames into a single DataFrame
if dataframes:
    df_aux = pd.concat(dataframes, ignore_index=True)
    print("All files appended successfully.")
else:
    df_aux = pd.DataFrame()

In [373]:
df_aux=pd.read_excel('/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/JOLTS/state_pctchange.xlsx', skiprows=3) 

df_aux.drop(columns=['View Description'], inplace=True, errors='ignore')

/opt/anaconda3/lib/python3.7/site-packages/openpyxl/styles/stylesheet.py:214: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [375]:
# Map Series ID to JOLTS names
df_aux['Name'] = df_aux['Series ID'].str[-3:].map(JOLTS_mapping)
df_aux['FIPSState'] = df_aux['Series ID'].str[9:11]
df_aux = df_aux.drop('Series ID', axis=1)

# Melt the DataFrame
df_aux = pd.melt(
            df_aux,
            id_vars=["Name", "FIPSState"],
            var_name="Date",
            value_name="value")

# Pivot the DataFrame
df_aux = df_aux.pivot_table(
            index=["FIPSState", "Date"],
            columns="Name",
            values="value",
            aggfunc="first").reset_index()

df_aux.columns.name = None  # Remove columns name

# Extract Year and Month
df_aux['Year'] = df_aux['Date'].str.split('\n').str[1]
df_aux['Month'] = df_aux['Date'].str.split('\n').str[0].map(month_mapping)
df_aux = df_aux.drop('Date', axis=1)

# Sort the DataFrame
df_aux = df_aux.sort_values(by=["FIPSState", "Year", "Month"])

# Convert columns to numeric
columns_to_convert = ['FIPSState', 'Year', 'Month']
for col in columns_to_convert:
    df_aux[col] = pd.to_numeric(df_aux[col], errors='coerce').fillna(-999).astype(int)

df_aux.rename(columns={'Hires':'hires', 'Job openings':'jobOpenings', 'Layoffs and discharges':'layoffs',
       'Other separations':'otherSeparations', 'Quits':'quits', 'Total separations':'totalSeparations',
       'Unemployed persons per job opening ratio':'unemployedPerOpening'}, inplace=True)

columns_to_rename = df_aux.columns.difference(['FIPSState','Year', 'Month'])
df_aux = df_aux.rename(columns={col: f"{col}_st" for col in columns_to_rename})

In [377]:
df_aux.to_excel('/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/JOLTS/State.xlsx', index=False)

## LAUS

In [378]:
# Define the LAUS mapping
LAUS_mapping = {
    "06": "laborForce",
    "05": "employment",
    "04": "unemployment",
    "03": "unemploymentRate"
}

### State

In [ ]:
df_aux=pd.read_excel('/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/LAUS/State/SeriesReport-20250112152909_91d9a2.xlsx', skiprows=3) 

In [ ]:
df_aux['FIPSState'] = df_aux['Series ID'].str[5:7]
df_aux['Name'] = df_aux['Series ID'].str[-2:].map(LAUS_mapping)
df_aux = df_aux.drop('Series ID', axis=1)

# Melt the DataFrame
df_aux = pd.melt(
            df_aux,
            id_vars=["Name", "FIPSState"],
            var_name="Date",
            value_name="value")

# Pivot the DataFrame
df_aux = df_aux.pivot_table(
            index=["FIPSState", "Date"],
            columns="Name",
            values="value",
            aggfunc="first").reset_index()

df_aux.columns.name = None  # Remove columns name

# Extract Year and Month
df_aux['Year'] = df_aux['Date'].str.split('\n').str[1]
df_aux['Month'] = df_aux['Date'].str.split('\n').str[0].map(month_mapping)
df_aux = df_aux.drop('Date', axis=1)

# Sort the DataFrame
df_aux = df_aux.sort_values(by=["FIPSState", "Year", "Month"])

# Convert columns to numeric
columns_to_convert = ['FIPSState', 'Year', 'Month']
for col in columns_to_convert:
    df_aux[col] = pd.to_numeric(df_aux[col], errors='coerce').fillna(-999).astype(int)
    
df_aux.rename(
    columns=lambda col: col + '_st' if col not in columns_to_convert else col,
    inplace=True
)

In [ ]:
df_aux.head()

In [ ]:
df_aux.to_csv('/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/LAUS/State/LAUS_state.csv', index=False)


### Metro Area

In [440]:
df_aux=pd.read_excel('/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/LAUS/ssamatab1.xlsx', skiprows=2) 

In [441]:
df_aux = df_aux.drop(index=df_aux.index[0]).reset_index(drop=True)
df_aux.drop(columns=['LAUS Code','Area'], errors='ignore', inplace=True)

In [442]:
df_aux.rename(columns={'ST FIPS Code':'FIPSState', 'Area FIPS Code':'BestFitMSA', "Civilian Labor Force": "laborForce",
    "Employment": "employment",
    "Unemployment": "unemployment",
    "Unemployment Rate": "unemploymentRate"}, inplace=True)


In [443]:
columns_to_convert = ['FIPSState', 'BestFitMSA', 'Year', 'Month']
for col in columns_to_convert:
    df_aux[col] = pd.to_numeric(df_aux[col], errors='coerce').fillna(-999).astype(int)

df_aux.rename(
    columns=lambda col: col + '_msa' if col not in columns_to_convert else col,
    inplace=True
)

# Drop rows where any column has -999
df_aux = df_aux[~(df_aux == -999).any(axis=1)]

# Iterate over columns not in columns_to_convert and convert them to float
for col in [c for c in df_aux.columns if c not in columns_to_convert]:
    df_aux[col] = pd.to_numeric(df_aux[col], errors='coerce').astype(float)


In [444]:
df_aux.dtypes

FIPSState                 int64
BestFitMSA                int64
Year                      int64
Month                     int64
laborForce_msa          float64
employment_msa          float64
unemployment_msa        float64
unemploymentRate_msa    float64
dtype: object

In [445]:
df_MSA=df_aux

In [446]:
df_MSA.head()

,FIPSState,BestFitMSA,Year,Month,laborForce_msa,employment_msa,unemployment_msa,unemploymentRate_msa
0,1,11500,1990,1,51709.0,48084.0,3625.0,7.0
1,1,12220,1990,1,44426.0,41104.0,3322.0,7.5
2,1,13820,1990,1,460283.0,435924.0,24359.0,5.3
3,1,19300,1990,1,46320.0,43881.0,2439.0,5.3
4,1,19460,1990,1,65563.0,61215.0,4348.0,6.6


### Micro Area

In [447]:
# Directory containing the files
directory = '/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/LAUS/'

# Collect all DataFrames
dataframes = [
    pd.read_excel(os.path.join(directory, file_name), skiprows=3)
    for file_name in os.listdir(directory)
    if file_name.startswith('Micro') and file_name.endswith('.xlsx')
]

# Concatenate all DataFrames into a single DataFrame
if dataframes:
    df_aux = pd.concat(dataframes, ignore_index=True)
    print("All files appended successfully.")
else:
    df_aux = pd.DataFrame()

/opt/anaconda3/lib/python3.7/site-packages/openpyxl/styles/stylesheet.py:214: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


All files appended successfully.


In [450]:
df_aux.head()

,FIPSState,BestFitMSA,employment_msa,laborForce_msa,unemployment_msa,unemploymentRate_msa,Year,Month
52,1,10700,35555.0,39977.0,4422.0,11.1,2010,1
39,1,10700,35825.0,40130.0,4305.0,10.7,2010,2
91,1,10700,36163.0,40264.0,4101.0,10.2,2010,3
0,1,10700,37562.0,41404.0,3842.0,9.3,2010,4
104,1,10700,37870.0,41623.0,3753.0,9.0,2010,5


In [449]:
df_aux['FIPSState'] = df_aux['Series ID'].str[5:7]
df_aux['BestFitMSA'] = df_aux['Series ID'].str[7:12]
df_aux['Name'] = df_aux['Series ID'].str[-2:].map(LAUS_mapping)
df_aux = df_aux.drop('Series ID', axis=1)

# Melt the DataFrame
df_aux = pd.melt(
            df_aux,
            id_vars=["Name", "FIPSState","BestFitMSA"],
            var_name="Date",
            value_name="value")

# Pivot the DataFrame
df_aux = df_aux.pivot_table(
            index=["FIPSState","BestFitMSA", "Date"],
            columns="Name",
            values="value",
            aggfunc="first").reset_index()

df_aux.columns.name = None  # Remove columns name

# Extract Year and Month
df_aux['Year'] = df_aux['Date'].str.split('\n').str[1]
df_aux['Month'] = df_aux['Date'].str.split('\n').str[0].map(month_mapping)
df_aux = df_aux.drop('Date', axis=1)

# Sort the DataFrame
df_aux = df_aux.sort_values(by=["FIPSState","BestFitMSA", "Year", "Month"])

# Convert columns to numeric
columns_to_convert = ['FIPSState',"BestFitMSA", 'Year', 'Month']
for col in columns_to_convert:
    df_aux[col] = pd.to_numeric(df_aux[col], errors='coerce').fillna(-999).astype(int)
    
df_aux.rename(
    columns=lambda col: col + '_msa' if col not in columns_to_convert else col,
    inplace=True
)

In [451]:
df_LAUS=pd.concat([df_MSA, df_aux], ignore_index=True)

In [455]:
df_LAUS.to_csv(directory + 'LAUS_msa.csv', index=False)


In [ ]:
df_county=pd.read_csv(directory + '/LAUS_ct.csv', delimiter=',') 


### County 

In [456]:
# Read files

# Initialize an empty DataFrame
df_county = pd.DataFrame()

# Iterate over all files in the directory
for filename in os.listdir(directory):
    # Check if the file starts with "Series" and ends with ".xlsx"
    if filename.startswith("Series") and filename.endswith(".xlsx"):
        # Construct the full file path
        file_path = os.path.join(directory, filename)
        
        # Read the Excel file
        df_aux = pd.read_excel(file_path, skiprows=3)
        df_aux['FIPSState'] = df_aux['Series ID'].str[5:7]
        df_aux['FIPSCounty'] = df_aux['Series ID'].str[7:10]
        df_aux['FIPS'] = df_aux['Series ID'].str[5:10]
        df_aux['Name'] = df_aux['Series ID'].str[-2:].map(LAUS_mapping)
        df_aux = df_aux.drop('Series ID', axis=1)

        # Melt the DataFrame
        df_aux = pd.melt(
                    df_aux,
                    id_vars=["Name", "FIPSState","FIPSCounty", "FIPS"],
                    var_name="Date",
                    value_name="value")

        # Pivot the DataFrame
        df_aux = df_aux.pivot_table(
                    index=["FIPSState","FIPSCounty", "Date", "FIPS"],
                    columns="Name",
                    values="value",
                    aggfunc="first").reset_index()

        df_aux.columns.name = None  # Remove columns name

        # Extract Year and Month
        df_aux['Year'] = df_aux['Date'].str.split('\n').str[1]
        df_aux['Month'] = df_aux['Date'].str.split('\n').str[0].map(month_mapping)
        df_aux = df_aux.drop('Date', axis=1)

        # Sort the DataFrame
        df_aux = df_aux.sort_values(by=["FIPSState","FIPSCounty","FIPS", "Year", "Month"])

        # Convert columns to numeric
        columns_to_convert = ['FIPSState',"FIPSCounty", 'Year', "FIPS" ,'Month']
        for col in columns_to_convert:
            df_aux[col] = pd.to_numeric(df_aux[col], errors='coerce').fillna(-999).astype(int)

        df_aux.rename(
            columns=lambda col: col + '_ct' if col not in columns_to_convert else col,
            inplace=True
        )
        # Concatenate directly with the combined DataFrame
        df_county = pd.concat([df_county, df_aux], ignore_index=True)


/opt/anaconda3/lib/python3.7/site-packages/openpyxl/styles/stylesheet.py:214: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [457]:
df_county = df_county[~(df_county == -999).any(axis=1)]


In [460]:
df_county.dtypes

FIPSState                int64
FIPSCounty               int64
FIPS                     int64
employment_ct          float64
laborForce_ct          float64
unemployment_ct        float64
unemploymentRate_ct    float64
Year                     int64
Month                    int64
dtype: object

In [461]:
df_county.to_csv(directory + 'LAUS_ct.csv', index=False)

## OESM

### Bartik Share

*** Keeping this here just in case but currently not using

In [ ]:
import os
import pandas as pd
import numpy as np

def process_oesm_year(year, folder_base, mapping_ST, cross_occ):
    """
    Reads all MSA .xls files for the given year, merges them, 
    calculates <year>SOCShare, merges with cross_occ, and returns
    a DataFrame with columns:
      [FIPSState, BestFitMSA, SOC, OccFam, <year>SOCShare]
    """
    folder_name = f"oesm{str(year)[2:]}ma"  # e.g. "oesm05ma"
    folder_path = os.path.join(folder_base, folder_name)

    # Gather data from MSA files
    df_year_all_msa = pd.DataFrame()
    if not os.path.isdir(folder_path):
        print(f"⚠️ Folder not found for {year}: {folder_path}")
        return df_year_all_msa  # Return empty DataFrame

    # Read each .xls
    for file_name in os.listdir(folder_path):
        if "MSA" in file_name and file_name.endswith(".xls"):
            file_path = os.path.join(folder_path, file_name)
            df_temp = pd.read_excel(file_path)
            df_year_all_msa = pd.concat([df_year_all_msa, df_temp], ignore_index=True)

    if df_year_all_msa.empty:
        print(f"⚠️ No MSA .xls files found for {year}")
        return df_year_all_msa  # Return empty DataFrame

    # Rename columns
    df_year_all_msa.rename(columns={
        'PRIM_STATE': 'FIPSState',
        'AREA': 'BestFitMSA',
        'AREA_NAME': 'BestFitMSAName',
        'OCC_CODE': 'SOC',
        'OCC_TITLE': 'SOCName',
        'TOT_EMP': 'TOT_EMP'
    }, inplace=True)

    # Map or convert columns
    df_year_all_msa['FIPSState'] = df_year_all_msa['FIPSState'].map(mapping_ST)
    for col in ['FIPSState', 'BestFitMSA', 'TOT_EMP']:
        df_year_all_msa[col] = (pd.to_numeric(df_year_all_msa[col], errors='coerce')
                                       .fillna(-999)
                                       .astype(int))

    # Separate major occupations & total employment for MSA
    df_msa_total = df_year_all_msa[df_year_all_msa['SOC'] == '00-0000'].rename(columns={'TOT_EMP': 'empTotal'})
    df_year_all_msa = df_year_all_msa[df_year_all_msa['SOC'] != '00-0000']  

    # Separate occupation families (SOC ending in '0000')
    df_occ = df_year_all_msa[df_year_all_msa['SOC'].str[-4:] == '0000'].copy()
    df_year_all_msa = df_year_all_msa[df_year_all_msa['SOC'].str[-4:] != '0000']

    df_occ['OccFam'] = pd.to_numeric(df_occ['SOC'].astype(str).str[:2], errors='coerce').fillna(-999).astype(int)
    df_occ = df_occ[df_occ['OccFam'] != -999]
    df_occ['SOC'] = '-999'  # Ensure it's a string

    # Merge to get total employment
    df_year_all_msa = df_year_all_msa.merge(df_msa_total[['BestFitMSA', 'empTotal']], on='BestFitMSA', how='left')
    df_occ = df_occ.merge(df_msa_total[['BestFitMSA', 'empTotal']], on='BestFitMSA', how='left')

    # Compute employment share
    year_str = str(year)[2:]  # e.g. "05"
    share_col = f"{year_str}SOCShare"

    df_year_all_msa[share_col] = df_year_all_msa['TOT_EMP'] / df_year_all_msa['empTotal'].replace(0, np.nan)
    df_occ[share_col] = df_occ['TOT_EMP'] / df_occ['empTotal'].replace(0, np.nan)
    df_year_all_msa[share_col] = df_year_all_msa[share_col].clip(lower=0)

    df_year_all_msa['OccFam'] = pd.to_numeric(df_year_all_msa['SOC'].astype(str).str[:2], errors='coerce').fillna(-999).astype(int)

    # Keep only necessary columns
    df_year_all_msa = df_year_all_msa[['FIPSState', 'BestFitMSA', 'SOC', 'OccFam', share_col]]
    df_occ = df_occ[['FIPSState', 'BestFitMSA', 'SOC', 'OccFam', share_col]]

    # Merge with cross_occ to fix SOC codes
    df_year_all_msa['SOC'] = df_year_all_msa['SOC'].astype(str)
    cross_occ['SOC'] = cross_occ['SOC'].astype(str)

    # Merge to add mapped SOC codes
    df_cross = pd.merge(
        df_year_all_msa,
        cross_occ,
        on='SOC',
        how='inner'
    )

    df_cross=df_cross.drop_duplicates(subset=['Mapped SOC', 'BestFitMSA']).drop(columns=['SOC'], errors='ignore')
    df_cross.rename(columns={'Mapped SOC': 'SOC'}, inplace=True, errors='ignore')
    df_year_all_msa=pd.concat([df_year_all_msa, df_cross], ignore_index=True)
    df_year_all_msa.drop_duplicates(subset=['BestFitMSA','SOC'], inplace=True)

    # Remove duplicates & concatenate occupation family data
    df_year_all_msa.drop_duplicates(subset=['FIPSState', 'BestFitMSA', 'SOC','OccFam'], inplace=True)
    df_year_all_msa = pd.concat([df_year_all_msa, df_occ.drop_duplicates(['FIPSState', 'BestFitMSA', 'SOC', 'OccFam'])], ignore_index=True)

    return df_year_all_msa


In [ ]:
years = range(2005, 2008)
folder_path = '/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/OESM/All/'


df_final = pd.DataFrame()

for i, year in enumerate(years):
    df_year = process_oesm_year(year, folder_path, mapping_ST, cross_occ)
    if df_year.empty:
        # If no data returned, skip
        continue

    if i == 0:
        # First year becomes the starting DataFrame
        df_final = df_year
    else:
        # Merge on the same keys to get a wide format
        df_final = pd.concat([df_final, df_year], ignore_index=True)
      

In [ ]:
df_final['0507SOCShare'] = df_final[['05SOCShare', '06SOCShare', '07SOCShare']].mean(axis=1)
df_final=df_final.groupby(['FIPSState', 'BestFitMSA', 'SOC','OccFam']).mean().reset_index()

In [ ]:
df_final.drop(columns=['05SOCShare', '06SOCShare', '07SOCShare'], errors='ignore', inplace=True)

In [ ]:
df_final2=df_final.groupby(['FIPSState', 'SOC','OccFam']).mean().reset_index()
df_final2.drop_duplicates(subset=['FIPSState', 'SOC','OccFam'], inplace=True)
df_final2['BestFitMSA']=-999
df_final=pd.concat([df_final, df_final2],ignore_index=True)
df_final.to_parquet('/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/OESM/All/CLEAN/SOCShare.parquet', compression='zstd', engine='pyarrow', index=False)


In [ ]:
df_final.head()

### 1. Before 2011

In [462]:
# State
#Leaving this here just in case for future iterations but currently only using at the MSA level

In [555]:
df_aux=pd.read_excel('/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/OESM/All/oesm10st/state_M2009_dl.xls') 

In [556]:
df_aux.rename(columns={'AREA':'FIPSState', 'OCC_CODE':'SOC','OCC_TITLE':'SOCName'}, inplace=True)
df_aux['SOCName'] = df_aux['SOCName'].str.replace('*', '', regex=False)
df_aux['SOCName'] = df_aux['SOCName'].str.replace(r'[^a-zA-Z0-9\s-]', '', regex=True)
df_aux['SOCName'] = df_aux['SOCName'].str.strip()

In [557]:
columns_to_convert = ['FIPSState']
for col in columns_to_convert:
    df_aux[col] = pd.to_numeric(df_aux[col], errors='coerce').fillna(-999).astype(int)


In [558]:
FIPSState = df_aux.groupby('FIPSState', as_index=False).agg({'ST': 'first'})
mapping_ST = dict(zip(FIPSState['ST'],FIPSState['FIPSState']))

In [559]:
df_aux.drop(columns=['ANNUAL','HOURLY','ST','STATE'], errors='ignore', inplace=True)

In [560]:
columns_to_convert = df_aux.columns.difference(['FIPSState', 'SOC', 'SOCName','GROUP'])  # Exclude x, y, z
df_aux[columns_to_convert] = df_aux[columns_to_convert].replace(['*', '#'], np.nan).apply(pd.to_numeric, errors='coerce')


In [561]:
# Rename the columns: lowercase, remove '_', add '_st'
df_aux.rename(columns=lambda x: x.lower()  if x in columns_to_convert else x, inplace=True)


In [562]:
df_aux['BestFitMSA']=-999

In [ ]:
# Merge to add mapped SOC codes
df_cross = pd.merge(
    df_aux,
    cross_occ,
    on='SOC',
    how='inner'
)

df_cross=df_cross.drop_duplicates(subset=['Mapped SOC', 'FIPSState']).drop(columns=['SOC'], errors='ignore')
df_cross.rename(columns={'Mapped SOC': 'SOC'}, inplace=True, errors='ignore')
df_aux=pd.concat([df_aux, df_cross], ignore_index=True)
df_aux.drop_duplicates(subset=['FIPSState','SOC'], inplace=True)

In [ ]:
df_state=df_aux

In [ ]:
# MSA

In [653]:
mapping_ST={'AL': 1,
 'AK': 2,'AZ': 4, 'AR': 5,'CA': 6, 'CO': 8, 'CT': 9,'DE': 10,
 'DC': 11, 'FL': 12,'GA': 13,'HI': 15, 'ID': 16, 'IL': 17,
 'IN': 18,'IA': 19,'KS': 20,'KY': 21,'LA': 22,'ME': 23,'MD': 24,
 'MA': 25,'MI': 26, 'MN': 27,'MS': 28,'MO': 29,'MT': 30, 'NE': 31,
 'NV': 32,'NH': 33,'NJ': 34,'NM': 35, 'NY': 36,'NC': 37,'ND': 38,
 'OH': 39,'OK': 40,'OR': 41,'PA': 42,'RI': 44,'SC': 45,'SD': 46,
 'TN': 47,'TX': 48,'UT': 49,'VT': 50,'VA': 51,'WA': 53,'WV': 54,
 'WI': 55,'WY': 56,'GU': 66,'PR': 72,'VI': 78}

In [672]:
# Specify the folder containing the files
folder_path = '/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/OESM/All/oesm09ma/'

# Initialize an empty list to store all rows
all_data = []

# Loop through files in the folder
for file_name in os.listdir(folder_path):
    # Check if the file name contains 'MSA' and is a .csv file
    if 'MSA' in file_name and file_name.endswith('.xls'):  # Adjust for file format if necessary
        file_path = os.path.join(folder_path, file_name)
        
        # Read the file and append its contents to the list
        all_data.append(pd.read_excel(file_path)) # Use pd.read_excel(file_path) for Excel files

# Combine all the data into a single DataFrame
df_aux = pd.concat(all_data, ignore_index=True)


In [673]:
df_aux.rename(columns={'PRIM_STATE':'FIPSState','AREA':'BestFitMSA','AREA_NAME':'BestFitMSAName','OCC_CODE':'SOC','OCC_TITLE':'SOCName'}, inplace=True)


In [674]:
df_aux['FIPSState'] = df_aux['FIPSState'].map(mapping_ST)


In [675]:
columns_to_convert = ['FIPSState','BestFitMSA']
for col in columns_to_convert:
    df_aux[col] = pd.to_numeric(df_aux[col], errors='coerce').fillna(-999).astype(int)

In [676]:
df_aux.drop(columns=['ANNUAL','HOURLY','ST','STATE', 'PRIM_STATE'], errors='ignore', inplace=True)

In [677]:

columns_to_convert = df_aux.columns.difference(['FIPSState','BestFitMSA', 'BestFitMSAName','SOC', 'SOCName','GROUP'])  # Exclude x, y, z
df_aux[columns_to_convert] = df_aux[columns_to_convert].replace(['*', '#'], np.nan).apply(pd.to_numeric, errors='coerce')
# Rename the columns: lowercase, remove '_', add '_msa'
df_aux.rename(columns=lambda x: x.lower() if x in columns_to_convert else x, inplace=True)


In [678]:
# Merge to add mapped SOC codes
df_cross = pd.merge(
    df_aux,
    cross_occ,
    on='SOC',
    how='inner'
)

df_cross=df_cross.drop_duplicates(subset=['Mapped SOC', 'BestFitMSA']).drop(columns=['SOC'], errors='ignore')
df_cross.rename(columns={'Mapped SOC': 'SOC'}, inplace=True, errors='ignore')
df_aux=pd.concat([df_aux, df_cross], ignore_index=True)
df_aux.drop_duplicates(subset=['BestFitMSA','SOC'], inplace=True)

In [679]:
#df_oesm = pd.concat([df_state, df_aux], ignore_index=True)


In [680]:
df_aux.head()

,FIPSState,BestFitMSA,BestFitMSAName,SOC,SOCName,GROUP,tot_emp,emp_prse,jobs_1000,h_mean,a_mean,mean_prse,h_pct10,h_pct25,h_median,h_pct75,h_pct90,a_pct10,a_pct25,a_median,a_pct75,a_pct90
0,2,11260,"Anchorage, AK",00-0000,All Occupations,total,165640.0,1.1,1000.000,23.91,49730.0,1.3,9.56,12.99,19.64,30.79,42.99,19880.0,27020.0,40860.0,64050.0,89410.0
1,2,11260,"Anchorage, AK",11-0000,Management occupations,major,11910.0,2.5,71.930,41.34,85990.0,1.3,18.97,26.34,37.30,50.90,67.58,39460.0,54790.0,77590.0,105860.0,140560.0
2,2,11260,"Anchorage, AK",11-1011,Chief executives,NaN,640.0,5.7,3.850,71.82,149390.0,3.3,35.59,48.99,64.92,NaN,NaN,74020.0,101900.0,135040.0,NaN,NaN
3,2,11260,"Anchorage, AK",11-1021,General and operations managers,NaN,4930.0,4.7,29.774,39.59,82350.0,2.3,17.50,24.00,35.58,49.65,66.61,36400.0,49910.0,74000.0,103280.0,138540.0
4,2,11260,"Anchorage, AK",11-1031,Legislators,NaN,30.0,2.2,0.192,NaN,42440.0,8.3,NaN,NaN,NaN,NaN,NaN,15450.0,25230.0,32420.0,47970.0,57490.0


In [681]:
df_msa = df_aux[df_aux['SOC'].str[-7:] != '00-0000']


In [682]:
df_msa.head()

,FIPSState,BestFitMSA,BestFitMSAName,SOC,SOCName,GROUP,tot_emp,emp_prse,jobs_1000,h_mean,a_mean,mean_prse,h_pct10,h_pct25,h_median,h_pct75,h_pct90,a_pct10,a_pct25,a_median,a_pct75,a_pct90
1,2,11260,"Anchorage, AK",11-0000,Management occupations,major,11910.0,2.5,71.930,41.34,85990.0,1.3,18.97,26.34,37.30,50.90,67.58,39460.0,54790.0,77590.0,105860.0,140560.0
2,2,11260,"Anchorage, AK",11-1011,Chief executives,NaN,640.0,5.7,3.850,71.82,149390.0,3.3,35.59,48.99,64.92,NaN,NaN,74020.0,101900.0,135040.0,NaN,NaN
3,2,11260,"Anchorage, AK",11-1021,General and operations managers,NaN,4930.0,4.7,29.774,39.59,82350.0,2.3,17.50,24.00,35.58,49.65,66.61,36400.0,49910.0,74000.0,103280.0,138540.0
4,2,11260,"Anchorage, AK",11-1031,Legislators,NaN,30.0,2.2,0.192,NaN,42440.0,8.3,NaN,NaN,NaN,NaN,NaN,15450.0,25230.0,32420.0,47970.0,57490.0
5,2,11260,"Anchorage, AK",11-2011,Advertising and promotions managers,NaN,70.0,28.7,0.449,31.27,65030.0,5.3,20.90,23.74,29.62,34.32,45.19,43480.0,49380.0,61610.0,71380.0,93990.0


In [683]:
# National

In [685]:
df_aux=pd.read_excel('/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/OESM/All/oesm09nat/national_M2009_dl.xls') 

In [687]:
#df_aux.rename(columns={'OCC_CODE':'SOC','OCC_TITLE':'SOCName'}, inplace=True)
df_aux.rename(columns={'occ_code':'SOC','occ_title':'SOCName'}, inplace=True)
#df_aux.drop(columns=['ANNUAL','HOURLY','ST','STATE', 'PRIM_STATE'], errors='ignore', inplace=True)
df_aux.drop(columns=['annual','hourly'], errors='ignore', inplace=True)

columns_to_convert = df_aux.columns.difference(['SOC', 'SOCName','group'])  
df_aux[columns_to_convert] = df_aux[columns_to_convert].replace(['*', '#'], np.nan).apply(pd.to_numeric, errors='coerce')
df_aux.rename(columns=lambda x: x.lower()  if x in columns_to_convert else x, inplace=True)


In [688]:
df_aux.head()

,SOC,SOCName,group,tot_emp,emp_prse,h_mean,a_mean,mean_prse,h_pct10,h_pct25,h_median,h_pct75,h_pct90,a_pct10,a_pct25,a_median,a_pct75,a_pct90
0,00-0000,All Occupations,total,130647610,0.1,20.90,43460.0,0.1,8.24,10.55,15.95,25.44,39.02,17140.0,21940.0,33190.0,52920.0,81160.0
1,11-0000,Management occupations,major,6116380,0.2,49.47,102900.0,0.2,20.86,29.76,42.95,61.67,NaN,43400.0,61910.0,89330.0,128270.0,NaN
2,11-1011,Chief executives,NaN,297640,0.5,80.43,167280.0,0.5,34.13,50.26,77.27,NaN,NaN,71000.0,104550.0,160720.0,NaN,NaN
3,11-1021,General and operations managers,NaN,1689680,0.3,53.15,110550.0,0.2,22.01,30.65,44.55,66.81,NaN,45780.0,63750.0,92650.0,138960.0,NaN
4,11-1031,Legislators,NaN,65750,1.4,NaN,37530.0,1.1,NaN,NaN,NaN,NaN,NaN,14830.0,15830.0,18810.0,50540.0,81150.0


In [689]:
# Merge to add mapped SOC codes
df_cross = pd.merge(
    df_aux,
    cross_occ,
    on='SOC',
    how='inner'
)

df_cross=df_cross.drop_duplicates(subset=['Mapped SOC']).drop(columns=['SOC'], errors='ignore')
df_cross.rename(columns={'Mapped SOC': 'SOC'}, inplace=True, errors='ignore')
df_aux=pd.concat([df_aux, df_cross], ignore_index=True)
df_aux.drop_duplicates(subset=['SOC'], inplace=True)

df_nat=df_aux[['SOC','tot_emp','a_mean']]
df_nat = df_nat[df_nat['SOC'].str[-7:] != '00-0000']


In [690]:

df_nat.rename(columns={'tot_emp':'natEmp', 'a_mean': 'a_mean_nt'},inplace=True)
df_nat=df_nat.drop_duplicates()
df_oesm = pd.merge(
    df_msa,         
    df_nat,    
    on='SOC',          
    how='left',indicator=True          
)
del df_nat

df_oesm['excludedOcc'] = df_oesm['natEmp'] - df_oesm['tot_emp']


# Separate occupation families (SOC ending in '0000')
df_occ = df_oesm[df_oesm['SOC'].str[-4:] == '0000'].copy()
df_oesm = df_oesm[df_oesm['SOC'].str[-4:] != '0000']

df_occ['OccFam'] = pd.to_numeric(df_occ['SOC'].astype(str).str[:2], errors='coerce').fillna(-999).astype(int)
df_occ['SOC'] = '-999'  # Ensure it's a string

df_msa_sum = (
    df_oesm.groupby(['FIPSState','BestFitMSA'], as_index=False)['tot_emp']
             .sum()
             .rename(columns={'tot_emp': 'totmsa_emp'})
)

df_excl_sum = (
    df_oesm.groupby(['FIPSState','BestFitMSA'], as_index=False)['excludedOcc']
             .sum()
             .rename(columns={'excludedOcc': 'excludedTotal'})
)

df_excl_sum_occ = (
    df_occ.groupby(['FIPSState','BestFitMSA'], as_index=False)['excludedOcc']
             .sum()
             .rename(columns={'excludedOcc': 'excludedTotal'})
)

df_oesm = pd.merge(
    df_oesm, 
    df_msa_sum, 
    on=['FIPSState','BestFitMSA'], 
    how='left'
)

df_oesm = pd.merge(
    df_oesm, 
    df_excl_sum, 
    on=['FIPSState','BestFitMSA'], 
    how='left'
)

df_occ = pd.merge(
    df_occ, 
    df_msa_sum, 
    on=['FIPSState','BestFitMSA'], 
    how='left'
)

df_occ = pd.merge(
    df_occ, 
    df_excl_sum_occ, 
    on=['FIPSState','BestFitMSA'], 
    how='left'
)

df_oesm['LSOCShare'] = (df_oesm['tot_emp'] / df_oesm['totmsa_emp'])*100
df_oesm['SOCShare'] = (df_oesm['excludedOcc'] / df_oesm['excludedTotal'])*100

df_occ['SOCShare'] = (df_occ['excludedOcc'] / df_occ['excludedTotal'])*100
df_occ['LSOCShare'] = (df_occ['tot_emp'] / df_occ['totmsa_emp'])*100

df_oesm['OccFam'] = pd.to_numeric(df_oesm['SOC'].astype(str).str[:2], errors='coerce').fillna(-999).astype(int)
df_oesm=pd.concat([df_oesm, df_occ], ignore_index=True)

##Creation of new variables
df_oesm['loga_wage']= np.log(df_oesm['a_mean'])

df_oesm.drop_duplicates(subset=['FIPSState','BestFitMSA','SOC','SOCName','OccFam'], inplace=True)
df_oesm = df_oesm.sort_values(by=['FIPSState', 'BestFitMSA', 'SOC','OccFam'])

df_oesm=df_oesm[['FIPSState', 'BestFitMSA','SOC', 'SOCName','a_mean', 'a_mean_nt', 'LSOCShare',
       'SOCShare', 'OccFam', 'loga_wage']]

In [691]:
df_oesm.head()

,FIPSState,BestFitMSA,SOC,SOCName,a_mean,a_mean_nt,LSOCShare,SOCShare,OccFam,loga_wage
172839,1,11500,-999,Management occupations,79860.0,102900.0,3.160788,4.682185,11,11.288030
172840,1,11500,-999,Business and financial operations occupations,56240.0,65900.0,3.366926,4.641757,13,10.937384
172841,1,11500,-999,Computer and mathematical science occupations,69480.0,76290.0,1.030692,2.529253,15,11.148794
172842,1,11500,-999,Architecture and engineering occupations,64560.0,73590.0,1.511681,1.846894,17,11.075350
172843,1,11500,-999,"Life, physical, and social science occupations",52040.0,65660.0,0.595511,1.001612,19,10.859768


In [692]:
df_oesm.to_parquet('/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/OESM/All/CLEAN/With Bartik/2009.parquet', compression='zstd', engine='pyarrow', index=False)


### 1.2 After 2010

In [ ]:
# Directory containing the files
directory = '/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/OESM/All/'


In [ ]:
# Loop through the years from 11 to 22
for xx in range(11, 23):
    zip_filename = f'oesm{xx}all.zip'
    zip_path = os.path.join(directory, zip_filename)
    
    # Check if the zip file exists
    if not os.path.exists(zip_path):
        print(f"File {zip_filename} does not exist. Skipping...")
        continue

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        # Find the Excel file inside the zip
        xls_files = [f for f in zip_ref.namelist() if f.endswith('.xls') or f.endswith('.xlsx')]
        
        if not xls_files:
            print(f"No Excel files found in {zip_filename}. Skipping...")
            continue
        
        # Process the first Excel file (if multiple, extend logic to handle all)
        excel_file = xls_files[0]
        print(f"Processing {excel_file} from {zip_filename}...")
        
        # Extract and load the Excel file
        extracted_path = zip_ref.extract(excel_file, path=directory)
        df_oesm = pd.read_excel(extracted_path)
        # Convert all column names to lowercase
        df_oesm.columns = df_oesm.columns.str.lower()

        # Perform transformations
        df_oesm.rename(columns={
            'area': 'AreaCode', 
            'area_title': 'AreaCodeName', 
            'area_type': 'AreaType',
            'occ_code': 'SOC',
            'occ code': 'SOC', 
            'occ_title': 'SOCName',
            'occ title': 'SOCName',
            'naics': 'NAICS',
            'naics_title': 'NAICSName'
        }, inplace=True)
        df_oesm['SOCName'] = df_oesm['SOCName'].str.replace('*', '', regex=False)
        df_oesm['SOCName'] = df_oesm['SOCName'].str.replace(r'[^a-zA-Z0-9\s-]', '', regex=True)
        df_oesm['SOCName'] = df_oesm['SOCName'].str.strip()
        df_oesm['OccFam'] = df_oesm['SOC'].str[:2]
        # Ensure NAICS is treated as a string for length checks
        df_oesm['NAICS'] = df_oesm['NAICS'].astype(str)

        # Case 1: If NAICS has less than 6 digits
        df_oesm.loc[df_oesm['NAICS'].str.len() < 6, 'Sector'] = df_oesm['NAICS']
        df_oesm.loc[df_oesm['NAICS'].str.len() < 6, 'NAICS6'] = -999

        # Case 2: If NAICS has 6 digits
        df_oesm.loc[df_oesm['NAICS'].str.len() == 6, 'Sector'] = df_oesm['NAICS'].str[:2]
        df_oesm.loc[df_oesm['NAICS'].str.len() == 6, 'NAICS6'] = df_oesm['NAICS']

        # Convert Sector and NAICS6 to integers
        df_oesm['Sector'] = pd.to_numeric(df_oesm['Sector'], errors='coerce').fillna(-999).astype(int)
        df_oesm['NAICS6'] = pd.to_numeric(df_oesm['NAICS6'], errors='coerce').fillna(-999).astype(int)


        df_oesm.drop(columns=['annual', 'hourly', 'own_code','NAICS'], errors='ignore', inplace=True)

        columns_to_convert = df_oesm.columns.difference(['AreaCode', 'AreaCodeName', 'NAICS6', 'NAICSName','Sector', 'SOC', 'SOCName','OccFam','group'])
        df_oesm[columns_to_convert] = df_oesm[columns_to_convert].replace(['*', '#'], np.nan).apply(pd.to_numeric, errors='coerce')

        # AreaType transformations
        df_oesm['FIPSState'] = np.where(df_oesm['AreaType'] == 1, 0, -999)
        df_oesm.loc[(df_oesm['AreaType'] > 1) & (df_oesm['AreaType'] < 4), 'FIPSState'] = df_oesm['AreaCode']
        df_oesm.loc[(df_oesm['AreaType'] > 1) & (df_oesm['AreaType'] < 4), 'BestFitMSA'] = -999
        df_oesm.loc[(df_oesm['AreaType'] > 3) & (df_oesm['AreaType'] < 6), 'FIPSState'] = (
            df_oesm.loc[(df_oesm['AreaType'] > 3) & (df_oesm['AreaType'] < 6), 'AreaCodeName']
            .str.extract(r',\s*(\d{2})')[0]
            .map(mapping_ST)
        )
        df_oesm.loc[(df_oesm['AreaType'] > 3) & (df_oesm['AreaType'] < 6), 'BestFitMSA'] = df_oesm['AreaCode']

        # Ensure columns are of the correct type
        df_oesm['FIPSState'] = pd.to_numeric(df_oesm['FIPSState'], errors='coerce').fillna(-999).astype('int64')
        df_oesm['BestFitMSA'] = pd.to_numeric(df_oesm['BestFitMSA'], errors='coerce').fillna(-999).astype('int64')

        columns_to_convert = df_oesm.columns.difference(['AreaCode', 'AreaCodeName', 'AreaType', 'NAICS6', 'NAICSName', 'FIPSState', 'BestFitMSA', 'SOC', 'SOCName','OccFam','Sector', 'group'])
        df_oesm.rename(columns=lambda x: x.lower() if x in columns_to_convert else x, inplace=True)
        
        # Merge to add mapped SOC codes
        df_oesm = pd.merge(
            df_oesm,
            cross_occ,
            on='SOC',
            how='left'
        )

        df_oesm['SOC'] = df_oesm['Mapped SOC'].fillna(df_oesm['SOC'])
        df_oesm = df_oesm.drop(columns=['Mapped SOC'], errors='ignore').drop_duplicates()

        # Save to Parquet
        output_path = directory + 'CLEAN/' + f'20{xx}.parquet'
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        df_oesm.to_parquet(output_path, index=False, compression="zstd")

        # Clean up extracted file
        os.remove(extracted_path)

        print(f"Saved processed data to {output_path}.")

### Bartik

In [652]:

# Define base directory and years
base_dir = '/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/OESM/All/CLEAN/'
years = range(2011, 2022)  # 2011 to 2021

# Loop through each year
for year in years:
    print(f"🔄 Processing year: {year}...")

    # Define input and output file paths
    file_path = os.path.join(base_dir, f"{year}.parquet")
    
    # Check if file exists
    if not os.path.exists(file_path):
        print(f"⚠️ Skipping {year}: File not found.")
        continue

    # Read OESM data
    df_oesm = pd.read_parquet(file_path)
    df_oesm = df_oesm[df_oesm['SOC'].str[-7:] != '00-0000']

    # -------------------------------
    # 1️⃣ Process National-Level Data
    # -------------------------------
    df_nat = df_oesm[df_oesm['AreaType'] == 1]  # National-level only
    df_nat = df_nat[df_nat['NAICSName'] == 'Cross-industry']  # Filter to cross-industry only
    df_nat = df_nat[['SOC', 'tot_emp', 'a_mean']].drop_duplicates()  # Keep only necessary columns

    # Rename columns
    df_nat.rename(columns={'tot_emp': 'natEmp', 'a_mean': 'a_mean_nt'}, inplace=True)

    # -------------------------------
    # 2️⃣ Process MSA-Level Data
    # -------------------------------
    df_oesm = df_oesm[df_oesm['AreaType'] != 1]  # Remove national-level data

    # Drop unnecessary columns
    df_oesm.drop(columns=['NAICSName', 'NAICS6', 'Sector'], inplace=True, errors='ignore')

    # Merge national employment data into MSA-level data
    df_oesm = pd.merge(df_oesm, df_nat, on='SOC', how='left', indicator=True)
    
    # Drop df_nat to free memory
    del df_nat

    # -------------------------------
    # 3️⃣ Compute Excluded Occupation
    # -------------------------------
    df_oesm['excludedOcc'] = df_oesm['natEmp'] - df_oesm['tot_emp']
    df_occ = df_oesm[df_oesm['SOC'].str[-4:] == '0000'].copy()
    df_oesm = df_oesm[df_oesm['SOC'].str[-4:] != '0000']

    df_occ['OccFam'] = pd.to_numeric(df_occ['SOC'].astype(str).str[:2], errors='coerce').fillna(-999).astype(int)
    df_occ['SOC'] = '-999'  # Ensure it's a string

    # Compute total excluded employment per MSA
    df_msa_sum = (
    df_oesm.groupby(['AreaCode'], as_index=False)['tot_emp']
             .sum()
             .rename(columns={'tot_emp': 'totmsa_emp'})
    )
    
    
    df_excl_sum = (
        df_oesm.groupby('AreaCode', as_index=False)['excludedOcc']
               .sum()
               .rename(columns={'excludedOcc': 'excludedTotal'})
    )

    df_excl_sum_occ = (
        df_occ.groupby(['AreaCode'], as_index=False)['excludedOcc']
                 .sum()
                 .rename(columns={'excludedOcc': 'excludedTotal'})
    )
    
    # Merge excluded totals back into the main DataFrame
    df_oesm = pd.merge(df_oesm, df_excl_sum, on='AreaCode', how='left')
    df_oesm = pd.merge(df_oesm, df_msa_sum, on='AreaCode', how='left')
    df_occ = pd.merge(df_occ, df_excl_sum_occ, on='AreaCode', how='left')
    df_occ = pd.merge(df_occ, df_msa_sum, on='AreaCode', how='left')

    df_oesm['SOCShare'] = (df_oesm['excludedOcc'] / df_oesm['excludedTotal'])*100
    df_oesm['LSOCShare'] = (df_oesm['tot_emp'] / df_oesm['totmsa_emp'])*100

    df_occ['SOCShare'] = (df_occ['excludedOcc'] / df_occ['excludedTotal'])*100
    df_occ['LSOCShare'] = (df_occ['tot_emp'] / df_occ['totmsa_emp'])*100

    

    # -------------------------------
    # 4️⃣ Merge with SOC Mappings
    # -------------------------------
    # Merge to add mapped SOC codes
    df_cross = pd.merge(
        df_oesm,
        cross_occ,
        on='SOC',
        how='inner'
    )

    df_cross=df_cross.drop_duplicates(subset=['Mapped SOC', 'AreaCode']).drop(columns=['SOC'], errors='ignore')
    df_cross.rename(columns={'Mapped SOC': 'SOC'}, inplace=True, errors='ignore')
    df_oesm=pd.concat([df_oesm, df_cross], ignore_index=True)
    df_oesm.drop_duplicates(subset=['AreaCode','SOC'], inplace=True)

    # Remove duplicates
    df_oesm.drop_duplicates(inplace=True)
    
    df_oesm['OccFam'] = pd.to_numeric(df_oesm['SOC'].astype(str).str[:2], errors='coerce').fillna(-999).astype(int)
    df_oesm=pd.concat([df_oesm, df_occ], ignore_index=True)
    df_oesm.drop_duplicates(subset=['FIPSState','BestFitMSA','SOC','SOCName','OccFam'], inplace=True)
    
    ##Creation of new variables
    df_oesm['loga_wage']= np.log(df_oesm['a_mean'])


    df_oesm = df_oesm.sort_values(by=['FIPSState', 'BestFitMSA', 'SOC','OccFam'])
    
    # Drop unnecessary columns after merging
    df_oesm=df_oesm[['FIPSState', 'BestFitMSA','SOC', 'SOCName','a_mean', 'a_mean_nt', 'LSOCShare',
       'SOCShare', 'OccFam', 'loga_wage']]
   
    df_oesm=df_oesm[df_oesm['BestFitMSA']!=-999] #Currently dropping the state observations

    # -------------------------------
    # 6️⃣ Save Output as Parquet
    # -------------------------------
    output_file = os.path.join(base_dir,'With Bartik', f"{year}.parquet")
    df_oesm.to_parquet(output_file, compression='zstd', engine='pyarrow', index=False)

    print(f"✅ Saved processed file for {year}: {output_file}")

print("🎉 All years processed successfully!")


🔄 Processing year: 2011...
✅ Saved processed file for 2011: /Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/OESM/All/CLEAN/With Bartik/2011.parquet
🔄 Processing year: 2012...
✅ Saved processed file for 2012: /Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/OESM/All/CLEAN/With Bartik/2012.parquet
🔄 Processing year: 2013...
✅ Saved processed file for 2013: /Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/OESM/All/CLEAN/With Bartik/2013.parquet
🔄 Processing year: 2014...
✅ Saved processed file for 2014: /Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/OESM/All/CLEAN/With Bartik/2014.parquet
🔄 Processing year: 2015...
✅ Saved processed file for 2015: /Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/OESM/All/CLEAN/With Bartik/2015.parquet
🔄 Processing year: 2016...
✅ Saved processed file for 2016: /Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/OESM/All/CLEAN/With Bartik/2016.

In [693]:
df_aux=pd.read_parquet('/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/OESM/All/CLEAN/With Bartik/2009.parquet')

In [694]:
df_aux.head()

,FIPSState,BestFitMSA,SOC,SOCName,a_mean,a_mean_nt,LSOCShare,SOCShare,OccFam,loga_wage
0,1,11500,-999,Management occupations,79860.0,102900.0,3.160788,4.682185,11,11.288030
1,1,11500,-999,Business and financial operations occupations,56240.0,65900.0,3.366926,4.641757,13,10.937384
2,1,11500,-999,Computer and mathematical science occupations,69480.0,76290.0,1.030692,2.529253,15,11.148794
3,1,11500,-999,Architecture and engineering occupations,64560.0,73590.0,1.511681,1.846894,17,11.075350
4,1,11500,-999,"Life, physical, and social science occupations",52040.0,65660.0,0.595511,1.001612,19,10.859768


In [695]:
# Define directory path
directory = "/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/OESM/All/CLEAN/With Bartik/"

# List parquet files (assuming format XXXX.parquet where XXXX is the year)
years = list(range(2009, 2022))  # From 2009 to 2021
file_paths = {year: os.path.join(directory, f"{year}.parquet") for year in years}

# Load all data into a dictionary
df_years = {year: pd.read_parquet(file_paths[year]) for year in years}

# Ensure column names for employment exist
employment_col = "SOCShare"  # Column for LOO national employment
employmentloc_col = "LSOCShare"  # Column for LOO national employment
wage_col='loga_wage' # Wage Column

In [696]:
# Compute Shift_k,t (log difference of national employment)
for year in years[1:]:  # Start from 2010 to compare with 2009
    prev_year = year - 1

    if prev_year in df_years and year in df_years:
        # Create a temporary column with the first 4 digits of BestFitMSA for both years
        df_years[year]['BestFitMSA_4'] = df_years[year]['BestFitMSA'].astype(str).str[:4]
        df_years[prev_year]['BestFitMSA_4'] = df_years[prev_year]['BestFitMSA'].astype(str).str[:4]

        # Merge current year with previous year data on the first 4 digits of BestFitMSA, SOC, and OccFam
        df_merged = pd.merge(
            df_years[year], 
            df_years[prev_year][['BestFitMSA_4', 'SOC', 'OccFam', employment_col, employmentloc_col, wage_col]],
            on=['BestFitMSA_4', 'SOC', 'OccFam'],
            how='left',
            suffixes=('', '_prev')
        )
        
        # Drop the temporary column after merging if no longer needed
        df_merged.drop(columns=['BestFitMSA_4'], inplace=True)
        # Compute the Shift component: log difference
        df_merged["shiftSOC"] = np.log(df_merged[employment_col]) - np.log(df_merged[f"{employment_col}_prev"])
        df_merged["shiftLSOC"] = np.log(df_merged[employmentloc_col]) - np.log(df_merged[f"{employmentloc_col}_prev"])
        df_merged["shiftwage"] = np.log(df_merged[wage_col]) - np.log(df_merged[f"{wage_col}_prev"])

        # Drop auxiliary columns
        df_merged.drop(columns=[f"{employment_col}_prev",f"{wage_col}_prev",f"{employmentloc_col}_prev",'h_pct25', 'h_median', 'h_pct75', 'h_pct90', 'a_pct10',
        'a_pct25','a_median', 'a_pct75', 'a_pct90', 'BestFitMSAName','GROUP', 'group',
        'excludedOcc', 'excludedTotal','o_group','prim_state', 'i_group','pct_total', 'pct_rpt', 
         'loc quotient', 'a_mean', 'mean_prse', 'h_pct10','h_mean', 'FIPSState'], inplace=True, errors='ignore')
        
        df_merged=df_merged[df_merged['SOC']!=-999]
        # Save updated file
        output_file = os.path.join(directory, f"{year}.parquet")
        df_merged.to_parquet(output_file, compression='zstd', engine='pyarrow', index=False)
        
        print(f"✅ Computed and saved Shift_k_t for {year}")

print("🎉 All years processed successfully!")


/opt/anaconda3/lib/python3.7/site-packages/pandas/core/arraylike.py:364: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


✅ Computed and saved Shift_k_t for 2010
✅ Computed and saved Shift_k_t for 2011
✅ Computed and saved Shift_k_t for 2012
✅ Computed and saved Shift_k_t for 2013
✅ Computed and saved Shift_k_t for 2014
✅ Computed and saved Shift_k_t for 2015
✅ Computed and saved Shift_k_t for 2016
✅ Computed and saved Shift_k_t for 2017
✅ Computed and saved Shift_k_t for 2018
✅ Computed and saved Shift_k_t for 2019
✅ Computed and saved Shift_k_t for 2020
✅ Computed and saved Shift_k_t for 2021
🎉 All years processed successfully!


## Economic Profile

In [701]:
# Specify the path to the .zip file
zip_file_path = '/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/Economic Profile/CAINC30.zip'

# Specify the directory where you want to extract the contents
extract_dir = '/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/Economic Profile/'

# Open and extract the .zip file
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)
    


In [702]:
df_cainc = pd.read_csv(extract_dir + 'CAINC30__ALL_AREAS_1969_2023.csv', delimiter=',', encoding='latin')


/opt/anaconda3/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3331: DtypeWarning: Columns (2,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [704]:
df_cainc['GeoFIPS'] = df_cainc['GeoFIPS'].str.replace('"', '', regex=False)
df_cainc['GeoFIPS'] = df_cainc['GeoFIPS'].str.strip()
df_cainc = df_cainc.drop(columns=[col for col in df_cainc.columns if col.isdigit() and int(col) < 2009])
df_cainc = df_cainc.iloc[:-4]
df_cainc['FIPSState'] = df_cainc['GeoFIPS'].str[:2].astype(int)
df_cainc['FIPSCounty'] = df_cainc['GeoFIPS'].str[-3:]

In [705]:
# Define a manual mapping for statistical-analysis-friendly names
short_names_mapping = {
    'Personal income (thousands of dollars)': 'pInc',
    'Net earnings by place of residence': 'netEarnbyRes',
    'Personal current transfer receipts': 'pTransReceipt',
    'Income maintenance benefits': 'incManBen',
    'Unemployment insurance compensation': 'UIC',
    'Retirement and other': 'retirement',
    'Dividends, interest, and rent': 'DivIntRent',
    'Population (persons)': 'pop',
    'Per capita personal income': 'percapInc',
    'Per capita net earnings': 'percapEarn',
    'Per capita personal current transfer receipts': 'percapTrans',
    'Per capita income maintenance benefits': 'percapIncMan',
    'Per capita unemployment insurance compensation': 'percapUIC',
    'Per capita retirement and other': 'percapRet',
    'Per capita dividends, interest, and rent': 'percapDivRent',
    'Earnings by place of work': 'earnWork',
    'Wages and salaries': 'wages',
    'Supplements to wages and salaries': 'supplWages',
    'Employer contributions for employee pension and insurance funds': 'emplPensionContr',
    'Employer contributions for government social insurance': 'emplGovContr',
    "Proprietors' income": 'propInc',
    "Farm proprietors' income": 'farmPropInc',
    "Nonfarm proprietors' income": 'nonfarmPropInc',
}

df_cainc['Description'] = df_cainc['Description'].str.strip().str.rstrip("12345/").str.strip()  # Clean the column values


In [706]:
df_cainc['Description'] = df_cainc['Description'].map(short_names_mapping)

In [707]:
df_cainc.rename(columns={'GeoFIPS':'FIPS'}, inplace=True)
df_cainc.drop(columns=['TableName','Region','Unit','LineCode', 'IndustryClassification','GeoName'], errors='ignore', inplace=True)

In [709]:
df_cainc[0:10]

,FIPS,Description,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,FIPSState,FIPSCounty
0,00000,pInc,12058253000,12547501000,13299818000,13905749000,14063283000,14778160000,15467113000,15884741000,16658962000,17514402000,18349584000,19600945000,21403979000,22077232000,23380269000,0,000
1,00000,netEarnbyRes,7729106000,8044939000,8532526000,8909292000,9081761000,9465614000,9840315000,10077316000,10553893000,11082597000,11569524000,11726635000,12801829000,13599086000,14305741000,0,000
2,00000,pTransReceipt,2146693000,2325117000,2358654000,2362988000,2424317000,2541516000,2685370000,2776983000,2855645000,2976239000,3147059000,4228795000,4653728000,4139146000,4267994000,0,000
3,00000,incManBen,230848000,256034000,264137000,267614000,271171000,270971000,273685000,270675000,269786000,262772000,271877000,314807000,472801000,533014000,343273000,0,000
4,00000,UIC,131990000,139715000,107871000,84439000,63050000,35788000,32861000,32266000,30634000,28026000,27839000,528892000,317523000,23825000,33159000,0,000
5,00000,retirement,1783855000,1929368000,1986646000,2010935000,2090096000,2234757000,2378824000,2474042000,2555225000,2685441000,2847343000,3385096000,3863404000,3582307000,3891562000,0,000
6,00000,DivIntRent,2182454000,2177445000,2408638000,2633469000,2557205000,2771030000,2941428000,3030442000,3249424000,3455566000,3633001000,3645515000,3948422000,4339000000,4806534000,0,000
7,00000,pop,306771529,309378433,311841632,314344331,316735375,319270047,321829327,324367742,326623063,328542157,330233102,331526933,332048977,333271411,334914895,0,000
8,00000,percapInc,39307,40557,42649,44237,44401,46287,48060,48971,51004,53309,55566,59123,64460,66244,69810,0,000
9,00000,percapEarn,25195,26004,27362,28342,28673,29648,30576,31068,32312,33733,35034,35372,38554,40805,42715,0,000


In [710]:
# Melt the DataFrame
df_cainc = pd.melt(
            df_cainc,
            id_vars=["FIPS", "FIPSState","FIPSCounty", "Description"],
            var_name='Year',  # Name for the year column
            value_name='Value')  # Name for the values column

# Rename 'Description' to 'Variable' for clarity
df_cainc.rename(columns={'Description': 'Variable'}, inplace=True)

# Final structure
df_cainc = df_cainc[["FIPS", "FIPSState","FIPSCounty", 'Year', 'Variable', 'Value']]



In [711]:
# Pivot the table to create columns per variable
df_cainc = df_cainc.pivot_table(
    index=["FIPS", "FIPSState","FIPSCounty", 'Year'],  # Keys to keep in rows
    columns='Variable',  # Values to turn into columns
    values='Value',  # Values for the new columns
    aggfunc='first'
    ).reset_index()
df_cainc['Year'] = df_cainc['Year'].astype(int)  


In [712]:

df_cainc = df_cainc.apply(pd.to_numeric, errors='coerce')
df_cainc = df_cainc.sort_values(by=['FIPS', 'FIPSState', 'FIPSCounty', 'Year'])

# 2️⃣ Select all numeric columns **excluding** the grouping keys
numeric_cols = df_cainc.select_dtypes(include=['number']).columns.difference(['FIPS', 'FIPSState', 'FIPSCounty', 'Year'])

# 3️⃣ Compute percent change for all numeric columns within each group
df_cainc[numeric_cols] = df_cainc.groupby(['FIPS', 'FIPSState', 'FIPSCounty'])[numeric_cols].pct_change() * 100


In [715]:
df_cainc.head()

Variable,FIPS,FIPSState,FIPSCounty,Year,DivIntRent,UIC,earnWork,emplGovContr,emplPensionContr,farmPropInc,incManBen,netEarnbyRes,nonfarmPropInc,pInc,pTransReceipt,percapDivRent,percapEarn,percapInc,percapIncMan,percapRet,percapTrans,percapUIC,pop,propInc,retirement,supplWages,wages
15,-999,1,-999,2009,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16,-999,1,-999,2010,1.917896,6.311287,2.755101,5.291244,1.494376,-19.563912,12.017571,2.750162,18.549762,4.112574,9.520893,1.273108,2.107134,3.459975,11.331776,8.610126,8.836443,5.652174,0.631786,15.522248,9.295239,2.613612,1.299223
17,-999,1,-999,2011,4.270930,-20.089771,2.496748,2.967916,1.939052,-64.503751,5.706510,4.033639,9.861808,3.586967,1.953787,3.754090,3.510691,3.066560,5.141658,1.707245,1.437216,-20.576132,0.504177,5.748772,2.218095,2.250254,2.125846
18,-999,1,-999,2012,5.984975,-29.229978,2.115070,1.429014,-6.124827,125.356335,-1.673629,2.338791,5.688096,2.513229,0.365069,5.377593,1.767871,1.940551,-2.195609,0.919574,-0.186428,-29.533679,0.563637,7.909751,1.498166,-3.823974,2.817591
19,-999,1,-999,2013,-2.533120,-29.295353,2.896963,1.447377,4.152628,294.609731,0.932708,1.506439,0.911307,0.940853,2.195647,-3.024098,0.981873,0.418963,0.408163,2.473243,1.656083,-29.411765,0.519459,12.298284,2.994014,3.283620,1.444661


In [714]:
df_cainc = df_cainc[df_cainc['FIPSState'] != 0]  # Remove national-level data
df_cainc.loc[df_cainc['FIPSCounty'] == 0, ['FIPS', 'FIPSCounty']] = -999



In [716]:
df_cainc.to_csv(extract_dir + 'ALL_CAINC.csv', index=False)


In [ ]:
df_aux=pd.merge(df,df_cainc, on=['FIPS', 'FIPSState', 'FIPSCounty', 'Year'],how='left')

In [ ]:
len(df_aux)

## GDP

In [717]:
df_cagdp = pd.read_csv('/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/GDP/CAGDP9/' + 'CAGDP9__ALL_AREAS_2001_2023.csv', delimiter=',', encoding='latin')


/opt/anaconda3/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3331: DtypeWarning: Columns (2) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [718]:
df_cagdp['GeoFIPS'] = df_cagdp['GeoFIPS'].str.replace('"', '', regex=False)
df_cagdp['GeoFIPS'] = df_cagdp['GeoFIPS'].str.strip()
df_cagdp = df_cagdp.drop(columns=[col for col in df_cagdp.columns if col.isdigit() and int(col) < 2009])
df_cagdp = df_cagdp.iloc[:-4]
df_cagdp['FIPSState'] = df_cagdp['GeoFIPS'].str[:2].astype(int)
df_cagdp['FIPSCounty'] = df_cagdp['GeoFIPS'].str[-3:]
df_cagdp.rename(columns={'GeoFIPS':'FIPS', 'IndustryClassification':'Sector',"Description":"SectorName"}, inplace=True)
df_cagdp.drop(columns=['TableName','Region','Unit','LineCode','GeoName'], errors='ignore', inplace=True)

In [719]:
df_cagdp['Sector'] = pd.to_numeric(df_cagdp['Sector'], errors='coerce').fillna(-999).astype(int)

In [720]:
# Melt the DataFrame
df_cagdp = pd.melt(
            df_cagdp,
            id_vars=["FIPS", "FIPSState","FIPSCounty", "Sector","SectorName"],
            var_name='Year',  # Name for the year column
            value_name='gdp')  # Name for the values column



In [721]:
df_cagdp['Year'] = df_cagdp['Year'].astype(int)  


In [722]:
cols_to_convert = df_cagdp.columns.difference(['SectorName'])  # Exclude 'SectorName'
df_cagdp[cols_to_convert] = df_cagdp[cols_to_convert].apply(pd.to_numeric, errors='coerce').astype('Int64')

In [723]:
# Replace 0s with a tiny positive number
df_cagdp['gdp'] = df_cagdp['gdp'].replace(0,1)
# Sort values to ensure correct year-over-year calculation
df_cagdp['lgdp_loc'] = np.log(df_cagdp['gdp']) 

In [724]:
df_cagdp.head()

,FIPS,FIPSState,FIPSCounty,Sector,SectorName,Year,gdp,lgdp_loc
0,0,0,0,-999,All industry total,2009,16349110000,23.517439
1,0,0,0,-999,Private industries,2009,13941688000,23.358149
2,0,0,0,11,"Agriculture, forestry, fishing and hunting",2009,159975000,18.890528
3,0,0,0,21,"Mining, quarrying, and oil and gas extraction",2009,224143000,19.227795
4,0,0,0,22,Utilities,2009,256992000,19.364556


In [728]:

df_cagdp = df_cagdp.sort_values(by=['FIPS', 'FIPSState', 'FIPSCounty', 'Sector', 'SectorName', 'Year'])

# Compute year-over-year change in log GDP
df_cagdp['lgdp_change'] = df_cagdp.groupby(['FIPS', 'FIPSState', 'FIPSCounty', 'Sector', 'SectorName'])['lgdp_loc'].diff()


In [729]:
df_cagdp.drop(columns=['gdp', 'lgdp_loc','SectorName'], inplace=True, errors='ignore')
df_cagdp=df_cagdp[df_cagdp['Year']!=2009]

In [730]:
df_cagdp=df_cagdp[df_cagdp['Sector']!=999]
df_cagdp=df_cagdp[df_cagdp['FIPS']!=0]
df_cagdp.loc[df_cagdp['FIPSCounty'] == 0, 'FIPSCounty'] = -999


In [731]:
df_cagdp.drop(columns=['FIPS'],inplace=True)

In [732]:
df_cagdp.drop_duplicates(subset=['FIPSState','FIPSCounty','Sector','Year'], inplace=True)

In [733]:
df_cagdp.to_csv('/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/GDP/' + 'ALL_CAGDP.csv', index=False)


In [ ]:
df_aux=pd.merge(df,df_cagdp, on=['FIPSState', 'FIPSCounty','Sector','Year'], how='left')

In [ ]:
len(df)

## Union

In [ ]:
## Industry

directory='/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/Union/'

In [ ]:
# Initialize an empty DataFrame to store the union of all DataFrames
df_union = pd.DataFrame()

# Iterate over all files in the directory
for filename in os.listdir(directory):
    # Check if the file starts with "ind" and ends with ".xlsx"
    if filename.startswith("ind_"):
        # Construct the full file path
        file_path = os.path.join(directory, filename)
        
        # Read the Excel file, skipping the first two rows
        df_aux = pd.read_excel(file_path, skiprows=2)
        
        # Rename and drop columns as specified
        df_aux.rename(columns={'% Mem': 'pctunion_ind'}, inplace=True)
        df_aux.drop(columns=['Obs', 'Employment (in 1000s)', 'Members (in 1000s)', 'Covered (in 1000s)', '% Cov'], 
                    errors='ignore', inplace=True)
        # Remove the last three rows
        df_aux = df_aux.iloc[:-3, :]
        # Add a 'Year' column extracted from the filename
        df_aux['Year'] = int(filename[4:8])  # Adjust slicing as per the filename format
        
        # Append the processed DataFrame to df_union
        df_union = pd.concat([df_union, df_aux], ignore_index=True)


In [ ]:
df_aux = pd.read_excel(directory + '2022-Census-Industry-Code-List-with-Crosswalk.xlsx', skiprows=6, sheet_name='2007 to 2022 Industry Crosswalk')


In [ ]:
# Reshape columns into a long format
df_long = pd.DataFrame()

# Extract unique year groups dynamically
df_aux.columns = df_aux.columns.str.lower()
years = set(col.split()[0] for col in df_aux.columns if col.split()[0].isdigit())

for year in years:
    temp = df_aux[[f"{year} census industry code", f"{year} naics code", f"{year} census industry description"]].copy()
    temp.columns = ["census industry code", "naics code", "census industry description"]
    temp["year"] = year
    df_long = pd.concat([df_long, temp], ignore_index=True)


In [ ]:
df_long.rename(columns={'census industry code': 'CIC', 'naics code':'NAICS','census industry description':'Industry'}, inplace=True)
df_long = df_long.dropna(how='all', axis=1)
df_union['CIC'] = pd.to_numeric(df_union['CIC'], errors='coerce')
df_union['CIC'] = df_union['CIC'].astype('Int64')
df_long['CIC'] = pd.to_numeric(df_long['CIC'], errors='coerce').astype('Int64')


In [ ]:
# Function to format the text
def capitalize_first_letter(text):
    words = text.lower().split()
    capitalized_words = [word.capitalize() if word not in ["&"] else word for word in words]
    return " ".join(capitalized_words)

# Apply to the DataFrame
df_union['Industry'] = df_union['Industry'].apply(capitalize_first_letter)
df_long['Industry'] = df_long['Industry'].str.strip()
df_long['Industry'] = df_long['Industry'].fillna('').apply(capitalize_first_letter)


In [ ]:
df_long.head()

In [ ]:
df_aux=pd.merge(df_union, df_long.drop_duplicates(subset=['CIC', 'NAICS'], keep='first'), on='CIC', how='left',indicator=True)

In [ ]:
df_aux['NAICS'] = df_aux['NAICS'].astype(str)

In [ ]:
industry_to_naics = {'Agriculture, Forestry, Fishing, & Hunting': '11',
 'Mining': '21',
 'Utilities': '22',
 'Construction': '23',
 'Manufacturing': '31-33',
 'Wholesale Trade': '42',
 'Retail Trade': '44-45',
 'Transportation & Warehousing': '48-49',
 'Information': '51',
 'Finance & Insurance': '52',
 'Real Estate & Rental & Leasing': '53',
 'Prof., Scientific, Technical Services': '54',
 'Management & Related Services': '55',
 'Administrative & Support & Waste Management & Remediation Services': '56',
 'Educational Services': '61',
 'Health Care & Social Assistance': '62',
 'Arts, Entertainment, & Recreation': '71',
 'Accomodation & Food Services': '72',
 'Other Services, Exc. Public Admin.': '81',
 'Public Administration': '92'}

In [ ]:
# Apply the mapping only when NAICS is NaN
df_aux.loc[df_aux['NAICS']=='nan', 'NAICS'] = df_aux['Industry_x'].map(industry_to_naics)
df_aux = df_aux[df_aux['NAICS'].notna()].reset_index(drop=True)


In [ ]:
# Step 1: Replace unwanted substrings and standardize formatting
df_aux['NAICS'] = df_aux['NAICS'].str.replace('Part of', '', regex=False)
df_aux['NAICS'] = df_aux['NAICS'].str.replace('Pts.', '', regex=False)
df_aux['NAICS'] = df_aux['NAICS'].str.replace('pt.', '', regex=False)
df_aux['NAICS'] = df_aux['NAICS'].str.replace('exc.', '', regex=False)
df_aux['NAICS'] = df_aux['NAICS'].str.replace('and', ',', regex=False)  # Replace "and" with ","
df_aux['NAICS'] = df_aux['NAICS'].str.strip()

# Step 2: Split on commas and explode into separate rows
df_aux = (
    df_aux.assign(NAICS=df_aux['NAICS'].astype(str).str.split(','))
    .explode('NAICS')
    .reset_index(drop=True)
)

# Step 3: Split on dashes (ranges) and explode again
df_aux = (
    df_aux.assign(NAICS=df_aux['NAICS'].astype(str).str.split('-'))
    .explode('NAICS')
    .reset_index(drop=True)
)

# Step 4: Strip any leading/trailing spaces in the new `NAICS` column values
df_aux['NAICS'] = df_aux['NAICS'].str.strip()
df_aux['NAICS'] = df_aux['NAICS'].str.replace(r'\s+', '', regex=True)  # Remove any remaining spaces

# Ensure 'NAICS' is treated as a string
df_aux['NAICS'] = df_aux['NAICS'].astype(str)

In [ ]:
# Assign values to Sector, NAICS3, NAICS4, NAICS5, and NAICS6 based on NAICS length
df_aux.loc[df_aux['NAICS'].str.len() >= 2, 'Sector'] = df_aux['NAICS'].str[:2]
df_aux.loc[df_aux['NAICS'].str.len() >= 3, 'NAICS3'] = df_aux['NAICS'].str[:3]
df_aux.loc[df_aux['NAICS'].str.len() >= 4, 'NAICS4'] = df_aux['NAICS'].str[:4]
df_aux.loc[df_aux['NAICS'].str.len() >= 5, 'NAICS5'] = df_aux['NAICS'].str[:5]
df_aux.loc[df_aux['NAICS'].str.len() == 6, 'NAICS6'] = df_aux['NAICS']

# Replace missing values with NaN for consistency
df_aux[['Sector', 'NAICS3', 'NAICS4', 'NAICS5', 'NAICS6']] = df_aux[
    ['Sector', 'NAICS3', 'NAICS4', 'NAICS5', 'NAICS6']
].fillna(-999)

# Convert to numeric where applicable
for col in ['Sector', 'NAICS3', 'NAICS4', 'NAICS5', 'NAICS6']:
    df_aux[col] = pd.to_numeric(df_aux[col], errors='coerce').astype(int)

In [ ]:
df_aux.drop(columns=['CIC','Industry_x','NAICS','Industry_y', '_merge','year'], errors='ignore', inplace=True)

In [ ]:
df_aux.Year=df_aux.Year.astype(int)

In [ ]:
df_aux.head()

In [ ]:
df_aux.to_csv(directory + 'industry.csv', index=False)


In [ ]:
## Occupation

In [ ]:
# Initialize an empty DataFrame to store the union of all DataFrames
df_union = pd.DataFrame()

# Iterate over all files in the directory
for filename in os.listdir(directory):
    # Check if the file starts with "ind" and ends with ".xlsx"
    if filename.startswith("occ_"):
        # Construct the full file path
        file_path = os.path.join(directory, filename)
        
        # Read the Excel file, skipping the first two rows
        df_aux = pd.read_excel(file_path, skiprows=2)
        
        # Rename and drop columns as specified
        df_aux.rename(columns={'% Mem': 'pctunion_occ'}, inplace=True)
        df_aux.drop(columns=['Obs', 'Employment (in 1000s)', 'Members (in 1000s)', 'Covered (in 1000s)', '% Cov'], 
                    errors='ignore', inplace=True)
        # Remove the last three rows
        df_aux = df_aux.iloc[:-3, :]
        # Add a 'Year' column extracted from the filename
        df_aux['Year'] = int(filename[4:8])  # Adjust slicing as per the filename format
        
        # Append the processed DataFrame to df_union
        df_union = pd.concat([df_union, df_aux], ignore_index=True)


In [ ]:
df_aux = pd.read_excel(directory + '2018-occupation-code-list-and-crosswalk.xlsx', skiprows=4, sheet_name='2018 Census Occ Code List')


In [ ]:
df_aux['2018 Census Title '] = df_aux['2018 Census Title '].fillna(df_aux['Unnamed: 0'])
df_aux['Occupation 2010 Description'] = df_aux['Occupation 2010 Description'].fillna(df_aux['Unnamed: 4'])
df_aux.drop(columns=['Unnamed: 0','Unnamed: 4'], errors='ignore', inplace=True)


In [ ]:
df_aux = df_aux.iloc[5:].reset_index(drop=True)

df_aux=df_aux.astype(str)

In [ ]:
# Extract unique year groups dynamically
df_aux.columns = df_aux.columns.str.lower()
df_aux.columns = df_aux.columns.str.strip()
years = set(col.split()[0] for col in df_aux.columns if col.split()[0].isdigit())
df_aux.rename(columns={'occupation 2010 description': '2010 census title'}, inplace=True)


In [ ]:
# Reshape columns into a long format
df_long = pd.DataFrame()

for year in years:
    temp = df_aux[[f"{year} census title", f"{year} census code", f"{year} soc code"]].copy()
    temp.columns = ["census title", "census code", "soc code"]
    temp["year"] = year
    df_long = pd.concat([df_long, temp], ignore_index=True)

In [ ]:
# Check if the first three columns contain the string "nan"
rows_to_drop = df_long.iloc[:, :3].eq("nan").all(axis=1)
df_long = df_long[~rows_to_drop].reset_index(drop=True)
df_long.rename(columns={'census code': 'COC','soc code': 'SOC','census title':"Occupation"}, inplace=True)

df_long = (
    df_long.assign(COC=df_long['COC'].astype(str).str.split('-'))
    .explode('COC')
    .reset_index(drop=True)
)
df_long['COC'] = df_long['COC'].str.strip()
df_long['SOC'] = df_long['SOC'].str.strip()
df_long['Occupation'] = df_long['Occupation'].str.replace(':', '')
df_long['SOC'] = df_long['SOC'].str.replace('X', '0')
df_long = (
    df_long.assign(SOC=df_long['SOC'].astype(str).str.split(' - '))
    .explode('SOC')
    .reset_index(drop=True)
)


In [ ]:
df_union['COC'] = pd.to_numeric(df_union['COC'], errors='coerce')
df_union['COC'] = df_union['COC'].astype('Int64')
df_long['COC'] = pd.to_numeric(df_long['COC'], errors='coerce').astype('Int64')


In [ ]:
# Apply to the DataFrame

df_union['Occupation'] = df_union['Occupation'].apply(capitalize_first_letter)
df_long['Occupation'] = df_long['Occupation'].str.strip()
df_long['Occupation'] = df_long['Occupation'].fillna('').apply(capitalize_first_letter)

In [ ]:
df_aux=pd.merge(df_union, df_long.drop_duplicates(subset=['COC', 'SOC'], keep='first').drop(columns='Occupation', errors='ignore'), on='COC', how='left',indicator=True)

In [ ]:
# Step 1: Identify unmatched rows
unmatched = df_aux[df_aux['_merge'] == 'left_only'].drop(columns=['_merge'], errors='ignore')

# Step 2: Perform the new merge based on 'Occupation'
df_occupation_merge = pd.merge(
    unmatched.drop(columns=['COC','SOC','year'], errors='ignore'),
    df_long.drop_duplicates(subset=['COC', 'SOC'], keep='first'),  # Replace with the appropriate DataFrame to merge with
    on='Occupation',  # Key to merge on
    how='left',
    indicator=True
)

# Step 3: Update values in the original DataFrame
for col in df_occupation_merge.columns:
    if col not in df_aux.columns or col == '_merge':  # Skip if the column doesn't exist in the original
        continue
    df_aux.loc[df_aux['_merge'] == 'left_only', col] = df_occupation_merge[col]

# Step 4: Drop the '_merge' column if needed
df_aux = df_aux.drop(columns=['_merge'], errors='ignore')


In [ ]:
df_aux['OccFam'] = df_aux['SOC'].str[:2].where(df_aux['SOC'].str.len() >= 2, np.nan)


In [ ]:
df_aux = df_aux.dropna(how='all').reset_index(drop=True)


In [ ]:
soc_to_occfam = {
    "11": "Management Occupations",
    "13": "Business and Financial Operations Occupations",
    "15": "Computer and Mathematical Occupations",
    "17": "Architecture and Engineering Occupations",
    "19": "Life, Physical, and Social Science Occupations",
    "21": "Community and Social Service Occupations",
    "23": "Legal Occupations",
    "25": "Education, Training, and Library Occupations",
    "27": "Arts, Design, Entertainment, Sports, and Media Occupations",
    "29": "Healthcare Practitioners and Technical Occupations",
    "31": "Healthcare Support Occupations",
    "33": "Protective Service Occupations",
    "35": "Food Preparation and Serving Related Occupations",
    "37": "Building and Grounds Cleaning and Maintenance Occupations",
    "39": "Personal Care and Service Occupations",
    "41": "Sales and Related Occupations",
    "43": "Office and Administrative Support Occupations",
    "45": "Farming, Fishing, and Forestry Occupations",
    "47": "Construction and Extraction Occupations",
    "49": "Installation, Maintenance, and Repair Occupations",
    "51": "Production Occupations",
    "53": "Transportation and Material Moving Occupations"
}

# Extract the first two digits of the SOC code
df_aux['OccFam'] = df_aux['SOC'].str[:2]

# Map the OccFam to their corresponding names
df_aux['OccFamName'] = df_aux['OccFam'].map(soc_to_occfam)

In [ ]:
# Update `SOCName` where the last four digits of `SOC` are not "0000"
df_aux.loc[~df_aux['SOC'].str.endswith('0000'), 'SOCName'] = df_aux.loc[
    ~df_aux['SOC'].str.endswith('0000'), 'Occupation'
]


In [ ]:
df_aux.Year=df_aux.Year.astype(int)
df_aux.drop(columns=['COC','Occupation','year'], errors='ignore', inplace=True)

In [ ]:
df_aux.to_csv(directory + 'soc_union.csv', index=False)


In [ ]:
## MSA

In [ ]:
# Initialize an empty DataFrame to store the union of all DataFrames
df_union = pd.DataFrame()

# Iterate over all files in the directory
for filename in os.listdir(directory):
    # Check if the file starts with "ind" and ends with ".xlsx"
    if filename.startswith("msa_"):
        # Construct the full file path
        file_path = os.path.join(directory, filename)
        
        # Read the Excel file, skipping the first two rows
        df_aux = pd.read_excel(file_path, skiprows=2)
        
        # Rename and drop columns as specified
        df_aux.rename(columns={'% Mem': 'pctunion_msa'}, inplace=True)
        df_aux.drop(columns=['Obs', 'Employment (in 1000s)', 'Members (in 1000s)', 'Covered (in 1000s)', '% Cov'], 
                    errors='ignore', inplace=True)
        # Remove the last three rows
        df_aux = df_aux.iloc[:-3, :]
        # Add a 'Year' column extracted from the filename
        df_aux['Year'] = int(filename[4:8])  # Adjust slicing as per the filename format
        
        # Append the processed DataFrame to df_union
        df_union = pd.concat([df_union, df_aux], ignore_index=True)


In [ ]:
df_union = df_union[df_union['Sector'] == 'Total'].reset_index(drop=True)
df_union.drop(columns=['Sector'], errors='ignore', inplace=True)

In [ ]:
# Filter the DataFrame to keep only rows where FIPS has 5 or more digits
df_union = df_union[df_union['FIPS code'].astype(str).str.len() >= 5].reset_index(drop=True)


In [ ]:
df_union.rename(columns={'Metropolitan Area': 'BestFitMSAName','FIPS code':'BestFitMSA'}, inplace=True)


In [ ]:
df_union.head()


In [ ]:
df_union.Year=df_union.Year.astype(int)
df_union.to_csv(directory + 'msas_union.csv', index=False)


In [ ]:
df_aux = pd.read_excel(directory + 'union_state.xlsx', skiprows=3)

# Filter the DataFrame to keep only rows where Series ID starts with 'LUU02048996'
df_aux = df_aux[df_aux['Series ID'].astype(str).str.startswith('LUU02048996')].reset_index(drop=True)
df_aux['FIPSState']=df_aux['Series ID'].str[-2:]
df_aux = df_aux.drop(columns=['Series ID'], errors='ignore')
df_aux.columns = [
    col if col == "FIPSState" else col.split("\n")[-1] if "\n" in col else col
    for col in df_aux.columns
]

In [ ]:
# Melt the DataFrame
df_aux = pd.melt(
            df_aux,
            id_vars=[ "FIPSState"],
            var_name='Year',  # Name for the year column
            value_name='pctunion_st')  # Name for the values column



In [ ]:
df_aux.Year=df_aux.Year.astype(int)
df_aux.to_csv(directory + 'state_unionb.csv', index=False)


In [ ]:
df=df_union

In [ ]:
##Merge

In [ ]:
df_msa = pd.read_csv(directory + '/msas_union.csv')
df_st = pd.read_csv(directory + '/state_unionb.csv')

In [ ]:
df_msa.dtypes

In [ ]:
df_st=df_st[df_st['FIPSState']!=0]
df_st=df_st[df_st['Year']!=2009]
df_st['BestFitMSA']=-999
df_st.rename(columns={'pctunion_st':'lpctunion'}, inplace=True)

In [ ]:
df_msa.drop(columns='BestFitMSAName',inplace=True, errors='ignore')
df_msa.rename(columns={'pctunion_msa':'lpctunion'}, inplace=True, errors='ignore')
df_msa['FIPSState']=-999


In [ ]:
df_loc=pd.concat([df_msa,df_st], ignore_index=True).drop_duplicates()

In [ ]:
df_loc.to_csv(directory + 'loc_union.csv', index=False)


In [ ]:
df_loc=pd.read_csv(directory + '/loc_union.csv')

df=pd.merge(df.drop(columns=['_merge'], errors='ignore'), df_loc[df_loc['BestFitMSA']!=-999].drop(columns=['FIPSState'], errors='ignore'), on=['Year', 'BestFitMSA'], how='left', indicator=True)
unmatched = df[df['_merge'] == 'left_only'].drop(columns=['_merge','lpctunion'], errors='ignore')
# Step 4: Second merge using FIPSState & FIPSCounty
df_aux = pd.merge(
    unmatched,
    df_loc[df_loc['BestFitMSA']==-999].drop(columns=['BestFitMSA'], errors='ignore'),
    on=[ 'Year', 'FIPSState'],
    how='left', indicator=True
)
df = pd.concat([
    df[df['_merge'] != 'left_only'],  # Keep the successfully merged ones from first merge
    df_aux.drop(columns=['_merge'], errors='ignore')  # Append the successfully merged from second merge
], ignore_index=True)
del df_loc, df_aux

df_ind=pd.read_csv(directory + '/industry.csv')
df=pd.merge(df,df_ind.drop_duplicates(subset=['Year', 'Sector', 'NAICS3', 'NAICS4', 'NAICS5',
       'NAICS6']) , on=['Year', 'Sector', 'NAICS3', 'NAICS4', 'NAICS5',
       'NAICS6'],how='left', indicator=True)

del df_ind

In [ ]:
df_soc.loc[df_soc['SOC'].astype(str).str[-5:] == '-0000', 'SOC'] = '-999'


In [ ]:
df_soc.to_csv(directory + 'soc_union.csv', index=False)


In [ ]:
df_soc=pd.read_csv(directory + 'Union/soc_union.csv')
df_aux=pd.merge(df.drop(columns=['_merge'], errors='ignore'), df_soc[df_soc['SOC']!='-999'].drop(columns=['OccFam','OccFamName','SOCName'],
                            errors='ignore').drop_duplicates(subset=['Year','SOC']), 
                            on=['Year', 'SOC'], how='left', indicator=True)
unmatched = df_aux[df_aux['_merge'] == 'left_only'].drop(columns=['_merge','pctunion_occ'], errors='ignore')
df_aux2 = pd.merge(
    unmatched,
    df_soc[df_soc['SOC']!='-999'].drop(columns=['OccFam','OccFamName','SOC'],
                            errors='ignore').drop_duplicates(subset=['Year','SOCName']), 
                            on=['Year', 'SOCName'], how='left', indicator=True)
unmatched = df_aux2[df_aux2['_merge'] == 'left_only'].drop(columns=['_merge','pctunion_occ'], errors='ignore')
df_aux3 = pd.merge(
    unmatched,
    df_soc[df_soc['SOC']=='-999'].drop(columns=['OccFamName','SOC','SOCName'],
                            errors='ignore').drop_duplicates(subset=['Year','OccFam']), 
                            on=['Year', 'OccFam'], how='left', indicator=True)
df_aux = pd.concat([
    df_aux[df_aux['_merge'] != 'left_only'],  # Keep the successfully merged ones from first merge
    df_aux2[df_aux2['_merge'] != 'left_only'].drop(columns=['_merge'], errors='ignore')  # Append the successfully merged from second merge
], ignore_index=True)

df_aux = pd.concat([
    df_aux,  
    df_aux3.drop(columns=['_merge'], errors='ignore')  # Append the successfully merged from second merge
], ignore_index=True)
df=df_aux
del df_aux, df_aux2, df_aux3, df_soc, unmatched

## Race and Gender

In [ ]:
## AUX SOCName to SOC codes

In [ ]:
# Function to clean and format the Group/SOC column
def format_group_name(name):
    # Check if name is not NaN
    if pd.isna(name):
        return name
    # Split the name into parts by '-'
    parts = name.split('-', 1)  # Split only at the first occurrence of '-'
    # Title-case the rest and join back
    return ' '.join(word.capitalize() for word in parts[-1].split())

In [ ]:
def clean_soc_mapping(df):

    # Replace commas in the DataFrame
    df = df.replace(',', '', regex=True)
    
    # Clean 'SOCName' column
    df['SOCName'] = df['SOCName'].str.title()
    df['SOCName'] = df['SOCName'].replace('--', '-', regex=True)
    df['SOCName'] = df['SOCName'].str.replace(
        r'(?<=[-/])([a-z])', 
        lambda match: match.group(1).upper(),
        regex=True
    )
    
    # Strip whitespace from 'SOC' and 'SOCName'
    df['SOC'] = df['SOC'].str.strip()
    df['SOCName'] = df['SOCName'].str.strip()
    
    # Drop duplicates to ensure unique pairs
    df = df.drop_duplicates()

    
    return df

In [ ]:
# Load the data
df_occ = pd.read_excel('/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/soc_2010_to_2018_crosswalk.xlsx', skiprows=8) 

# Clean SOC Titles
df_occ['2010 SOC Title'] = df_occ['2010 SOC Title'].str.replace(r'\(.*?\)', '', regex=True).str.strip()
df_occ['2018 SOC Title'] = df_occ['2018 SOC Title'].str.replace(r'\(.*?\)', '', regex=True).str.strip()

# Combine 2010 and 2018 SOC codes and titles
df_mapping = pd.concat(
    [
        df_occ[['2010 SOC Code', '2010 SOC Title']].rename(
            columns={'2010 SOC Code': 'SOC', '2010 SOC Title': 'SOCName'}
        ),
        df_occ[['2018 SOC Code', '2018 SOC Title']].rename(
            columns={'2018 SOC Code': 'SOC', '2018 SOC Title': 'SOCName'}
        ),
    ],
    ignore_index=True
)

# Ensure SOC Codes are strings and replace any 'X' with '0'
df_mapping['SOC'] = df_mapping['SOC'].astype(str).str.replace('X', '0')

# Split SOC codes and add rows for each
df_mapping['SOC'] = df_mapping['SOC'].str.split(',')  # Split by comma if multiple codes exist
df_mapping = df_mapping.explode('SOC')  # Add additional rows
df_mapping['SOCName'] = df_mapping['SOCName'].str.title()  # Word capitalize titles



In [ ]:
text="""<ul style="word-break:break-word;"><li>119021: MGR-Construction Managers</li><li>119070: MGR-Entertainment And Recreation Managers</li><li>119081: MGR-Lodging Managers</li><li>131011: BUS-Agents And Business Managers Of Artists, Performers, And Athletes</li><li>131023: BUS-Purchasing Agents, Except Wholesale, Retail, And Farm Products</li><li>131051: BUS-Cost Estimators</li><li>151251: CMM-Computer Programmers</li><li>1721XX: ENG-Petroleum, Mining And Geological Engineers, Including Mining Safety Engineers</li><li>17301X: ENG-Other Drafters</li><li>191030: SCI-Conservation Scientists And Foresters</li><li>211011: CMS-Substance Abuse And Behavioral Disorder Counselors</li><li>211014: CMS-Mental Health Counselors</li><li>211015: CMS-Rehabilitation Counselors</li><li>211023: CMS-Mental Health And Substance Abuse Social Workers</li><li>271025: ENT-Interior Designers</li><li>273031: ENT-Public Relations Specialists</li><li>274021: ENT-Photographers</li><li>291123: MED-Physical Therapists</li><li>292010: MED-Clinical Laboratory Technologists And Technicians</li><li>292035: MED-Magnetic Resonance Imaging Technologists</li><li>292042: MED-Emergency Medical Technicians</li><li>292090: MED-Miscellaneous Health Technologists And Technicians</li><li>31909X: HLS-Other Healthcare Support Workers</li><li>319096: HLS-Veterinary Assistants And Laboratory Animal Caretakers</li><li>351012: EAT-First-Line Supervisors Of Food Preparation And Serving Workers</li><li>371011: CLN-First-Line Supervisors Of Housekeeping And Janitorial Workers</li><li>393010: PRS-Gambling Services Workers</li><li>399032: PRS-Recreation Workers</li><li>412021: SAL-Counter And Rental Clerks</li><li>433021: OFF-Billing And Posting Clerks</li><li>434161: OFF-Human Resources Assistants, Except Payroll And Timekeeping</li><li>435021: OFF-Couriers And Messengers</li><li>435061: OFF-Production, Planning, And Expediting Clerks</li><li>435111: OFF-Weighers, Measurers, Checkers, And Samplers, Recordkeeping</li><li>436012: OFF-Legal Secretaries And Administrative Assistants</li><li>439XXX: OFF-Other Office And Administrative Support Workers</li><li>472031: CON-Carpenters</li><li>472080: CON-Drywall Installers, Ceiling Tile Installers, And Tapers</li><li>472130: CON-Insulation Workers</li><li>472221: CON-Structural Iron And Steel Workers</li><li>4740XX: CON-Other Construction And Related Workers</li><li>493011: RPR-Aircraft Mechanics And Service Technicians</li><li>493031: RPR-Bus And Truck Mechanics And Diesel Engine Specialists</li><li>499010: RPR-Control And Valve Installers And Repairers</li><li>499021: RPR-Heating, Air Conditioning, And Refrigeration Mechanics And Installers</li><li>499052: RPR-Telecommunications Line Installers And Repairers</li><li>499060: RPR-Precision Instrument And Equipment Repairers</li><li>512031: PRD-Engine And Other Machine Assemblers</li><li>513011: PRD-Bakers</li><li>515112: PRD-Printing Press Operators</li><li>516040: PRD-Shoe And Leather Workers</li><li>519020: PRD-Crushing, Grinding, Polishing, Mixing, And Blending Workers</li><li>519197: PRD-Tire Builders</li><li>536051: TRN-Transportation Inspectors</li><li>537051: TRN-Industrial Truck And Tractor Operators</li><li>537063: TRN-Machine Feeders And Offbearers</li><li>551010: MIL-Military Officer Special And Tactical Operations Leaders</li><li>1930XX: SCI-Other Social Scientists</li><li>273099: ENT-Media And Communication Workers, All Other</li><li>274030: ENT-Television, Video, And Film Camera Operators And Editors</li><li>131111: BUS-Management Analysts</li><li>131199: BUS-Business Operations Specialists, All Other</li><li>1320XX: FIN-Other Financial Specialists</li><li>132081: FIN-Tax Examiners And Collectors, And Revenue Agents</li><li>151221: CMM-Computer And Information Research Scientists</li><li>151252: CMM-Software Developers</li><li>172121: ENG-Marine Engineers And Naval Architects</li><li>173023: ENG-Electrical and Electronic Engineering Technologists And Technicians</li><li>192021: SCI-Atmospheric And Space Scientists</li><li>193033: SCI-Clinical And Counseling Psychologists</li><li>194021: SCI-Biological Technicians</li><li>212021: CMS-Directors, Religious Activities And Education</li><li>232011: LGL-Paralegals And Legal Assistants</li><li>232093: LGL-Title Examiners, Abstractors, And Searchers</li><li>251000: EDU-Postsecondary Teachers</li><li>2590XX: EDU-Other Educational Instruction and Library Workers</li><li>271023: ENT-Floral Designers</li><li>272099: ENT-Entertainers And Performers, Sports And Related Workers, All Other</li><li>291299: MED-Healthcare Diagnosing Or Treating Practitioners, All Other</li><li>292053: MED-Psychiatric Technicians</li><li>311122: HLS-Personal Care Aides</li><li>319094: HLS-Medical Transcriptionists</li><li>331011: PRT-First-Line Supervisors Of Correctional Officers</li><li>332011: PRT-Firefighters</li><li>339094: PRT-School Bus Monitors</li><li>353041: EAT-Food Servers, Nonrestaurant</li><li>359031: EAT-Hosts And Hostesses, Restaurant, Lounge, And Coffee Shop</li><li>37201X: CLN-Janitors And Building Cleaners</li><li>394031: PRS-Morticians, Undertakers, And Funeral Arrangers</li><li>395012: PRS-Hairdressers, Hairstylists, And Cosmetologists</li><li>411012: SAL-First-Line Supervisors Of Non-Retail Sales Workers</li><li>412022: SAL-Parts Salespersons</li><li>413091: SAL-Sales Representatives Of Services, Except Advertising, Insurance, Financial Services, And Travel</li><li>419041: SAL-Telemarketers</li><li>419099: SAL-Sales And Related Workers, All Other</li><li>432011: OFF-Switchboard Operators, Including Answering Service</li><li>439051: OFF-Mail Clerks And Mail Machine Operators, Except Postal Service</li><li>453031: FFF-Fishing And Hunting Workers</li><li>472111: CON-Electricians</li><li>472151: CON-Pipelayers</li><li>474061: CON-Rail-Track Laying And Maintenance Equipment Operators</li><li>4750YY: EXT-Derrick, Rotary Drill, And Service Unit Operators, And Roustabouts, Oil, Gas, And Mining</li><li>492020: RPR-Radio And Telecommunications Equipment Installers And Repairers</li><li>492098: RPR-Security And Fire Alarm Systems Installers</li><li>493023: RPR-Automotive Service Technicians And Mechanics</li><li>499096: RPR-Riggers</li><li>513099: PRD-Food Processing Workers, All Other</li><li>514111: PRD-Tool And Die Makers</li><li>514120: PRD-Welding, Soldering, And Brazing Workers</li><li>517011: PRD-Cabinetmakers And Bench Carpenters</li><li>517042: PRD-Woodworking Machine Setters, Operators, And Tenders, Except Sawing</li><li>518031: PRD-Water And Wastewater Treatment Plant And System Operators</li><li>518090: PRD-Miscellaneous Plant And System Operators</li><li>519041: PRD-Extruding, Forming, Pressing, And Compacting Machine Setters, Operators, And Tenders</li><li>519071: PRD-Jewelers And Precious Stone And Metal Workers</li><li>519191: PRD-Adhesive Bonding Machine Operators And Tenders</li><li>533052: TRN-Bus Drivers, Transit And Intercity</li><li>5340XX: TRN-Other Rail Transportation Workers</li><li>5360XX: TRN-Other Transportation Workers</li><li>537021: TRN-Crane And Tower Operators</li><li>194010: SCI-Agricultural And Food Science Technicians</li><li>2911XX: MED-Nurse Practitioners, And Nurse Midwives</li><li>331021: PRT-First-Line Supervisors Of Firefighting And Prevention Workers</li><li>339030: PRT-Security Guards And Gambling Surveillance Officers</li><li>359011: EAT-Dining Room And Cafeteria Attendants And Bartender Helpers</li><li>474021: CON-Elevator And Escalator Installers And Repairers</li><li>111021: MGR-General And Operations Managers</li><li>112021: MGR-Marketing Managers</li><li>119013: MGR-Farmers, Ranchers, And Other Agricultural Managers</li><li>131070: BUS-Human Resources Workers</li><li>113012: MGR-Administrative Services Managers</li><li>113013: MGR-Facilities Managers</li><li>113021: MGR-Computer And Information Systems Managers</li><li>113051: MGR-Industrial Production Managers</li><li>113111: MGR-Compensation And Benefits Managers</li><li>113121: MGR-Human Resources Managers</li><li>119161: MGR-Emergency Management Directors</li><li>131030: BUS-Claims Adjusters, Appraisers, Examiners, And Investigators</li><li>131161: BUS-Market Research Analysts And Marketing Specialists</li><li>132052: FIN-Personal Financial Advisors</li><li>151253: CMM-Software Quality Assurance Analysts And Testers</li><li>173011: ENG-Architectural And Civil Drafters</li><li>191010: SCI-Agricultural And Food Scientists</li><li>191020: SCI-Biological Scientists</li><li>192010: SCI-Astronomers And Physicists</li><li>192041: SCI-Environmental Scientists And Specialists, Including Health</li><li>193034: SCI-School Psychologists</li><li>193051: SCI-Urban And Regional Planners</li><li>211021: CMS-Child, Family, And School Social Workers</li><li>21109X: CMS-Other Community And Social Service Specialists</li><li>2310XX: LGL-Lawyers, And Judges, Magistrates, And Other Judicial Workers</li><li>252030: EDU-Secondary School Teachers</li><li>272030: ENT-Dancers And Choreographers</li><li>272042: ENT-Musicians And Singers</li><li>273091: ENT-Interpreters And Translators</li><li>273092: ENT-Court Reporters And Simultaneous Captioners</li><li>291041: MED-Optometrists</li><li>291081: MED-Podiatrists</li><li>291122: MED-Occupational Therapists</li><li>29203X: MED-Nuclear Medicine Technologists And Medical Dosimetrists</li><li>292031: MED-Cardiovascular Technologists And Technicians</li><li>299000: MED-Other Healthcare Practitioners And Technical Occupations</li><li>372021: CLN-Pest Control Workers</li><li>37301X: CLN-Other Grounds Maintenance Workers</li><li>373011: CLN-Landscaping And Groundskeeping Workers</li><li>3930XX: PRS-Other Entertainment Attendants And Related Workers</li><li>395011: PRS-Barbers</li><li>395094: PRS-Skincare Specialists</li><li>4330XX: OFF-Other Financial Clerks</li><li>433051: OFF-Payroll And Timekeeping Clerks</li><li>434XXX: OFF-Correspondence Clerks And Order Clerks</li><li>434051: OFF-Customer Service Representatives</li><li>435053: OFF-Postal Service Mail Sorters, Processors, And Processing Machine Operators</li><li>436013: OFF-Medical Secretaries And Administrative Assistants</li><li>4520XX: FFF-Other Agricultural Workers</li><li>493021: RPR-Automotive Body And Related Repairers</li><li>493022: RPR-Automotive Glass Installers And Repairers</li><li>493040: RPR-Heavy Vehicle And Mobile Equipment Service Technicians And Mechanics</li><li>499098: RPR-Helpers--Installation, Maintenance, And Repair Workers</li><li>516021: PRD-Pressers, Textile, Garment, And Related Materials</li><li>516050: PRD-Tailors, Dressmakers, And Sewers</li><li>5170XX: PRD-Other Woodworkers</li><li>517021: PRD-Furniture Finishers</li><li>517041: PRD-Sawing Machine Setters, Operators, And Tenders, Wood</li><li>518010: PRD-Power Plant Operators, Distributors, And Dispatchers</li><li>5191XX: PRD-Miscellaneous Production Workers, Including Equipment Operators and Tenders</li><li>519111: PRD-Packaging And Filling Machine Operators And Tenders</li><li>533011: TRN-Ambulance Drivers And Attendants, Except Emergency Medical Technicians</li><li>533051: TRN-Bus Drivers, School</li><li>533053: TRN-Shuttle Drivers and Chauffeurs</li><li>5370XX: TRN-Conveyor, Dredge, And Hoist And Winch Operators</li><li>537062: TRN-Laborers And Freight, Stock, And Material Movers, Hand</li><li>537064: TRN-Packers And Packagers, Hand</li><li>151299: CMM-Computer Occupations, All Other</li><li>1910XX: SCI-Other Life Scientists</li><li>519198: PRD-Helpers--Production Workers</li><li>112030: MGR-Public Relations And Fundraising Managers</li><li>113071: MGR-Transportation, Storage, And Distribution Managers</li><li>119141: MGR-Property, Real Estate, And Community Association Managers</li><li>119151: MGR-Social And Community Service Managers</li><li>131021: BUS-Buyers And Purchasing Agents, Farm Products</li><li>131131: BUS-Fundraisers</li><li>132031: FIN-Budget Analysts</li><li>1720XX: ENG-Biomedical And Agricultural Engineers</li><li>172041: ENG-Chemical Engineers</li><li>173031: ENG-Surveying And Mapping Technicians</li><li>1940XX: SCI-Environmental Science And Geoscience Technicians, And Nuclear Technicians</li><li>211012: CMS-Educational, Guidance, And Career Counselors And Advisors</li><li>211013: CMS-Marriage And Family Therapists</li><li>211019: CMS-Counselors, All Other</li><li>212099: CMS-Religious Workers, All Other</li><li>252020: EDU-Elementary And Middle School Teachers</li><li>273011: ENT-Broadcast Announcers And Radio Disc Jockeys</li><li>291124: MED-Radiation Therapists</li><li>291181: MED-Audiologists</li><li>291292: MED-Dental Hygienists</li><li>292061: MED-Licensed Practical And Licensed Vocational Nurses</li><li>331012: PRT-First-Line Supervisors Of Police And Detectives</li><li>333012: PRT-Correctional Officers And Jailers</li><li>352010: EAT-Cooks</li><li>433061: OFF-Procurement Clerks</li><li>434031: OFF-Court, Municipal, And License Clerks</li><li>434121: OFF-Library Assistants, Clerical</li><li>434141: OFF-New Accounts Clerks</li><li>434171: OFF-Receptionists And Information Clerks</li><li>435051: OFF-Postal Service Clerks</li><li>439022: OFF-Word Processors And Typists</li><li>472011: CON-Boilermakers</li><li>472040: CON-Carpet, Floor, And Tile Installers And Finishers</li><li>472050: CON-Cement Masons, Concrete Finishers, And Terrazzo Workers</li><li>472121: CON-Glaziers</li><li>472231: CON-Solar Photovoltaic Installers</li><li>49209X: RPR-Other Electrical And Electronic Equipment Mechanics, Installers, and Repairers</li><li>493050: RPR-Small Engine Mechanics</li><li>4990XX: RPR-Other Installation, Maintenance, And Repair Workers</li><li>499071: RPR-Maintenance And Repair Workers, General</li><li>499094: RPR-Locksmiths And Safe Repairers</li><li>513091: PRD-Food And Tobacco Roasting, Baking, And Drying Machine Operators And Tenders</li><li>519030: PRD-Cutting Workers</li><li>519061: PRD-Inspectors, Testers, Sorters, Samplers, And Weighers</li><li>211029: CMS-Social Workers, All Other</li><li>435071: OFF-Shipping, Receiving, And Inventory Clerks</li><li>119121: MGR-Natural Sciences Managers</li><li>132070: FIN-Credit Counselors And Loan Officers</li><li>151241: CMM-Computer Network Architects</li><li>1520XX: CMM-Other Mathematical Science Occupations</li><li>152011: CMM-Actuaries</li><li>171020: ENG-Surveyors, Cartographers, And Photogrammetrists</li><li>172131: ENG-Materials Engineers</li><li>172141: ENG-Mechanical Engineers</li><li>193011: SCI-Economists</li><li>212011: CMS-Clergy</li><li>252010: EDU-Preschool And Kindergarten Teachers</li><li>2530XX: EDU-Other Teachers And Instructors</li><li>254022: EDU-Librarians And Media Collections Specialists</li><li>271010: ENT-Artists And Related Workers</li><li>271024: ENT-Graphic Designers</li><li>272011: ENT-Actors</li><li>272021: ENT-Athletes And Sports Competitors</li><li>272023: ENT-Umpires, Referees, And Other Sports Officials</li><li>273041: ENT-Editors</li><li>291011: MED-Chiropractors</li><li>291031: MED-Dietitians And Nutritionists</li><li>291051: MED-Pharmacists</li><li>291071: MED-Physician Assistants</li><li>29112X: MED-Other Therapists</li><li>291125: MED-Recreational Therapists</li><li>291126: MED-Respiratory Therapists</li><li>291151: MED-Nurse Anesthetists</li><li>292034: MED-Radiologic Technologists And Technicians</li><li>312010: HLS-Occupational Therapy Assistants And Aides</li><li>319011: HLS-Massage Therapists</li><li>339021: PRT-Private Detectives And Investigators</li><li>33909X: PRT-Other Protective Service Workers</li><li>353011: EAT-Bartenders</li><li>372012: CLN-Maids And Housekeeping Cleaners</li><li>373013: CLN-Tree Trimmers And Pruners</li><li>397010: PRS-Tour And Travel Guides</li><li>399011: PRS-Childcare Workers</li><li>414010: SAL-Sales Representatives, Wholesale And Manufacturing</li><li>419010: SAL-Models, Demonstrators, And Product Promoters</li><li>419091: SAL-Door-To-Door Sales Workers, News And Street Vendors, And Related Workers</li><li>433011: OFF-Bill And Account Collectors</li><li>433031: OFF-Bookkeeping, Accounting, And Auditing Clerks</li><li>435032: OFF-Dispatchers, Except Police, Fire, And Ambulance</li><li>439021: OFF-Data Entry Keyers</li><li>472152: CON-Plumbers, Pipefitters, And Steamfitters</li><li>472161: CON-Plasterers And Stucco Masons</li><li>472181: CON-Roofers</li><li>474011: CON-Construction And Building Inspectors</li><li>491011: RPR-First-Line Supervisors Of Mechanics, Installers, And Repairers</li><li>492011: RPR-Computer, Automated Teller, And Office Machine Repairers</li><li>492091: RPR-Avionics Technicians</li><li>499043: RPR-Maintenance Workers, Machinery</li><li>499051: RPR-Electrical Power-Line Installers And Repairers</li><li>5120XX: PRD-Other Assemblers and Fabricators</li><li>513093: PRD-Food Cooking Machine Operators And Tenders</li><li>514020: PRD-Forming Machine Setters, Operators, And Tenders, Metal And Plastic</li><li>514031: PRD-Cutting, Punching, And Press Machine Setters, Operators, And Tenders, Metal And Plastic</li><li>515111: PRD-Prepress Technicians And Workers</li><li>519196: PRD-Paper Goods Machine Setters, Operators, And Tenders</li><li>531000: TRN-Supervisors Of Transportation And Material Moving Workers</li><li>533054: TRN-Taxi Drivers</li><li>536030: TRN-Transportation Service Attendants</li><li>537070: TRN-Pumping Station Operators</li><li>553010: MIL-Military Enlisted Tactical Operations And Air/Weapons Specialists And Crew Members</li><li>132051: FIN-Financial And Investment Analysts</li><li>436014: OFF-Secretaries And Administrative Assistants, Except Legal, Medical, And Executive</li><li>2740XX: ENT-Other Media And Communication Equipment Workers</li><li>291240: MED-Surgeons</li><li>291291: MED-Acupuncturists</li><li>292055: MED-Surgical Technologists</li><li>292072: MED-Medical Records Specialists</li><li>292081: MED-Opticians, Dispensing</li><li>311121: HLS-Home Health Aides</li><li>319095: HLS-Pharmacy Aides</li><li>331090: PRT-Miscellaneous First-Line Supervisors, Protective Service Workers</li><li>333011: PRT-Bailiffs</li><li>333050: PRT-Police Officers</li><li>339093: PRT-Transportation Security Screeners</li><li>353031: EAT-Waiters And Waitresses</li><li>359021: EAT-Dishwashers</li><li>371012: CLN-First-Line Supervisors Of Landscaping, Lawn Service, And Groundskeeping Workers</li><li>396010: PRS-Baggage Porters, Bellhops, And Concierges</li><li>399031: PRS-Exercise Trainers And Group Fitness Instructors</li><li>399099: PRS-Personal Care And Service Workers, All Other</li><li>412010: SAL-Cashiers</li><li>413021: SAL-Insurance Sales Agents</li><li>432021: OFF-Telephone Operators</li><li>432099: OFF-Communications Equipment Operators, All Other</li><li>434YYY: OFF-Other Information And Records Clerks</li><li>434081: OFF-Hotel, Motel, And Resort Desk Clerks</li><li>434181: OFF-Reservation And Transportation Ticket Agents And Travel Clerks</li><li>436011: OFF-Executive Secretaries And Executive Administrative Assistants</li><li>439071: OFF-Office Machine Operators, Except Computer</li><li>452041: FFF-Graders And Sorters, Agricultural Products</li><li>454020: FFF-Logging Workers</li><li>474051: CON-Highway Maintenance Workers</li><li>511011: PRD-First-Line Supervisors Of Production And Operating Workers</li><li>512041: PRD-Structural Metal Fabricators And Fitters</li><li>515113: PRD-Print Binding And Finishing Workers</li><li>516031: PRD-Sewing Machine Operators</li><li>516093: PRD-Upholsterers</li><li>518021: PRD-Stationary Engineers And Boiler Operators</li><li>519010: PRD-Chemical Processing Machine Setters, Operators, And Tenders</li><li>519051: PRD-Furnace, Kiln, Oven, Drier, And Kettle Operators And Tenders</li><li>519120: PRD-Painting Workers</li><li>519151: PRD-Photographic Process Workers And Processing Machine Operators</li><li>519194: PRD-Etchers And Engravers</li><li>532020: TRN-Air Traffic Controllers And Airfield Operations Specialists</li><li>532031: TRN-Flight Attendants</li><li>537065: TRN-Stockers And Order Fillers</li><li>5371XX: TRN-Other Material Moving Workers</li><li>559830: MIL-Military, Rank Not Specified</li><li>131121: BUS-Meeting, Convention, And Event Planners</li><li>359099: EAT-Food Preparation And Serving Related Workers, All Other</li><li>514033: PRD-Grinding, Lapping, Polishing, And Buffing Machine Tool Setters, Operators, And Tenders, Metal and Plastic</li><li>999920: Unemployed, With No Work Experience In The Last 5 Years Or Earlier Or Never Worked</li><li>113031: MGR-Financial Managers</li><li>119041: MGR-Architectural And Engineering Managers</li><li>131081: BUS-Logisticians</li><li>131082: BUS-Project Management Specialists</li><li>131151: BUS-Training And Development Specialists</li><li>132011: FIN-Accountants And Auditors</li><li>151230: CMM-Computer Support Specialists</li><li>15124X: CMM-Database Administrators And Architects</li><li>152031: CMM-Operations Research Analysts</li><li>171012: ENG-Landscape Architects</li><li>172051: ENG-Civil Engineers</li><li>172061: ENG-Computer Hardware Engineers</li><li>172110: ENG-Industrial Engineers, Including Health And Safety</li><li>192030: SCI-Chemists And Materials Scientists</li><li>1940YY: SCI-Other Life, Physical, And Social Science Technicians</li><li>211022: CMS-Healthcare Social Workers</li><li>254010: EDU-Archivists, Curators, And Museum Technicians</li><li>272022: ENT-Coaches And Scouts</li><li>272041: ENT-Music Directors And Composers</li><li>1110XX: MGR-Chief Executives And Legislators</li><li>112022: MGR-Sales Managers</li><li>113061: MGR-Purchasing Managers</li><li>113131: MGR-Training And Development Managers</li><li>119030: MGR-Education And Childcare Administrators</li><li>132020: FIN-Property Appraisers And Assessors</li><li>132053: FIN-Insurance Underwriters</li><li>132061: FIN-Financial Examiners</li><li>1721YY: ENG-Other Engineers</li><li>17302X: ENG-Other Engineering Technologists And Technicians, Except Drafters</li><li>192099: SCI-Physical Scientists, All Other</li><li>194031: SCI-Chemical Technicians</li><li>195010: SCI-Occupational Health And Safety Specialists And Technicians</li><li>211092: CMS-Probation Officers And Correctional Treatment Specialists</li><li>254031: EDU-Library Technicians</li><li>259040: EDU-Teaching Assistants</li><li>271022: ENT-Fashion Designers</li><li>272091: ENT-Disc Jockeys, Except Radio</li><li>273023: ENT-News Analysts, Reporters, And Journalists</li><li>291020: MED-Dentists</li><li>291127: MED-Speech-Language Pathologists</li><li>291131: MED-Veterinarians</li><li>291141: MED-Registered Nurses</li><li>291210: MED-Physicians</li><li>292032: MED-Diagnostic Medical Sonographers</li><li>292043: MED-Paramedics</li><li>29205X: MED-Dietetic Technicians And Ophthalmic Medical Technicians</li><li>292052: MED-Pharmacy Technicians</li><li>31113X: HLS-Orderlies And Psychiatric Aides</li><li>319091: HLS-Dental Assistants</li><li>332020: PRT-Fire Inspectors</li><li>333021: PRT-Detectives And Criminal Investigators</li><li>351011: EAT-Chefs And Head Cooks</li><li>352021: EAT-Food Preparation Workers</li><li>392011: PRS-Animal Trainers</li><li>392021: PRS-Animal Caretakers</li><li>411011: SAL-First-Line Supervisors Of Retail Sales Workers</li><li>413011: SAL-Advertising Sales Agents</li><li>413031: SAL-Securities, Commodities, And Financial Services Sales Agents</li><li>419020: SAL-Real Estate Brokers And Sales Agents</li><li>434061: OFF-Eligibility Interviewers, Government Programs</li><li>434071: OFF-File Clerks</li><li>435041: OFF-Meter Readers, Utilities</li><li>439061: OFF-Office Clerks, General</li><li>439081: OFF-Proofreaders And Copy Markers</li><li>451011: FFF-First-Line Supervisors Of Farming, Fishing, And Forestry Workers</li><li>454011: FFF-Forest And Conservation Workers</li><li>472XXX: CON-Brickmasons, Blockmasons, Stonemasons, And Reinforcing Iron And Rebar Workers</li><li>472061: CON-Construction Laborers</li><li>472070: CON-Construction Equipment Operators</li><li>472211: CON-Sheet Metal Workers</li><li>499031: RPR-Home Appliance Repairers</li><li>49904X: RPR-Industrial And Refractory Machinery Mechanics</li><li>499044: RPR-Millwrights</li><li>513020: PRD-Butchers And Other Meat, Poultry, And Fish Processing Workers</li><li>514XXX: PRD-Other Metal Workers And Plastic Workers</li><li>514041: PRD-Machinists</li><li>516060: PRD-Textile Machine Setters, Operators, And Tenders</li><li>51609X: PRD-Other Textile, Apparel, And Furnishings Workers</li><li>519160: PRD-Computer Numerically Controlled Tool Operators And Programmers</li><li>519195: PRD-Molders, Shapers, And Casters, Except Metal And Plastic</li><li>533030: TRN-Driver/Sales Workers And Truck Drivers</li><li>533099: TRN-Motor Vehicle Operators, All Other</li><li>534031: TRN-Railroad Conductors And Yardmasters</li><li>5350XX: TRN-Sailors And Marine Oilers, And Ship Engineers</li><li>537061: TRN-Cleaners Of Vehicles And Equipment</li><li>-1: N/A (less than 16 years old/NILF who last worked more than 5 years ago or never worked)</li><li>151255: CMM-Web And Digital Interface Designers</li><li>271026: ENT-Merchandise Displayer And Windows Trimmers</li><li>3940XX: PRS-Embalmers, Crematory Operators And Funeral Attendants</li><li>475020: EXT-Surface Mining Machine Operators And Earth Drillers</li><li>536021: TRN-Parking Attendants</li><li>391000: PRS-Supervisors of Personal Care And Service Workers</li><li>393031: PRS-Ushers, Lobby Attendants, And Ticket Takers</li><li>39509X: PRS-Other Personal Appearance Workers</li><li>395092: PRS-Manicurists and Pedicurists</li><li>399041: PRS-Residential Advisors</li><li>412031: SAL-Retail Salespersons</li><li>413041: SAL-Travel Agents</li><li>419031: SAL-Sales Engineers</li><li>433071: OFF-Tellers</li><li>434041: OFF-Credit Authorizers, Checkers, And Clerks</li><li>434111: OFF-Interviewers, Except Eligibility And Loan</li><li>434131: OFF-Loan Interviewers And Clerks</li><li>435011: OFF-Cargo And Freight Agents</li><li>435031: OFF-Public Safety Telecommunicators</li><li>435052: OFF-Postal Service Mail Carriers</li><li>439041: OFF-Insurance Claims And Policy Processing Clerks</li><li>439111: OFF-Statistical Assistants</li><li>452011: FFF-Agricultural Inspectors</li><li>471011: CON-First-Line Supervisors Of Construction Trades And Extraction Workers</li><li>472140: CON-Painters And Paperhangers</li><li>473010: CON-Helpers, Construction Trades</li><li>474031: CON-Fence Erectors</li><li>474041: CON-Hazardous Materials Removal Workers</li><li>4750XX: EXT-Other Extraction Workers</li><li>475032: EXT-Explosives Workers, Ordnance Handling Experts, And Blasters</li><li>475040: EXT-Underground Mining Machine Operators</li><li>492092: RPR-Electric Motor, Power Tool, And Related Repairers</li><li>493090: RPR-Miscellaneous Vehicle And Mobile Equipment Mechanics, Installers, And Repairers</li><li>499091: RPR-Coin, Vending, And Amusement Machine Servicers And Repairers</li><li>512020: PRD-Electrical, Electronics, And Electromechanical Assemblers</li><li>513092: PRD-Food Batchmakers</li><li>5140XX: PRD-Model Makers, Patternmakers, And Molding Machine Setters, Metal And Plastic</li><li>514050: PRD-Metal Furnace Operators, Tenders, Pourers, And Casters</li><li>516011: PRD-Laundry And Dry-Cleaning Workers</li><li>532010: TRN-Aircraft Pilots And Flight Engineers</li><li>534010: TRN-Locomotive Engineers And Operators</li><li>535020: TRN-Ship And Boat Captains And Operators</li><li>536061: TRN-Passenger Attendants</li><li>537081: TRN-Refuse And Recyclable Material Collectors</li><li>552010: MIL-First-Line Enlisted Military Supervisors</li><li>151254: CMM-Web Developers</li><li>19204X: SCI-Geoscientists And Hydrologists, Except Geographers</li><li>431011: OFF-First-Line Supervisors Of Office And Administrative Support Workers</li><li>492097: RPR-Audiovisual Equipment Installers And Repairers</li><li>51403X: PRD-Other Machine Tool Setters, Operators, And Tenders, Metal And Plastic</li><li>519080: PRD-Dental And Ophthalmic Laboratory Technicians And Medical Appliance Technicians</li><li>112011: MGR-Advertising And Promotions Managers</li><li>119051: MGR-Food Service Managers</li><li>1191XX: MGR-Other Managers</li><li>119111: MGR-Medical And Health Services Managers</li><li>131022: BUS-Wholesale And Retail Buyers, Except Farm Products</li><li>131041: BUS-Compliance Officers</li><li>131141: BUS-Compensation, Benefits, And Job Analysis Specialists</li><li>132041: FIN-Credit Analysts</li><li>132082: FIN-Tax Preparers</li><li>151211: CMM-Computer Systems Analysts</li><li>151212: CMM-Information Security Analysts</li><li>151244: CMM-Network And Computer Systems Administrators</li><li>171011: ENG-Architects, Except Landscape And Naval</li><li>172011: ENG-Aerospace Engineers</li><li>172070: ENG-Electrical And Electronics Engineers</li><li>172081: ENG-Environmental Engineers</li><li>19303X: SCI-Other Psychologists</li><li>211093: CMS-Social And Human Service Assistants</li><li>231012: LGL-Judicial Law Clerks</li><li>232099: LGL-Legal Support Workers, All Other</li><li>252050: EDU-Special Education Teachers</li><li>253041: EDU-Tutors</li><li>27102X: ENT-Other Designers</li><li>271021: ENT-Commercial And Industrial Designers</li><li>272012: ENT-Producers And Directors</li><li>273042: ENT-Technical Writers</li><li>273043: ENT-Writers And Authors</li><li>292056: MED-Veterinary Technologists And Technicians</li><li>311131: HLS-Nursing Assistants</li><li>312020: HLS-Physical Therapist Assistants And Aides</li><li>319092: HLS-Medical Assistants</li><li>319097: HLS-Phlebotomists</li><li>3330XX: PRT-Fish And Game Wardens And Parking Enforcement Officers</li><li>339011: PRT-Animal Control Workers</li><li>339091: PRT-Crossing Guards And Flaggers</li><li>353023: EAT-Fast Food And Counter Workers</li></ul>"""

# Parse the text data into a dictionary
soc_mapping_census = {}
lines = re.findall(r"(\d+[Xx]*):\s*([\w\s,\-\(\)&]+)", text)
# Replace 'X' or 'x' in SOC codes with '0' and sort by SOC code
#lines = sorted([(code.replace('X', '0').replace('x', '0'), name) for code, name in lines], key=lambda x: int(x[0]))

for soc_code, occupation in lines:
    soc_mapping_census[soc_code] = occupation
    
df_soc = pd.DataFrame(list(soc_mapping_census.items()), columns=["SOC", "SOCName"])
df_soc['SOCName'] = df_soc['SOCName'].astype(str).apply(format_group_name)  # Apply formatting

# Add a '-' after the first two digits
df_soc["SOC"] = df_soc["SOC"].str.replace(r"^(\d{2})(\d+)", r"\1-\2", regex=True)


In [ ]:
df_occ2 = pd.read_excel('/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/occsoc_crosswalk_2000_onward.xlsx') 

# Drop rows where all columns except 'Unnamed: 0' are NaN
df_occ2 = df_occ2.dropna(subset=df_occ2.columns.difference(['Unnamed: 0']), how='all')

# Drop the 'Unnamed: 0' column
df_occ2 = df_occ2.drop(columns=['Unnamed: 0'])

# Create a new column 'SOC' with the last non-NaN value in each row
df_occ2['SOC'] = df_occ2.iloc[:, 1:-1].apply(lambda row: row.dropna().iloc[-1] if not row.dropna().empty else None, axis=1)

# Ensure SOC Codes are strings and replace any 'X' with '0'
df_occ2['SOC'] = df_occ2['SOC'].astype(str).str.replace('X', '0')

# Add a '-' after the first two digits
df_occ2["SOC"] = df_occ2["SOC"].str.replace(r"^(\d{2})(\d+)", r"\1-\2", regex=True)

df_occ2.rename(
            columns={'Occupation title': 'SOCName'}, inplace=True)
df_occ2['SOCName'] = df_occ2['SOCName'].str.title()  # Word capitalize titles


In [ ]:
df_occ3 = pd.read_excel('/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/census_soc.xlsx') 


In [ ]:
df_census = pd.concat([df_soc, df_occ2[['SOC', 'SOCName']]],
    ignore_index=True)
df_census = pd.concat([df_census, df_occ3],
    ignore_index=True)

In [ ]:
df_census = clean_soc_mapping(df_census)
df_mapping = clean_soc_mapping(df_mapping)


In [ ]:
df_merge=pd.merge(df_census, df_mapping.drop(columns=['SOCName']), on='SOC', how='left', indicator=True)

In [ ]:
difference=pd.merge(df_merge[df_merge['_merge']=='left_only'].drop(columns=['_merge']), df_mapping, on='SOCName', how='left', indicator=True)

In [ ]:
df_aux = pd.concat(
    [
        df_merge[df_merge['_merge'] == 'both'].drop(columns=['_merge']),
        difference[difference['_merge'] == 'both']
        .drop(columns=['SOC_x', '_merge'])
        .rename(columns={'SOC_y': 'SOC'})
    ],
    ignore_index=True
)


In [ ]:
df_difference=difference[difference['_merge']=='left_only'].drop(columns=['_merge','SOC_y']).rename(columns={'SOC_x':'SOC'})

In [ ]:
merged_df = pd.merge(
    df_difference, 
    df_mapping, 
    left_on=df_difference['SOC'].str[:6], 
    right_on=df_mapping['SOC'].str[:6],
    how='left', 
    indicator=True
)

In [ ]:
# Step 3: Function to check if two columns share at least two words
def shares_two_words(row):
    col1_words = set(str(row['SOCName_x']).lower().split()) if pd.notna(row['SOCName_x']) else set()
    col2_words = set(str(row['SOCName_y']).lower().split()) if pd.notna(row['SOCName_y']) else set()
    return len(col1_words & col2_words) >= 2  # Check for at least 2 shared words


# Step 4: Apply the word-sharing logic to filter matches
merged_df['SharedWords'] = merged_df.apply(shares_two_words, axis=1)


In [ ]:
filtered_df = merged_df[merged_df['SharedWords']].drop(columns=['_merge', 'key_0', 'SharedWords'])


In [ ]:
df_aux = pd.concat(
    [
        df_aux,
        filtered_df[['SOCName_x','SOC_y']]
        .rename(columns={'SOCName_x':'SOCName','SOC_y':'SOC'})
    ],
    ignore_index=True
)


In [ ]:
df_mapping = pd.concat(
    [
        df_mapping,
        df_census
    ],
    ignore_index=True
)

df_mapping = pd.concat(
    [
        df_mapping,
        df_aux
    ],
    ignore_index=True
)


In [ ]:
df_mapping=df_mapping.drop_duplicates()

In [ ]:
unfiltered_df = merged_df[~merged_df['SharedWords']].drop(columns=['key_0', 'SharedWords'])


In [ ]:
# Step 1: Extract the generic SOC codes from the specific SOC codes
df_mapping['Generic_SOC'] = df_mapping['SOC'].str[:6]

# Step 2: Group specific SOC codes by their corresponding generic code
soc_mapping = df_mapping.groupby('Generic_SOC')['SOC'].apply(list).to_dict()
df_mapping.drop(columns=['Generic_SOC'], inplace=True)

# Step 3: Merge the mapping back to the generic SOC dataframe
unfiltered_df['Generic_SOC'] = unfiltered_df['SOC_x'].str[:6]
# Step 3: Map the SOC codes to the generic SOC codes in df_generic
unfiltered_df['Specific_SOCs'] = unfiltered_df['Generic_SOC'].map(soc_mapping)

# Step 1: Explode the `Specific_SOCs` column to duplicate rows by each specific SOC
unfiltered_df = unfiltered_df.explode('Specific_SOCs')

# Step 2: Drop duplicate rows based on the `Generic_SOC` and `Specific_SOCs` columns
unfiltered_df = unfiltered_df.drop_duplicates(subset=['Generic_SOC', 'Specific_SOCs']).reset_index(drop=True)


In [ ]:
df_mapping.head()

In [ ]:
df_aux=unfiltered_df[unfiltered_df['_merge']=='both'][['SOCName_x','Specific_SOCs']].rename(columns={'SOCName_x':'SOCName','Specific_SOCs':'SOC'})

In [ ]:
df_mapping = pd.concat(
    [
        df_mapping,
        df_aux
    ],
    ignore_index=True
)
df_mapping=df_mapping.drop_duplicates()

In [ ]:
def find_best_match(name, reference_list):
        # Find the best match for the SOCName in the reference list
        match = difflib.get_close_matches(name, reference_list, n=1, cutoff=0.6)
        return match[0] if match else name

    

In [ ]:
# Data construction

In [ ]:
directory='/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/Gender/'

state_mapping = {
    "Total": "00","Alabama": "01","Alaska": "02",
    "Arizona": "04","Arkansas": "05","California": "06","Colorado": "08","Connecticut": "09","Delaware": "10",
    "District of Columbia": "11","Florida": "12","Georgia": "13","Hawaii": "15","Idaho": "16","Illinois": "17","Indiana": "18","Iowa": "19",
    "Kansas": "20","Kentucky": "21","Louisiana": "22","Maine": "23","Maryland": "24","Massachusetts": "25",
    "Michigan": "26","Minnesota": "27","Mississippi": "28","Missouri": "29","Montana": "30","Nebraska": "31",
    "Nevada": "32","New Hampshire": "33","New Jersey": "34","New Mexico": "35","New York": "36","North Carolina": "37",
    "North Dakota": "38","Ohio": "39","Oklahoma": "40","Oregon": "41","Pennsylvania": "42","Rhode Island": "44",
    "South Carolina": "45","South Dakota": "46","Tennessee": "47","Texas": "48","Utah": "49","Vermont": "50",
    "Virginia": "51","Washington": "53","West Virginia": "54","Wisconsin": "55","Wyoming": "56","Puerto Rico": "72","Virgin Islands": "78"
}

In [ ]:
df_gender=pd.DataFrame()
# Iterate over all CSV files in the directory
for filename in os.listdir(directory):
    if filename.endswith('.csv'):
        
        # Read the CSV file, skipping the first 56 rows
        df_aux = pd.read_csv(os.path.join(directory, filename), skiprows=56)
        
        # Create a 'Year' column with the extracted year
        df_aux['Year'] = filename[9:13]
        
        # Forward-fill the 'Group' column to propagate the values
        df_aux['Group'] = df_aux['Group'].fillna(method='ffill')
        
        # Remove rows where all other columns (besides 'Group') are NaN
        df_aux = df_aux.dropna(subset=df_aux.columns.difference(['Group']), how='all').reset_index(drop=True)
        
        # Rename the column 'Selected Geographies' to 'FIPSState'
        df_aux.rename(columns={'Selected Geographies': 'FIPSState'}, inplace=True)
        
        # Map the state names to FIPS codes (ensure 'state_mapping' is defined)
        df_aux['FIPSState'] = df_aux['FIPSState'].map(state_mapping)
        
        # Apply cleaning and formatting to the 'Group' column
        df_aux['Group'] = df_aux['Group'].astype(str).str.strip()  # Trim leading/trailing spaces
        df_aux['Group'] = df_aux['Group'].str.replace('->', '', regex=False)  # Remove '->'
        df_aux['Group'] = df_aux['Group'].str.replace('*', '', regex=False)  # Remove '*'
        df_aux['Group'] = df_aux['Group'].replace('--', '-', regex=True)
        df_aux = df_aux.replace(',', '', regex=True)  # Remove commas from values
        df_aux['Group'] = df_aux['Group'].astype(str).apply(format_group_name)  # Apply formatting
        df_aux['Group'] = df_aux['Group'].str.replace(
            r'(?<=[-/])([a-z])', 
            lambda match: match.group(1).upper(),
            regex=True
        )
        df_aux['Group'] = df_aux['Group'].astype(str).str.strip()  # Trim leading/trailing spaces
        
        # Rename 'Group' column to 'SOCName'
        df_aux.rename(columns={'Group': 'SOCName'}, inplace=True)
        
        # Replace specific SOCName directly
        df_aux['SOCName'] = df_aux['SOCName'].replace(
            'Software Developersapplications And Systems Software',
            'Software Developers Applications And Systems Software'
        )
        df_aux['SOCName'] = df_aux['SOCName'].replace(
            'Merchandise Displayer And Windows Trimmers',
            'Merchandise Displayers And Windows Trimmers'
        )
        df_aux['SOCName'] = df_aux['SOCName'].replace(
            'Merchandise Displayers And Window Trimmers',
            'Merchandise Displayers And Windows Trimmers'
        )
        # Merge with df_mapping on 'SOCName' (ensure 'df_mapping' is defined)
        df_aux = pd.merge(
            df_aux, df_mapping, on=['SOCName'], how='left', indicator=True
        )
        
    # Append the processed DataFrame to the list
    df_gender=pd.concat([df_gender, df_aux],
    ignore_index=True)



In [ ]:
df_aux=df_gender[df_gender['_merge']=='left_only']

In [ ]:
# Apply the matching to SOCName column
# Step 1: Get unique SOCName values from df_aux
unique_socnames = df_aux['SOCName'].unique()

# Step 2: Create a mapping for the unique SOCName values
unique_socname_mapping = {
    socname: find_best_match(socname, df_mapping['SOCName']) for socname in unique_socnames
}

# Step 3: Map the results back to the entire DataFrame
df_aux['SOCName'] = df_aux['SOCName'].map(unique_socname_mapping)

In [ ]:
df_gender = df_gender[df_gender['_merge'] != 'left_only']
df_aux = df_aux.drop(columns=['SOC', '_merge'], errors='ignore')


In [ ]:
# Merge with df_mapping on 'SOCName' (ensure 'df_mapping' is defined)
df_aux = pd.merge(
            df_aux, df_mapping, on=['SOCName'], how='left', indicator=True
        )
df_aux = df_aux[df_aux['_merge'] != 'left_only']
df_gender=pd.concat([df_gender, df_aux],ignore_index=True)
df_gender = df_gender.drop(columns=['_merge'], errors='ignore')
df_gender = df_gender[~pd.isna(df_gender['FIPSState'])]


In [ ]:
exclude_columns = ['SOCName', 'SOC']
df_gender.columns = df_gender.columns.str.replace('<', '')

# Convert all columns except the excluded ones to integers
for col in df_gender.columns:
    if col not in exclude_columns:
        df_gender[col] = pd.to_numeric(df_gender[col], errors='coerce').fillna(-999).astype(int)

In [ ]:
df_gender['Male'] = (df_gender['Male'] / df_gender['Total Sex (SEX)']) * 100
df_gender['Female'] = (df_gender['Female'] / df_gender['Total Sex (SEX)']) * 100
df_gender = df_gender.drop(columns=['Total Sex (SEX)'], errors='ignore')


In [ ]:
directory='/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/Race/'

df_race=pd.DataFrame()
# Iterate over all CSV files in the directory
for filename in os.listdir(directory):
    if filename.endswith('.csv'):
        
        # Read the CSV file, skipping the first 56 rows
        df_aux = pd.read_csv(os.path.join(directory, filename), skiprows=56)
        
        # Create a 'Year' column with the extracted year
        df_aux['Year'] = filename[9:13]
        
        # Forward-fill the 'Group' column to propagate the values
        df_aux['Group'] = df_aux['Group'].fillna(method='ffill')
        
        # Remove rows where all other columns (besides 'Group') are NaN
        df_aux = df_aux.dropna(subset=df_aux.columns.difference(['Group']), how='all').reset_index(drop=True)
        
        # Rename the column 'Selected Geographies' to 'FIPSState'
        df_aux.rename(columns={'Selected Geographies': 'FIPSState'}, inplace=True)
        
        # Map the state names to FIPS codes (ensure 'state_mapping' is defined)
        df_aux['FIPSState'] = df_aux['FIPSState'].map(state_mapping)
        
        # Apply cleaning and formatting to the 'Group' column
        df_aux['Group'] = df_aux['Group'].astype(str).str.strip()  # Trim leading/trailing spaces
        df_aux['Group'] = df_aux['Group'].str.replace('->', '', regex=False)  # Remove '->'
        df_aux['Group'] = df_aux['Group'].str.replace('*', '', regex=False)  # Remove '*'
        df_aux['Group'] = df_aux['Group'].replace('--', '-', regex=True)
        df_aux = df_aux.replace(',', '', regex=True)  # Remove commas from values
        df_aux['Group'] = df_aux['Group'].astype(str).apply(format_group_name)  # Apply formatting
        df_aux['Group'] = df_aux['Group'].str.replace(
            r'(?<=[-/])([a-z])', 
            lambda match: match.group(1).upper(),
            regex=True
        )
        df_aux['Group'] = df_aux['Group'].astype(str).str.strip()  # Trim leading/trailing spaces
        
        # Rename 'Group' column to 'SOCName'
        df_aux.rename(columns={'Group': 'SOCName'}, inplace=True)
        
        # Replace specific SOCName directly
        df_aux['SOCName'] = df_aux['SOCName'].replace(
            'Software Developersapplications And Systems Software',
            'Software Developers Applications And Systems Software'
        )
        df_aux['SOCName'] = df_aux['SOCName'].replace(
            'Merchandise Displayer And Windows Trimmers',
            'Merchandise Displayers And Windows Trimmers'
        )
        df_aux['SOCName'] = df_aux['SOCName'].replace(
            'Merchandise Displayers And Window Trimmers',
            'Merchandise Displayers And Windows Trimmers'
        )
        # Merge with df_mapping on 'SOCName' (ensure 'df_mapping' is defined)
        df_aux = pd.merge(
            df_aux, df_mapping, on=['SOCName'], how='left', indicator=True
        )
        
    # Append the processed DataFrame to the list
    df_race=pd.concat([df_race, df_aux],
    ignore_index=True)


In [ ]:
df_aux=df_race[df_race['_merge']=='left_only']
# Apply the matching to SOCName column
# Step 1: Get unique SOCName values from df_aux
unique_socnames = df_aux['SOCName'].unique()

# Step 2: Create a mapping for the unique SOCName values
unique_socname_mapping = {
    socname: find_best_match(socname, df_mapping['SOCName']) for socname in unique_socnames
}

# Step 3: Map the results back to the entire DataFrame
df_aux['SOCName'] = df_aux['SOCName'].map(unique_socname_mapping)

In [ ]:
df_race = df_race[df_race['_merge'] != 'left_only']
df_aux = df_aux.drop(columns=['SOC', '_merge'], errors='ignore')


In [ ]:
# Merge with df_mapping on 'SOCName' (ensure 'df_mapping' is defined)
df_aux = pd.merge(
            df_aux, df_mapping, on=['SOCName'], how='left', indicator=True
        )

In [ ]:
df_aux = df_aux[df_aux['_merge'] != 'left_only']
df_race=pd.concat([df_race, df_aux],ignore_index=True)
df_race = df_race.drop(columns=['_merge'], errors='ignore')
df_race = df_race[~pd.isna(df_race['FIPSState'])]

In [ ]:
exclude_columns = ['SOCName', 'SOC']
df_race.columns = df_race.columns.str.replace('<', '')

# Convert all columns except the excluded ones to integers
for col in df_race.columns:
    if col not in exclude_columns:
        df_race[col] = pd.to_numeric(df_race[col], errors='coerce').fillna(-999).astype(int)

In [ ]:
# Identify columns containing "RAC1P"
rac1p_columns = [col for col in df_race.columns if 'RAC1P' in col]

# Combine these columns into one named 'Total' by summing non-null values
df_race['Total'] = df_race[rac1p_columns].max(axis=1)
df_race['someOtherRace'] = df_race[['Some Other Race alone', 'Some other race alone']].max(axis=1)

# Drop the original RAC1P columns if no longer needed
df_race = df_race.drop(columns=rac1p_columns+ ['Some Other Race alone', 'Some other race alone'], errors='ignore')


In [ ]:
# Rename columns to simplify and homogenize names
rename_mapping = {
    'White alone': 'white',
    'Black or African American alone': 'black',
    'American Indian alone': 'americanIndian',
    'Alaska Native alone': 'alaskaNative',
    'Asian alone': 'asian',
    'Native Hawaiian and Other Pacific Islander alone': 'hawaiianPacificIslander',
    'someOtherRace': 'someOtherRace'
}

# Apply renaming
df_race.rename(columns=rename_mapping, inplace=True)

In [ ]:

# List of columns to process
columns_to_process = list(rename_mapping.values())

# Divide specified columns by 'Total' and multiply by 100
for col in columns_to_process:
    df_race[col] = (df_race[col] / df_race['Total']) * 100

df_race = df_race.loc[:, df_race.columns.isin(['Year', 'SOC', 'SOCName', 'FIPSState'] + columns_to_process)]


In [ ]:
df_dem=pd.merge(
            df_gender, df_race, on=['SOC','SOCName','FIPSState','Year'], how='outer', indicator=True
        )

In [ ]:
df_dem = df_dem.drop(columns=['_merge'], errors='ignore')


In [ ]:
## Age

In [ ]:
directory='/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/Age/'

df_age=pd.DataFrame()
# Iterate over all CSV files in the directory
for filename in os.listdir(directory):
    if filename.endswith('.csv'):
        
        # Read the CSV file, skipping the first 56 rows
        df_aux = pd.read_csv(os.path.join(directory, filename), skiprows=55)
        
        # Create a 'Year' column with the extracted year
        df_aux['Year'] = filename[9:13]
        
        # Forward-fill the 'Group' column to propagate the values
        df_aux['Group'] = df_aux['Group'].fillna(method='ffill')
        
        # Remove rows where all other columns (besides 'Group') are NaN
        df_aux = df_aux.dropna(subset=df_aux.columns.difference(['Group']), how='all').reset_index(drop=True)
        
        # Rename the column 'Selected Geographies' to 'FIPSState'
        df_aux.rename(columns={'Selected Geographies': 'FIPSState'}, inplace=True)
        
        # Map the state names to FIPS codes (ensure 'state_mapping' is defined)
        df_aux['FIPSState'] = df_aux['FIPSState'].map(state_mapping)
        
        # Apply cleaning and formatting to the 'Group' column
        df_aux['Group'] = df_aux['Group'].astype(str).str.strip()  # Trim leading/trailing spaces
        df_aux['Group'] = df_aux['Group'].str.replace('->', '', regex=False)  # Remove '->'
        df_aux['Group'] = df_aux['Group'].str.replace('*', '', regex=False)  # Remove '*'
        df_aux['Group'] = df_aux['Group'].replace('--', '-', regex=True)
        df_aux = df_aux.replace(',', '', regex=True)  # Remove commas from values
        df_aux['Group'] = df_aux['Group'].astype(str).apply(format_group_name)  # Apply formatting
        df_aux['Group'] = df_aux['Group'].str.replace(
            r'(?<=[-/])([a-z])', 
            lambda match: match.group(1).upper(),
            regex=True
        )
        df_aux['Group'] = df_aux['Group'].astype(str).str.strip()  # Trim leading/trailing spaces
        
        # Rename 'Group' column to 'SOCName'
        df_aux.rename(columns={'Group': 'SOCName'}, inplace=True)
        
        # Replace specific SOCName directly
        df_aux['SOCName'] = df_aux['SOCName'].replace(
            'Software Developersapplications And Systems Software',
            'Software Developers Applications And Systems Software'
        )
        df_aux['SOCName'] = df_aux['SOCName'].replace(
            'Merchandise Displayer And Windows Trimmers',
            'Merchandise Displayers And Windows Trimmers'
        )
        df_aux['SOCName'] = df_aux['SOCName'].replace(
            'Merchandise Displayers And Window Trimmers',
            'Merchandise Displayers And Windows Trimmers'
        )
        # Merge with df_mapping on 'SOCName' (ensure 'df_mapping' is defined)
        df_aux = pd.merge(
            df_aux, df_mapping, on=['SOCName'], how='left', indicator=True
        )
        
    # Append the processed DataFrame to the list
    df_age=pd.concat([df_age, df_aux],
    ignore_index=True)


In [ ]:
df_aux=df_age[df_age['_merge']=='left_only']
# Step 1: Get unique SOCName values from df_aux
unique_socnames = df_aux['SOCName'].unique()

# Step 2: Create a mapping for the unique SOCName values
unique_socname_mapping = {
    socname: find_best_match(socname, df_mapping['SOCName']) for socname in unique_socnames
}

# Step 3: Map the results back to the entire DataFrame
df_aux['SOCName'] = df_aux['SOCName'].map(unique_socname_mapping)

In [ ]:
df_age = df_age[df_age['_merge'] != 'left_only']
df_aux = df_aux.drop(columns=['SOC', '_merge'], errors='ignore')

# Merge with df_mapping on 'SOCName' (ensure 'df_mapping' is defined)
df_aux = pd.merge(
            df_aux, df_mapping, on=['SOCName'], how='left', indicator=True
        )

In [ ]:
df_age.head()

In [ ]:
df_aux = df_aux[df_aux['_merge'] != 'left_only']
df_age=pd.concat([df_age, df_aux],ignore_index=True)
df_age = df_age.drop(columns=['_merge'], errors='ignore')


In [ ]:
df_age = df_age[~pd.isna(df_age['FIPSState'])]
exclude_columns = ['SOCName', 'SOC']
# Convert all columns except the excluded ones to integers
for col in df_age.columns:
    if col not in exclude_columns:
        df_age[col] = pd.to_numeric(df_age[col], errors='coerce').fillna(-999).astype(int)
df_age=df_age.rename(columns={'Average of AGEP':'ageMean'})

In [ ]:
df_dem=pd.merge(
            df_dem, df_age, on=['SOC','SOCName','FIPSState','Year'], how='outer', indicator=True
        )

In [ ]:
df_dem.head()

In [ ]:
df_dem = df_dem.drop(columns=['_merge'], errors='ignore')
df_dem.to_csv(directory + 'dem_occst.csv', index=False)


In [ ]:
df_dem=pd.read_csv('/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/Age/dem_occst.csv', delimiter=',') 
df_occ=pd.read_excel('/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/soc_2010_to_2018_crosswalk.xlsx', skiprows=8) 


In [ ]:
# Apply the function to the column
df = clean_soc_mapping(df)


In [ ]:
df_merge=pd.merge(
            df, df_dem.drop(columns=['SOCName']).drop_duplicates(subset=['SOC','FIPSState','Year']), on=['SOC','FIPSState','Year'], how='left', indicator=True
        )

In [ ]:
difference=df_merge[df_merge['_merge']=='left_only']

In [ ]:
columns_to_drop=['Male','Female','white','black','americanIndian','alaskaNative','asian','hawaiianPacificIslander','someOtherRace','ageMean','_merge']

In [ ]:
df_merge2=pd.merge(
            difference.drop(columns=columns_to_drop), df_dem.drop(columns=['SOC']).drop_duplicates(subset=['SOCName','FIPSState','Year']), on=['SOCName','FIPSState','Year'], how='left', indicator=True
        )

In [ ]:
df_merge=pd.concat([df_merge[df_merge['_merge']=='both'].drop(columns=['_merge']), df_merge2.drop(columns=['_merge'])],ignore_index=True)

In [ ]:
df_merge.head()

### By MSA

In [ ]:
directory='/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/Race/MSA'


In [ ]:
# Create a dictionary for renaming the columns
rename_columns = {
    "Estimate!!Total!!White alone": "white_msa",
    "Estimate!!Total!!Black or African American alone": "black_msa",
    "Estimate!!Total!!American Indian and Alaska Native alone": "americanIndianAlaskaNative_msa",
    "Estimate!!Total!!Asian alone": "asian_msa",
    "Estimate!!Total!!Native Hawaiian and Other Pacific Islander alone": "nativeHawaiianPacificIslander_msa",
    "Estimate!!Total!!Some other race alone": "someOtherRace_msa",
    "Estimate!!Total":"pop_msa",
    "Estimate!!Total:!!White alone": "white_msa",
    "Estimate!!Total:!!Black or African American alone": "black_msa",
    "Estimate!!Total:!!American Indian and Alaska Native alone": "americanIndianAlaskaNative_msa",
    "Estimate!!Total:!!Asian alone": "asian_msa",
    "Estimate!!Total:!!Native Hawaiian and Other Pacific Islander alone": "nativeHawaiianPacificIslander_msa",
    "Estimate!!Total:!!Some other race alone": "someOtherRace_msa",
    "Estimate!!Total:":"pop_msa"
}


In [ ]:
directory='/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/Race/MSA/'

df_race=pd.DataFrame()
# Iterate over all CSV files in the directory
for filename in os.listdir(directory):
    if filename.endswith('Data.csv'):
        
        # Read the CSV file, skipping the first row
        df_aux = pd.read_csv(os.path.join(directory, filename), skiprows=1)
        
        # Create a 'Year' column with the extracted year
        df_aux['Year'] = filename[7:11]
        
        df_aux = df_aux.loc[:, ~df_aux.columns.str.contains("Error!", na=False)]
        # Rename the columns
        df_aux.rename(columns=rename_columns, inplace=True)
        df_aux['BestFitMSA'] = df_aux['Geography'].str[-5:]
        df_aux.drop(columns=['Geography', 'Geographic Area Name','Estimate!!Total!!Two or more races','Estimate!!Total!!Two or more races!!Two races including Some other race','Estimate!!Total!!Two or more races!!Two races excluding Some other race, and three or more races','Unnamed: 22'], inplace=True, errors='ignore')

        columns_to_process=df_aux.columns.difference(['BestFitMSA','pop_msa','Year'])
        df_aux= df_aux.apply(pd.to_numeric, errors='coerce')
        df_aux[columns_to_process] = df_aux[columns_to_process].div(df_aux['pop_msa'], axis=0).mul(100)
        df_aux = df_aux.loc[:, ~df_aux.columns.str.startswith('Estimate')]

        
    # Append the processed DataFrame to the list
    df_race=pd.concat([df_race, df_aux],
    ignore_index=True)


In [ ]:
df_race=df_race.drop_duplicates(subset=['Year','BestFitMSA'])

In [ ]:
df_race=df_race.sort_values(['Year','BestFitMSA'])

In [ ]:
df_race.to_csv(directory + 'dem_msa.csv', index=False)


In [ ]:
df_aux=pd.merge(df, df_race, on=['Year','BestFitMSA'],how='left', indicator=True)

In [ ]:
df_race=pd.read_csv('/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/Race/MSA/dem_msa.csv', delimiter=',') 
   

In [ ]:
df_race.drop_duplicates(subset=['Year','BestFitMSA'], inplace=True)

In [ ]:
df=pd.merge(df, df_race, on=['Year','BestFitMSA'],how='left')
del df_race

## Industry Productivity

In [ ]:
df_aux = pd.read_excel('/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/Industry Productivity/SeriesReport-20250110160822_c3652e.xlsx', skiprows=3)


In [ ]:
# Create the mapping dictionary
sector_mapping = {
    "A": "Agriculture, Forestry, Fishing and Hunting",
    "B": "Mining",
    "C": "Utilities",
    "D": "Construction",
    "E": "Manufacturing",
    "G": "Wholesale Trade",
    "H": "Retail Trade",
    "I": "Transportation and Warehousing",
    "J": "Information",
    "K": "Finance and Insurance",
    "L": "Real Estate and Rental and Leasing",
    "M": "Professional, Scientific, and Technical Services",
    "O": "Management of Companies and Enterprises",
    "P": "Administrative and Support and Waste Management and Remediation Services",
    "Q": "Educational Services",
    "R": "Health Care and Social Assistance",
    "S": "Arts, Entertainment, and Recreation",
    "T": "Accommodation and Food Services",
    "U": "Other Services (except Public Administration)",
    "W": "Government",
    "Z": "Multi-Sector"
}

measure_mapping = {
    "C00": "Capital productivity (Index, 2017=100)",
    "C01": "Capital input (Index, 2017=100)",
    "C02": "Capital costs (Millions of current dollars)",
    "C03": "Capital share (Percentage)",
    "C06": "Capital intensity (Index, 2017=100)",
    "C07": "Contribution of capital intensity to labor productivity (Index, 2017=100)",
    "L00": "Labor productivity (Index, 2017=100)",
    "L01": "Hours worked (Index, 2017=100)",
    "L02": "Labor compensation (Millions of current dollars)",
    "L03": "Labor share (Percentage)",
    "L06": "Real labor compensation (Million $)",
    "L07": "Real labor compensation (Index, 2017=100)",
    "L20": "Hours worked (Millions of hours)",
    "M00": "Total factor productivity (Index, 2017=100)",
    "M01": "Combined inputs (Index, 2017=100)",
    "M02": "Combined inputs costs (Millions of current dollars)",
    "M05": "Combined inputs price deflator (Index, 2017=100)",
    "P00": "Intermediate inputs productivity (Index, 2017=100)",
    "P01": "Intermediate inputs (Index, 2017=100)",
    "P02": "Intermediate inputs costs (Millions of current dollars)",
    "P03": "Intermediate inputs share (Percentage)",
    "P06": "Intermediate inputs intensity (Index, 2017=100)",
    "P07": "Contribution of intermediate inputs intensity to labor productivity (Index, 2017=100)",
    "T01": "Real sectoral output (Index, 2017=100)",
    "T02": "Real value-added output (Index, 2017=100)",
    "T05": "Sectoral output price deflator (Index, 2017=100)",
    "T06": "Value-added output price deflator (Index, 2017=100)",
    "T30": "Sectoral output (Millions of current dollars)",
    "T39": "Value-added output (Millions of current dollars)",
    "U10": "Unit labor costs (Index, 2017=100)",
    "U11": "Labor compensation (Index, 2017=100)",
    "U12": "Hourly compensation (Index, 2017=100)",
    "U13": "Hourly compensation ($ / Hour)",
    "U14": "Real hourly compensation ($ / Hour)",
    "U15": "Real hourly labor compensation (Index, 2017=100)",
    "W00": "Output per worker (Index, 2017=100)",
    "W01": "Employment (Index, 2017=100)",
    "W20": "Employment (Thousands of jobs)"
}


In [ ]:
# Extract the 4th character from Series ID
df_aux['Sector'] = df_aux['Series ID'].str[3].map(sector_mapping)
df_aux['NAICS'] = df_aux['Series ID'].apply(lambda x: x.split('N')[1].split('_')[0] if 'N' in x and '_' in x else None)
df_aux['Series'] = df_aux['Series ID'].str[11:14].map(measure_mapping)
df_aux['NAICS'] = df_aux['NAICS'].replace(
            '622A',
            '622'
        )
df_aux.drop(columns=['Series ID'], inplace=True)

In [ ]:
df_aux.Series.unique()

In [ ]:
# Melt the DataFrame
df_prod = pd.melt(
            df_aux,
            id_vars=["Sector", 'NAICS', 'Series'],
            var_name='Year',  # Name for the year column
            value_name='Value')  # Name for the values column

In [ ]:
# Pivot the table to create columns per variable
df_prod = df_prod.pivot_table(
    index=["Sector", "NAICS",'Year'],  # Keys to keep in rows
    columns='Series',  # Values to turn into columns
    values='Value',  # Values for the new columns
    aggfunc='first'
    ).reset_index()

In [ ]:
df_prod.rename(columns={'Labor productivity (Index, 2017=100)':'lProd',
       'Capital productivity (Index, 2017=100)':'kProd',
       'Capital input (Index, 2017=100)':'kInp',
       'Capital costs (Millions of current dollars)':'kCost',
       'Capital share (Percentage)':'kShare',
       'Capital intensity (Index, 2017=100)':'kInt',
       'Contribution of capital intensity to labor productivity (Index, 2017=100)':'kLabCont'}, inplace=True)

In [ ]:
df_prod['Year'] = df_prod['Year'].str.extract(r'(\d{4})').astype(int)


In [ ]:
# Create the columns based on the length of the NAICS value
df_prod['NAICS6'] = df_prod['NAICS'].apply(lambda x: x if len(x) == 6 else -999).astype(int)
df_prod['NAICS5'] = df_prod['NAICS'].apply(lambda x: x[:5] if len(x) >= 5 else -999).astype(int)
df_prod['NAICS4'] = df_prod['NAICS'].apply(lambda x: x[:4] if len(x) >= 4 else -999).astype(int)
df_prod['NAICS3'] = df_prod['NAICS'].apply(lambda x: x[:3] if len(x) >= 3 else -999).astype(int)
df_prod['Sector'] = df_prod['NAICS'].apply(lambda x: x[:2] if len(x) >= 2 else -999).astype(int)


In [ ]:
df_prod.drop(columns=['NAICS'], inplace=True)

In [ ]:
df_prod.to_csv('/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/Industry Productivity/prod_ind.csv', index=False)


In [ ]:
df_prod = df_prod.loc[:, ~df_prod.columns.str.startswith('k')]
df_prod.drop(columns=['NAICS6'], inplace=True)

In [ ]:
# Define the merging levels and their hierarchy dynamically
merge_hierarchy = ["Sector", "NAICS3", "NAICS4", "NAICS5"]

# Generate dynamic merge levels with conditions
merge_levels = []
for i, level in enumerate(merge_hierarchy):
    # Levels above the current level should be checked for -999
    higher_levels = merge_hierarchy[:i]
    # Levels below the current level will be dropped
    drop_levels = merge_hierarchy[i+1:]

    # Define the condition for filtering rows
    def condition(df, level=level, higher_levels=higher_levels):
        return (df[level] != -999) & (df[higher_levels].eq(-999).all(axis=1))

    # Append the merge configuration
    merge_levels.append({
        "level": level,
        "drop": drop_levels,
        "condition": condition
    })
# Initialize the final merged DataFrame
final_merge = pd.DataFrame()
# Iterate through each dynamic merge level
for merge_config in merge_levels:
    level = merge_config["level"]
    drop_columns = merge_config["drop"]
    condition = merge_config["condition"]

    # Filter rows matching the current condition
    subset = df[condition(df)]

    # If no rows match the condition, skip this level
    if subset.empty:
        continue

    # Prepare the `df_ind` subset for the current level
    df_prod_subset = (
        df_prod[["Year", "lProd", level]]
        .dropna(subset=[level])
        .drop_duplicates(subset=["Year", level], keep="first")
    )

    # Perform the merge
    df_merge = pd.merge(
        subset,
        df_prod_subset.drop(columns=drop_columns, errors="ignore"),
        on=["Year", level],
        how="left"
    )

    # Append the successfully merged rows to the final DataFrame
    final_merge = pd.concat([final_merge, df_merge], ignore_index=True)


## NGOs

In [ ]:
df_aux = pd.read_csv('/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/NGOs/BMF_UNIFIED_V1.1.csv', delimiter=',')


In [ ]:
df_aux = df_aux[df_aux['ORG_YEAR_LAST']>2009].reset_index(drop=True)
df_aux = df_aux[df_aux['ORG_YEAR_FIRST']<2021].reset_index(drop=True)


In [ ]:
df_aux.head()

In [ ]:
def clean_company_name(name):
    if pd.isna(name):
        return name
    
    # Dictionary to map variations of terms to their normalized form (only at the end)
    replacements = {
        r'\bcorp(oration)?\b$': '',  # Remove "Corp" or "Corporation" at the end
        r'\binc(orporated)?\b$': '',  # Remove "Inc" or "Incorporated" at the end
        r'\bltd\b$': '',  # Remove "LTD" at the end
        r'\bco(mpany)?\b$': '',  # Remove "Co" or "Company" at the end
        r'\bplc\b$': '',  # Remove "PLC" at the end
    }
    
    # Normalize the name
    normalized_name = name.lower()  # Convert to lowercase
    normalized_name = re.sub(r'[.,]', '', normalized_name)  # Remove commas and periods
    normalized_name = re.sub(r'[^a-zA-Z0-9 &\']+', '', normalized_name)  # Remove special characters except &, '
    normalized_name = re.sub(r'-', ' ', normalized_name)  # Replace hyphens with spaces
    
    # Apply replacements only at the end
    for pattern, replacement in replacements.items():
        normalized_name = re.sub(pattern, replacement, normalized_name)
    
    # Capitalize the first letter of each word, including those after '&', but not after an apostrophe
    words = normalized_name.split()
    capitalized_words = []
    for word in words:
        if "'" in word:
            # Handle words with apostrophes: capitalize the first part, keep the rest as is
            parts = word.split("'")
            capitalized_words.append(parts[0].capitalize() + "'" + parts[1])
        else:
            capitalized_words.append(word.capitalize())  # Capitalize all other words
    
    return ' '.join(capitalized_words).strip()  # Join words and remove leading/trailing whitespace


In [ ]:

def clean_company_nameb(name):
    if pd.isna(name):
        return name
    
    # Dictionary to map variations of terms to their normalized form (only at the end of the name)
    replacements = {
        r'\bcorp(oration)?\b$': '',       # Remove "Corp" or "Corporation" at the end
        r'\binc(orporated)?\b$': '',     # Remove "Inc" or "Incorporated" at the end
        r'\bltd\b$': '',                 # Remove "LTD" at the end
        r'\bco(mpany)?\b$': '',          # Remove "Co" or "Company" at the end
        r'\bplc\b$': '',                 # Remove "PLC" at the end
        r'\bgroup\b$': '',               # Remove "Group" at the end
        r'\bholdings?\b$': '',           # Remove "Holding" or "Holdings" at the end
        r'\bnv\b$': '',                  # Remove "NV" at the end
        r'\bfund\b$': '',                # Remove "Fund" at the end
        r'\bllc\b$': '',                 # Remove "LLC" at the end
        r'\bfd\b$': '',                  # Remove "FD" at the end
        r'\bint\b$': '',                 # Remove "INT" at the end
        r'\binternational\b$': '',       # Remove "International" at the end
    }
    
    # Normalize the name
    normalized_name = name.lower()  # Convert to lowercase
    normalized_name = re.sub(r'[.,]', '', normalized_name)  # Remove commas and periods
    normalized_name = re.sub(r'[^a-zA-Z0-9 &\']+', '', normalized_name)  # Remove special characters except &, '
    normalized_name = re.sub(r'-', ' ', normalized_name)  # Replace hyphens with spaces
    
    # Apply replacements only at the end
    for pattern, replacement in replacements.items():
        normalized_name = re.sub(pattern, replacement, normalized_name)
    
    # Capitalize the first letter of each word, including those after '&' and handling apostrophes
    words = normalized_name.split()
    capitalized_words = []
    for word in words:
        if "'" in word:
            # Handle apostrophes: capitalize the first part and leave the rest
            parts = word.split("'")
            capitalized_words.append(parts[0].capitalize() + "'" + parts[1])
        else:
            capitalized_words.append(word.capitalize())  # Capitalize other words
    
    return ' '.join(capitalized_words).strip()  # Join words and remove leading/trailing whitespace


In [ ]:
# Apply the cleaning function to both DataFrames
df['clean_company_name'] = df['CanonEmployer'].apply(clean_company_name)
df_aux['clean_company_name'] = df_aux['ORG_NAME_CURRENT'].apply(clean_company_name)



In [ ]:
df_aux=df_aux[['clean_company_name', 'NCCS_LEVEL_1']]
df_aux=df_aux.rename(columns={'NCCS_LEVEL_1':'nonprofit'})

In [ ]:
df_aux.to_csv('/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/NGOs/ngos.csv', index=False)


In [ ]:
# Merge the two DataFrames using the cleaned company names
df = pd.merge(
    df,
    df_aux,
    on='clean_company_name',
    how='left', indicator=True
)

df.drop(columns=['clean_company_name'], inplace=True)


In [ ]:
df[df.nonprofit=='501CX NONPROFIT'][0:50]

### COMPUSTAT

In [ ]:
df_aux=pd.read_csv('/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/COMPUSTAT/compustat_data.csv', delimiter=',') 

In [ ]:
df_aux['roa']=(df_aux['ibq']/df_aux['atq'])*100
df_aux['roe']=(df_aux['ibq']/df_aux['ceqq'])*100
df_aux['logSize']=np.log(df_aux['atq'])
df_aux['gProfit']=((df_aux['saley']-df_aux['cogsy'])/df_aux['saley'])*100
df_aux['tobinq']=((df_aux['prccq']*df_aux['cshoq'])+(df_aux['dlttq']+df_aux['dlcq']))/df_aux['atq']
df_aux['lev']=df_aux['dlttq']/df_aux['atq']
df_aux.rename(columns={'fyearq': 'Year', 'fqtr': 'quarter', 'naics': 'comNAICS',
                       'loc':'comCountry','county':'comCounty', 'state':'comHeadquarters', 'idbflag':'comDom'}, inplace=True)
df_aux['Year']=df_aux['Year'].astype(int)
# Check if 'ipodate' is missing
df_aux['ipoYear'] = df_aux['ipodate'].str[:4].where(df_aux['ipodate'].notna(), None)

# Calculate company age, assigning NaN or a placeholder for missing IPO dates
df_aux['comAge'] = df_aux.apply(
    lambda x: x['Year'] - int(x['ipoYear']) if x['ipoYear'] is not None else None, axis=1
)

In [ ]:
df_aux=df_aux[['gvkey', 'Year', 'quarter','tic', 'cusip', 'conm', 
       'conml', 'comCounty', 'comDom',
       'comCountry', 'comNAICS', 'comHeadquarters','roa', 'roe',
       'logSize', 'gProfit', 'tobinq', 'lev','comAge']]

In [ ]:
# Create the quarter-to-month mapping
quarter_to_months = {
    1: [1, 2, 3],
    2: [4, 5, 6],
    3: [7, 8, 9],
    4: [10, 11, 12],
}

# Function to expand a dataset
def expand_quarters_to_months(data, quarter_col):
    # Validate the input
    if quarter_col not in data.columns:
        raise ValueError(f"Column '{quarter_col}' not found in the dataset.")
    
    # Expand the dataset
    expanded_data = (
        data
        .assign(Month=data[quarter_col].map(quarter_to_months))  # Map quarters to months
        .explode("Month", ignore_index=True)  # Expand each quarter into multiple rows
    )
    return expanded_data

In [ ]:
df_aux = expand_quarters_to_months(df_aux, "quarter")


In [ ]:
df_cross=pd.read_csv('/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/COMPUSTAT/Compustat Merge.csv', delimiter=',')
df_cross['gvkey']=df_aux['gvkey'].astype(int)

In [ ]:
df_test=pd.read_csv('/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/COMPUSTAT/all_identifiers.csv', delimiter=',')

In [ ]:
df_test=df_test[(df_test['symboltypecat']=='cusip') | (df_test['symboltypecat']=='gvkey')]
df_test=df_test.drop_duplicates(subset=['companyname','symbolvalue'])
df_test.head()

In [ ]:
df_comp=df_aux[['conm', 'conml', 'gvkey', 'cusip']].drop_duplicates()
df_bgt=df[['CanonEmployer']].drop_duplicates()
df_cross=df_cross[['bgt_cleanco','conml', 'conml_cleanco', 'cusip']].drop_duplicates()
df_test=df_test[['companyname','symboltypecat','symbolvalue']].drop_duplicates()

In [ ]:
# Apply the cleaning function to both DataFrames
df_bgt['clean_company_name'] = df_bgt['CanonEmployer'].apply(clean_company_name)
df_comp['clean_company_name'] = df_comp['conml'].apply(clean_company_name)


In [ ]:
df_merge=pd.merge(
    df_comp,
    df_cross.drop(columns=['cusip']),
    on=['conml'],
    how='left', indicator=True
)


In [ ]:
df_merge=pd.merge(
    df_merge.drop(columns=['_merge']),
    df_cross.drop(columns=['conml']),
    on=['cusip'],
    how='left', indicator=True
)
df_merge['bgt_cleanco'] = df_merge['bgt_cleanco_x'].combine_first(df_merge['bgt_cleanco_y'])


In [ ]:
df_merge.drop(columns=[ 'bgt_cleanco_x', 'conml_cleanco_x', 'bgt_cleanco_y', 'conml_cleanco_y',
       '_merge'], inplace=True)

In [ ]:
df_merge=df_merge.drop_duplicates()

df_merge=pd.merge(
    df_merge.drop(columns=['_merge'],errors='ignore'),
    df_test,
    left_on=['cusip'],
    right_on=['symbolvalue'],
    how='left', indicator=True
)

In [ ]:
df_merge['companynameco'] = df_merge['companyname'].apply(clean_company_name)


In [ ]:
# Filter the DataFrame for rows where 'symbolvalue' is 'gvkey'
df_test2 = df_test[df_test['symboltypecat'] == 'gvkey']

# Convert the 'symbolvalue' column to integers
# If the column contains non-numeric values, you may need to handle errors
df_test2['symbolvalue'] = df_test2['symbolvalue'].astype(int)


In [ ]:

df_merge=pd.merge(
    df_merge.drop(columns=['companyname', 'symboltypecat', 'symbolvalue', '_merge'],errors='ignore'),
    df_test2,
    left_on=['gvkey'],
    right_on=['symbolvalue'],
    how='left', indicator=True
)

In [ ]:
df_merge['companyname'] = df_merge['companyname'].apply(clean_company_name)
df_merge['companyname_co'] = df_merge['companynameco'].combine_first(df_merge['companyname'])


In [ ]:
df_merge=df_merge[['conm', 'conml', 'gvkey', 'cusip', 'clean_company_name', 'bgt_cleanco',
       'companyname_co']]

In [ ]:
df_crosswalk=pd.concat([df_merge[['conm', 'conml', 'gvkey', 'cusip', 'clean_company_name']],
                      df_merge[['conm', 'conml', 'gvkey', 'cusip','bgt_cleanco']].rename(columns={'bgt_cleanco':'clean_company_name'})])

In [ ]:
df_crosswalk=pd.concat([df_crosswalk,
                      df_merge[['conm', 'conml', 'gvkey', 'cusip','companyname_co']].rename(columns={'companyname_co':'clean_company_name'})])

In [ ]:
df_crosswalk=df_crosswalk.sort_values(by=['gvkey'])

In [ ]:
df_crosswalk=df_crosswalk.drop_duplicates()
df_crosswalk=df_crosswalk.dropna(subset=['clean_company_name'])
df_crosswalk['clean']=df_crosswalk['conml'].apply(clean_company_nameb)
df_crosswalk['cleanb']=df_crosswalk['clean'].apply(clean_company_nameb)

In [ ]:
df_compcross=pd.concat([df_crosswalk[['conm', 'conml', 'gvkey', 'cusip', 'clean_company_name']],
                      df_crosswalk[['conm', 'conml', 'gvkey', 'cusip','clean']].rename(columns={'clean':'clean_company_name'})])

df_compcross=pd.concat([df_compcross,
                      df_crosswalk[['conm', 'conml', 'gvkey', 'cusip','cleanb']].rename(columns={'cleanb':'clean_company_name'})])


In [ ]:
df_compcross=df_compcross.drop_duplicates()
df_compcross=df_compcross.sort_values(by=['gvkey'])

In [ ]:
def expand_dataframe_with_and_ampersand(df, columns):
    # If no columns are specified, default to all string columns
    if columns is None:
        columns = df.select_dtypes(include=['object']).columns
    
    # Replace '&' with 'And' for the specified columns
    df_and = df.copy()
    for column in columns:
        df_and[column] = df_and[column].str.replace('&', 'And', regex=False)
    
    # Replace 'And' with '&' for the specified columns
    df_ampersand = df.copy()
    for column in columns:
        df_ampersand[column] = df_ampersand[column].str.replace('And', '&', regex=False)
    
    # Combine the original DataFrame, the '&' → 'And', and the 'And' → '&' DataFrames
    expanded_df = pd.concat([df, df_and, df_ampersand], ignore_index=True)
    return expanded_df

In [ ]:
expanded_data = expand_dataframe_with_and_ampersand(df_compcross, columns=['clean_company_name'])


In [ ]:
df_crosswalk=expanded_data.drop_duplicates()
df_crosswalk=df_crosswalk.sort_values(by=['gvkey'])

In [ ]:
df_compustat[0:100]

In [ ]:

df_compustat=pd.merge(
    df_aux.drop(columns=['conm', 'conml'],errors='ignore'),
    df_crosswalk.drop(columns=['cusip'],errors='ignore'),
    on=['gvkey'],
    how='left', indicator=True
)


In [ ]:
df_compustat.drop(columns=['_merge'], inplace=True)

In [ ]:
df_compustat.to_parquet('/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/COMPUSTAT/compustat_quarter.parquet', engine='pyarrow', index=False)



In [ ]:
df['clean_company_name'] = df['CanonEmployer'].apply(clean_company_name)


In [ ]:
df_merge=pd.merge(
    df,
    df_compustat.drop_duplicates(subset=['clean_company_name','conm', 'conml']),
    on=['clean_company_name','Year','Month'],
    how='left', indicator=True
)

In [ ]:
df_merge['_merge'].value_counts()

In [ ]:
df_comp = pd.read_parquet('/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/COMPUSTAT/compustat_quarter.parquet')


In [ ]:
df_comp.columns

In [ ]:
df_comp[df_comp['clean_company_name']=="Kohl's"]

In [ ]:
df['clean_company_name'] = df['CanonEmployer'].apply(clean_company_name)


In [ ]:
duplicates = (
    df_merge
    .groupby(['BGTJobId'])
    .size()
    .reset_index(name='count')
)


In [ ]:
duplicates = duplicates[duplicates['count'] > 1]


In [ ]:
problematic_rows = df_merge.merge(duplicates, on=['BGTJobId'], how='inner')


In [ ]:
problematic_rows

In [ ]:
len(df)

In [ ]:
df.JobId.dtypes

In [ ]:
df_oesm = pd.read_parquet('/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/OESM/All/CLEAN/2019.parquet')


In [ ]:
df_oesm.NAICS6.unique()

## Migration

In [782]:
df_cbsa=pd.read_csv('/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/Population/cbsa-est2020-alldata.csv', encoding='latin', delimiter=',') 
df_cbsa1=pd.read_csv('/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/Population/cbsa-est2022.csv', encoding='latin', delimiter=',') 
df_cbsa2=pd.read_csv('/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/Population/cbsa-est2009-alldata.csv', encoding='latin', delimiter=',') 
df_cbsa2['CBSA'] = pd.to_numeric(df_cbsa2['CBSA'], errors='coerce')

df_cbsa1 = df_cbsa1[[col for col in df_cbsa1.columns if not col.endswith("2020")]]
df_cbsa2 = df_cbsa2.iloc[2:].reset_index(drop=True)

df_cbsa=pd.merge(df_cbsa, df_cbsa1.drop(columns=['MDIV','STCOU', 'NAME','LSAD']), how='outer', on=['CBSA'])
df_cbsa=pd.merge(df_cbsa, df_cbsa2.drop(columns=['MDIV','STCOU', 'NAME','LSAD']), how='outer', on=['CBSA'], indicator=True)


In [783]:
# 1. Filter rows where LSAD equals 'Metropolitan Statistical Area'
df_filtered = df_cbsa[df_cbsa['LSAD'] == 'Metropolitan Statistical Area'].copy()

# 2. Rename 'CBSA' and 'NAME' to 'BestFitMSA' and 'BestFitMSAName'
df_filtered = df_filtered.rename(columns={'CBSA': 'BestFitMSA', 'NAME': 'BestFitMSAName'})

# 3. Select only the columns we care about:
#    - The identifier columns: BestFitMSA and BestFitMSAName
#    - All columns that start with 'INTERNATIONAL' or 'DOMESTIC'
migration_cols = [col for col in df_filtered.columns 
                  if col.startswith('INTERNATIONAL') or col.startswith('DOMESTIC') or col.startswith('POPESTIMATE')]
df_filtered = df_filtered[['BestFitMSA', 'BestFitMSAName'] + migration_cols]

# 4. Reset the index to create a unique identifier column.
#    (If you already have a unique id column, you can use that instead.)
df_filtered = df_filtered.reset_index(drop=True).reset_index()  # This creates an 'index' column for reshaping

# 5. Use pd.wide_to_long to reshape the DataFrame from wide to long format.
#    The stub names are 'INTERNATIONALMIG' and 'DOMESTICMIG', with the year as suffix.
df_long = pd.wide_to_long(
    df_filtered,
    stubnames=['INTERNATIONALMIG', 'DOMESTICMIG','POPESTIMATE'],
    i='index',
    j='Year',
    sep='',
    suffix='\d+'
).reset_index() # Use reset_index() without drop=True to keep 'year' as a column


df_long=df_long[df_long['Year']>2008]
df_long=df_long.drop_duplicates(subset=['Year','BestFitMSA'])
df_long['intmig']=(df_long['INTERNATIONALMIG']/df_long['POPESTIMATE'])*100
df_long['dommig']=(df_long['DOMESTICMIG']/df_long['POPESTIMATE'])*100

In [793]:

df_long.sort_values(by=['BestFitMSA', 'Year'], inplace=True)
df_long['popchange'] = (
    df_long.groupby('BestFitMSA')['POPESTIMATE'].pct_change() * 100
)


df_long=df_long[['Year','BestFitMSA', 'intmig','dommig', 'popchange']]
df_long['BestFitMSA']=df_long['BestFitMSA'].astype(int)

In [796]:
df_long.head()

,Year,BestFitMSA,intmig,dommig,popchange
9,2009,10180,0.074967,-0.006247,NaN
10,2010,10180,0.053747,0.074280,3.448491
11,2011,10180,0.123612,0.040804,0.640739
12,2012,10180,0.315853,-0.272266,0.499247
13,2013,10180,0.222667,-0.635167,0.019106


In [797]:
df_long.to_parquet('/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/Population/migration_msa.parquet', engine='pyarrow', compression='zstd',index=False)



In [811]:
df_co=pd.read_csv('/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/Population/co-est2020-alldata.csv', encoding='latin', delimiter=',') 
df_co1=pd.read_csv('/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/Population/co-est2023-alldata.csv', encoding='latin', delimiter=',') 
df_co2=pd.read_csv('/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/Population/co-est2009-alldata.csv', encoding='latin', delimiter=',') 

df_co1 = df_co1[[col for col in df_co1.columns if not col.endswith("2020")]]

df_co=pd.merge(df_co, df_co1, how='left', on=['SUMLEV','REGION','DIVISION', 'STATE','COUNTY','STNAME','CTYNAME'])
df_co=pd.merge(df_co, df_co2, how='left', on=['SUMLEV','REGION','DIVISION', 'STATE','COUNTY','STNAME','CTYNAME'], indicator=True)


In [812]:
# 1. Rename columns: STATE -> FIPSState, COUNTY -> FIPSCounty
df_co = df_co.rename(columns={'STATE': 'FIPSState', 'COUNTY': 'FIPSCounty'})

# 2. Replace FIPSCounty values of 0 with -999
df_co.loc[df_co['FIPSCounty'] == 0, 'FIPSCounty'] = -999

# 3. Select only the columns to keep:
#    - The identifier columns: FIPSState and FIPSCounty
#    - All columns that start with 'INTERNATIONAL' or 'DOMESTIC'
migration_cols = [col for col in df_co.columns 
                  if col.startswith('INTERNATIONAL') or col.startswith('DOMESTIC') or col.startswith('POPESTIMATE')]
df_co_filtered = df_co[['FIPSState', 'FIPSCounty'] + migration_cols]

# 4. Reset the index to create a unique identifier column required for wide_to_long
df_co_filtered = df_co_filtered.reset_index(drop=True).reset_index()

# 5. Reshape the DataFrame from wide to long format.
#    The stubnames are 'INTERNATIONALMIG' and 'DOMESTICMIG', and the year is the suffix.
df_co_long = pd.wide_to_long(
    df_co_filtered,
    stubnames=['INTERNATIONALMIG', 'DOMESTICMIG','POPESTIMATE'],
    i='index',
    j='Year',
    sep='',
    suffix='\d+'
).reset_index()  # Do not drop the index here so that 'year' becomes a column


In [813]:
df_co_long=df_co_long[df_co_long['Year']>2008]


In [817]:
df_co_long.head()

,Year,FIPSCounty,FIPSState,intmig,dommig,popchange
73462,2009,-999,1,0.112961,0.234544,NaN
0,2010,-999,1,0.022004,0.025598,1.631148
3194,2011,-999,1,0.107550,-0.039649,0.295224
6388,2012,-999,1,0.130423,-0.002367,0.353985
9582,2013,-999,1,0.113710,0.047479,0.310466


In [815]:
df_co_long.sort_values(by=['FIPSCounty','FIPSState', 'Year'], inplace=True)
df_co_long['popchange'] = (
    df_co_long.groupby(['FIPSCounty','FIPSState'])['POPESTIMATE'].pct_change() * 100
)


In [816]:
df_co_long=df_co_long.drop_duplicates(subset=['Year','FIPSCounty','FIPSState'])
df_co_long['intmig']=(df_co_long['INTERNATIONALMIG']/df_co_long['POPESTIMATE'])*100
df_co_long['dommig']=(df_co_long['DOMESTICMIG']/df_co_long['POPESTIMATE'])*100
df_co_long=df_co_long[['Year','FIPSCounty','FIPSState', 'intmig','dommig', 'popchange']]

In [818]:
df_co_long.dtypes

Year            int64
FIPSCounty      int64
FIPSState       int64
intmig        float64
dommig        float64
popchange     float64
dtype: object

In [819]:
df_co_long.to_parquet('/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/Population/migration_ct.parquet', engine='pyarrow', compression='zstd',index=False)


### Educational Attainment

In [ ]:
directory='/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/Educational Attainment/'



In [ ]:
import glob

In [ ]:
# Build a file pattern for files ending with "S1501-Data.csv"
file_pattern = os.path.join(directory, "*S1501-Data.csv")
files = glob.glob(file_pattern)

dfs = []  # List to hold the DataFrames from each file

for filepath in files:
    filename = os.path.basename(filepath)
    
    # Extract a 4-digit year from the filename using regex.
    match = re.search(r'(\d{4})', filename)
    if match:
        year = int(match.group(1))
    else:
        print(f"No valid year found in filename: {filename}. Skipping file.")
        continue

    try:
        # Read the CSV file; use utf-8-sig to properly handle any BOM in the headers.
        df = pd.read_csv(filepath, encoding='utf-8-sig')
    except Exception as e:
        print(f"Error reading file {filename}: {e}. Skipping file.")
        continue

    # Check that GEO_ID exists.
    if "GEO_ID" not in df.columns:
        print(f"File {filename} is missing 'GEO_ID'. Skipping file.")
        continue

    # Check that S1501_C02_015E exists.
    if "S1501_C02_015E" not in df.columns:
        print(f"File {filename} is missing 'S1501_C02_015E'. Skipping file.")
        continue

    # Determine which column to use for college_st.
    # If the first row of S1501_C02_015E is 'Male!!Estimate!!Percent bachelor's degree or higher',
    # then use the values from S1501_C01_015E.
    if df["S1501_C02_015E"].iloc[0] == "Male!!Estimate!!Percent bachelor's degree or higher":
        if "S1501_C01_015E" not in df.columns:
            print(f"File {filename} is missing 'S1501_C01_015E' while S1501_C02_015E header indicates to use the other column. Skipping file.")
            continue
        df["college_st"] = df["S1501_C01_015E"]
    else:
        df["college_st"] = df["S1501_C02_015E"]

    # Add the Year column with the extracted year.
    df["Year"] = year

    # Keep only the desired columns: GEO_ID, Year, and college_st.
    df_subset = df[["GEO_ID", "Year", "college_st"]].copy()

    # Append this subset to the list.
    dfs.append(df_subset)

# Concatenate all the yearly DataFrames into one final DataFrame.
if dfs:
    final_df = pd.concat(dfs, ignore_index=True)
    print(final_df.head())
else:
    print("No files were processed successfully.")
    
# 2. Remove rows where GEO_ID is "Geography"
final_df = final_df[final_df['GEO_ID'] != "Geography"]

# 3. Ensure GEO_ID is treated as a string
final_df['GEO_ID'] = final_df['GEO_ID'].astype(str)

# 4. Extract the last 5 characters from GEO_ID
last5 = final_df['GEO_ID'].str[-5:]

# 5. Create FIPSState: first two characters of the last 5 characters, converted to int
final_df['FIPSState'] = last5.str[:2].astype(int)

# 6. Create FIPSCounty: last three characters of the last 5 characters, converted to int
final_df['FIPSCounty'] = last5.str[2:].astype(int)

In [ ]:
final_df.drop(columns=['GEO_ID'], inplace=True)

In [ ]:
final_df['college_st'] = pd.to_numeric(final_df['college_st'], errors='coerce')


In [ ]:
final_df.to_parquet('/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/Educational Attainment/education_ct.parquet', engine='pyarrow', compression='zstd',index=False)


### Industry GDP detailed

In [ ]:
df_ind=pd.read_excel('/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/Industry Productivity/GrossOutput.xlsx', sheet_name='TGO106-Q', skiprows=7) 
df_ind.drop(columns=['Unnamed: 2', 'Line'], inplace=True)
df_bea=pd.read_excel('/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/Industry Productivity/BEA-Industry-and-Commodity-Codes-and-NAICS-Concordance.xlsx', skiprows=4)




In [ ]:
def shift_row(row):
    # Get the non-NaN values as a list
    non_nan_values = row.dropna().tolist()
    # Calculate how many NaNs are needed to pad the row to its original length
    num_missing = len(row) - len(non_nan_values)
    # Create a new row with non-NaN values on the left and NaNs on the right
    new_values = non_nan_values + [np.nan] * num_missing
    # Return a Series with the same index as the original row
    return pd.Series(new_values, index=row.index)

# Apply the shifting function to each row
df_shifted = df_bea.apply(shift_row, axis=1)
df_shifted=df_shifted[['Sector','Summary']]
df_shifted=df_shifted.dropna()
df_shifted['Summary'] = df_shifted['Summary'].str.lower().str.strip()

In [ ]:
df_ind=df_ind.rename(columns={'Unnamed: 1':'Summary'})
df_ind['Summary'] = df_ind['Summary'].str.lower().str.strip()
df_ind.set_index('Summary', inplace=True)

# 3) Convert all remaining columns to numeric
#    - columns that cannot be converted (like pure text) will become NaN.
df_ind = df_ind.apply(pd.to_numeric, errors='coerce')

# 4) Now compute the percent change along columns
df_ind = df_ind.pct_change(axis=1)  # fractional change
df_ind = df_ind * 100               # convert to percentage

# 5) If desired, reset index
df_ind.reset_index(inplace=True)

In [ ]:
df_shifted=pd.merge(df_shifted, df_ind, on='Summary', how='inner')

In [ ]:
# 1. Drop rows where Sector is NaN

def parse_and_distribute(sector_value):
    """
    Remove all non-digit characters, strip trailing zeros,
    and return a Series with NAICS3 through NAICS6 (one will be set).
    """
    # Convert to string if not already
    sector_value = str(sector_value)
    # Remove all non-digit chars
    code = re.sub(r'[^0-9]', '', sector_value)
    # Remove trailing zeros
    code = code.rstrip('0')
    
    # Prepare a dictionary with defaults
    out = {'NAICS3': None}
    
    length = len(code)
    if length == 3:
        out['NAICS3'] = code



    return pd.Series(out)

# 2. Apply the parsing function to create new columns
df_shifted[['NAICS3']] = df_shifted['Sector'].apply(parse_and_distribute)

# 3. Drop duplicates based on columns 'Sector' and 'Summary'
df_shifted = df_shifted.drop_duplicates(subset=['Sector', 'Summary'])
df_shifted.drop(columns=['Summary'], inplace=True)

In [ ]:
# 1) Identify the columns that uniquely identify each row (the "id_vars")
id_vars = ['NAICS3',  'Sector']

df_long = df_shifted.melt(id_vars=id_vars, var_name='QuarterCol', value_name='Value')

# Parse Year and Quarter
df_long['Year'] = df_long['QuarterCol'].str[:4]  # e.g. "2022"
df_long['Q']    = df_long['QuarterCol'].str[-1:] # e.g. "Q1"
df_long['Sector'] = (
    df_long['Sector']
    .astype(str)
    .str[:2]
)
df_long['Sector'] = pd.to_numeric(df_long['Sector'], errors='coerce')
df_long['NAICS3'] = df_long['NAICS3'].fillna(-999)
df_long['NAICS3'] = pd.to_numeric(df_long['NAICS3'], errors='coerce').astype(str).str[:3].astype(int)
df_long['Q'] = df_long['Q'].astype(int)
df_long['Year'] = df_long['Year'].astype(int)

df_long.drop(columns=['QuarterCol'], inplace=True)

In [ ]:
quarter_to_months = {
    1: [1, 2, 3],   # Q1 => months 1,2,3
    2: [4, 5, 6],   # Q2 => months 4,5,6
    3: [7, 8, 9],   # Q3 => months 7,8,9
    4: [10, 11, 12] # Q4 => months 10,11,12
}

def expand_quarter(row):
    q = row['Q']
    months_list = quarter_to_months.get(q, [])  # empty if Q is missing/invalid
    new_rows = []
    for m in months_list:
        # Make a copy of the row and set this month's value
        expanded = row.copy()
        expanded['Month'] = m
        new_rows.append(expanded)
    return new_rows

expanded_rows = []
for _, r in df_long.iterrows():
    expanded_rows.extend(expand_quarter(r))

df_expanded = pd.DataFrame(expanded_rows)
df_expanded = df_expanded.dropna(subset=['Value'])
df_expanded.drop(columns=['Q'], inplace=True)

In [ ]:
df_expanded=df_expanded.drop_duplicates()
df_expanded=df_expanded.rename(columns={'Value':'igdpq_ch'})

In [ ]:
df_expanded = df_expanded.dropna(subset=['Sector'])


In [ ]:
df_expanded['Year'] = df_expanded['Year'].astype(int)
df_expanded['NAICS3'] = df_expanded['NAICS3'].astype(int)
df_expanded['Sector'] = df_expanded['Sector'].astype(int)
df_expanded['Month'] = df_expanded['Month'].astype(int)

In [ ]:
df_expanded.head()

In [ ]:
df_expanded.to_parquet('/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/Industry Productivity/gdpchange_quarter.parquet', engine='pyarrow', compression='zstd',index=False)


In [ ]:
df_crisis=df_expanded[df_expanded['Year']==2008]
df_crisis.drop(columns=['Year'], inplace=True)
df_crisis=df_crisis.rename(columns={'igdpq_ch':'crisis_ch'})

In [ ]:
df_crisis.to_parquet('/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/Industry Productivity/gdp_crisis.parquet', engine='pyarrow', compression='zstd',index=False)


In [ ]:
df_ind = pd.read_parquet(directory + 'Industry_Productivity/gdpchange_quarter.parquet')


### Detailed Industry

In [67]:
df_ind=pd.read_parquet('/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/Industry Productivity/gdpchange_quarter.parquet')



In [ ]:
df_ind.Year.unique()

In [ ]:
df_ind=df_ind[df_ind['Units']=='% Change from previous year']

In [ ]:
def fill_naics_cols(row):
    """
    Based on 'Digit' (e.g. '2-Digit', '3-Digit', etc.) and 'NAICS',
    return a Series with:
    - 'Sector'
    - 'NAICS3'
    - 'NAICS4'
    - 'NAICS5'
    - 'NAICS6'
    """
    digit_str = row['Digit']       # e.g. "2-Digit"
    naics_str = str(row['NAICS'])  # ensure string for slicing

    # Default all to -999
    sector  = -999
    naics3  = -999
    naics4  = -999
    naics5  = -999
    naics6  = -999

    if digit_str == '2-Digit':
        sector = naics_str

    elif digit_str == '3-Digit':
        sector = naics_str[:2]
        naics3 = naics_str

    elif digit_str == '4-Digit':
        sector = naics_str[:2]
        naics3 = naics_str[:3]
        naics4 = naics_str

    elif digit_str == '5-Digit':
        sector = naics_str[:2]
        naics3 = naics_str[:3]
        naics4 = naics_str[:4]
        naics5 = naics_str

    elif digit_str == '6-Digit':
        sector = naics_str[:2]
        naics3 = naics_str[:3]
        naics4 = naics_str[:4]
        naics5 = naics_str[:5]
        naics6 = naics_str

    return pd.Series([sector, naics3, naics4, naics5, naics6],
                     index=['Sector','NAICS3','NAICS4','NAICS5','NAICS6'])



In [ ]:
df_ind[['Sector','NAICS3','NAICS4','NAICS5','NAICS6']] = df_ind.apply(fill_naics_cols, axis=1)


In [ ]:
df_ind=df_ind[df_ind['Digit']!='Custom']
df_ind=df_ind[df_ind['Year']>2009]
df_ind.drop(columns=['NAICS','Industry','Units','Basis','Digit'], inplace=True, errors='ignore')

In [ ]:
excluded_values = [
    'Sectoral output', 
    'Sectoral output price deflator', 
    'Combined inputs',
    'Combined inputs price deflator', 
    'Capital input', 
    'Capital intensity',
    'Intermediate inputs', 
    'Intermediate inputs costs',
    'Intermediate inputs share', 
    'Intermediate inputs productivity',
    'Intermediate inputs intensity',
    'Contribution of intermediate inputs intensity to labor productivity',
    'Combined inputs costs', 'Hourly compensation','Employment'
]

df_ind = df_ind[~df_ind['Measure'].isin(excluded_values)]

In [ ]:
df_wide = df_ind.pivot(
    index=['Year','Sector','NAICS3','NAICS4','NAICS5','NAICS6'],   # Columns (or combo) that define each unique row
    columns='Measure',           # Column whose distinct values become new column headers
    values='Value'               # Column that fills the cells
).reset_index()

In [ ]:
rename_dict = {
    'Labor productivity': 'lprod',
    'Real sectoral output': 'routput',
    'Hours worked': 'hworked',
    'Output per worker': 'opw',
    'Unit labor costs': 'ulc',
    'Labor compensation': 'lcomp',
    'Total factor productivity': 'tfp',
    'Capital costs': 'kcosts',
    'Capital share': 'kshare',
    'Capital productivity': 'kprod',
    'Labor share': 'lshare',
    'Contribution of capital intensity to labor productivity':'ktolb'
}

# Suppose 'df' is your DataFrame. Then:
df_wide = df_wide.rename(columns=rename_dict)

In [ ]:
df_wide.head()

In [ ]:
cols_to_convert = ['Sector','NAICS3','NAICS4','NAICS5','NAICS6']

for col in cols_to_convert:
    # Convert to numeric (anything non-numeric becomes NaN),
    # then fill NaNs with -999 (or your choice),
    # finally convert to int.
    df_wide[col] = pd.to_numeric(df_wide[col], errors='coerce').fillna(-999).astype(int)

In [ ]:
df_merge=pd.merge(df,df_wide, on=['Year','Sector','NAICS3','NAICS4','NAICS5','NAICS6'],how='left',indicator=True)

In [ ]:
df_merge['_merge'].value_counts()

In [ ]:
df_wide.Year.unique()

In [ ]:
df_wide.to_parquet('/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/Industry Productivity/industry_all.parquet', engine='pyarrow', compression='zstd',index=False)

### Bartik

In [4]:
df_aux=pd.read_csv('/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/Industry Employment/2005.q1-q4.singlefile.csv', delimiter=',', encoding='latin') 


/opt/anaconda3/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3331: DtypeWarning: Columns (0) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [5]:
df_ind=df_aux

In [20]:

# Define input and output directories
input_dir = '/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/Industry Employment'  # Change this to the actual directory
output_dir = "/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/Industry Employment/Processed/NAICS6"  # Change this to where you want to save the files

# Ensure the output directory exists
os.makedirs(output_dir, exist_ok=True)

# Loop through years from 2004 to 2021
for year in range(2004, 2022):
    file_name = f"{year}_qtrly_singlefile.zip"
    file_path = os.path.join(input_dir, file_name)

    # Check if file exists
    if not os.path.exists(file_path):
        print(f"⚠️ File not found: {file_path}, skipping...")
        continue
    
    print(f"🔄 Processing {year}...")

    # Load data
    df_ind = pd.read_csv(file_path, compression="zip", dtype={'industry_code': str, 'area_fips': str, 'own_code': str})

    # Keep only industries with less than 3 digits
    df_ind = df_ind[df_ind['industry_code'].str.len() ==6]
    df_ind = df_ind[df_ind['own_code']!=0]
    df_ind = df_ind[~df_ind['industry_code'].str.startswith('10')]

    # Select relevant columns
    df_ind = df_ind[['area_fips', 'own_code', 'industry_code', 'year', 'qtr', 'month1_emplvl', 'taxable_qtrly_wages', 'qtrly_contributions', 'avg_wkly_wage']]
    agg_dict = {col: 'sum' for col in df_ind.columns if col not in ['area_fips', 'industry_code', 'year', 'qtr', 'avg_wkly_wage']}
    agg_dict['avg_wkly_wage'] = 'mean'  # Add the mean calculation separately

    df_ind = df_ind.groupby(['area_fips', 'industry_code', 'year', 'qtr'], as_index=False).agg(agg_dict)
    
    # Step 1: Get all unique combinations of area_fips, year, qtr
    unique_combinations = df_ind[['area_fips', 'year', 'qtr']].drop_duplicates()

    # Step 2: Get all unique industry_code values as a DataFrame
    all_industry_codes = pd.DataFrame({'industry_code': df_ind['industry_code'].unique()})

    # Step 3: Perform a cross join using merge with 'how=cross' (available in pandas 1.2+)
    complete_index = unique_combinations.merge(all_industry_codes, how='cross')

    # Step 4: Merge the complete index with the original data
    df_ind = complete_index.merge(df_ind, on=['area_fips', 'year', 'qtr', 'industry_code'], how='left')

    # Step 5: Replace NaNs in all other columns with zero
    cols_to_fill = df_ind.columns.difference(['area_fips', 'year', 'qtr', 'industry_code'])
    df_ind[cols_to_fill] = df_ind[cols_to_fill].fillna(0)

    # Sort the DataFrame by specified columns
    df_ind = df_ind.sort_values(by=['area_fips', 'year', 'industry_code', 'qtr']).reset_index(drop=True)

    # Separate national & non-national data
    df_nat = df_ind[df_ind['area_fips'] == 'US000']
    df_ind = df_ind[df_ind['area_fips'] != 'US000']

    # National employment calculations
    df_nat = df_nat[['industry_code', 'qtr', 'month1_emplvl']]
    df_total = df_nat.groupby('qtr', as_index=False)['month1_emplvl'].sum()
    df_total.rename(columns={'month1_emplvl': 'total_employment'}, inplace=True)
    df_total = df_total[['qtr', 'total_employment']]
    df_nat = df_nat.merge(df_total, on=['qtr'], how='left')
    df_nat['NatShare'] = (df_nat['month1_emplvl'] / df_nat['total_employment'])*100
    df_nat.rename(columns={'month1_emplvl': 'nationEmp'}, inplace=True)
    df_nat.drop(columns=['total_employment'], inplace=True, errors='ignore')

    #Local Shares
    df_total = df_ind.groupby(['qtr','area_fips'], as_index=False)['month1_emplvl'].sum()
    df_total.rename(columns={'month1_emplvl': 'totEmp'}, inplace=True)
    df_total = df_total[['area_fips', 'qtr', 'totEmp']]
    df_ind = df_ind.merge(df_total, on=['area_fips', 'qtr'], how='left')
    df_ind['LNAICSShare'] = (df_ind['month1_emplvl'] / df_ind['totEmp'])*100
    df_ind.drop(columns=['totEmp'], inplace=True)

    # Merge industry-level with national shares
    df_ind = pd.merge(df_ind, df_nat, on=['industry_code', 'qtr'], how='left')
    df_ind['excludedOcc'] = df_ind['nationEmp'] - df_ind['month1_emplvl']

    df_ind['NAICS6'] = pd.to_numeric(df_ind['industry_code'], errors='coerce').fillna(-999).astype(int)

    # Compute excluded employment totals per MSA
    df_excl_sum_naics = df_ind.groupby(['area_fips'], as_index=False)['excludedOcc'].sum().rename(columns={'excludedOcc': 'excludedTotal'})
    # Merge excluded employment totals
    df_ind = df_ind.merge(df_excl_sum_naics, on=['area_fips'], how='left')
    # Compute NAICS share
    df_ind['NAICSShare'] = (df_ind['excludedOcc'] / df_ind['excludedTotal'])*100

    # Drop duplicates
    df_ind.drop_duplicates(subset=['area_fips', 'NAICS6',  'qtr'], inplace=True)

    # Drop unnecessary columns
    df_ind.drop(columns=['nationEmp', 'excludedOcc', 'excludedTotal', 'industry_code', 'own_code'], inplace=True, errors='ignore')

    # Save to Parquet
    output_file = os.path.join(output_dir, f"{year}.parquet")
    df_ind.to_parquet(output_file, compression="zstd", engine="pyarrow", index=False)

    print(f"✅ Saved {output_file}")

print("🎉 All years processed successfully!")


🔄 Processing 2004...
✅ Saved /Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/Industry Employment/Processed/NAICS6/2004.parquet
🔄 Processing 2005...
✅ Saved /Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/Industry Employment/Processed/NAICS6/2005.parquet
🔄 Processing 2006...


/opt/anaconda3/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3331: DtypeWarning: Columns (7,16,25) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)


✅ Saved /Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/Industry Employment/Processed/NAICS6/2006.parquet
🔄 Processing 2007...


/opt/anaconda3/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3331: DtypeWarning: Columns (7,16) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)


✅ Saved /Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/Industry Employment/Processed/NAICS6/2007.parquet
🔄 Processing 2008...
✅ Saved /Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/Industry Employment/Processed/NAICS6/2008.parquet
🔄 Processing 2009...
✅ Saved /Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/Industry Employment/Processed/NAICS6/2009.parquet
🔄 Processing 2010...
✅ Saved /Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/Industry Employment/Processed/NAICS6/2010.parquet
🔄 Processing 2011...
✅ Saved /Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/Industry Employment/Processed/NAICS6/2011.parquet
🔄 Processing 2012...
✅ Saved /Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/Industry Employment/Processed/NAICS6/2012.parquet
🔄 Processing 2013...
✅ Saved /Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/Industry Employment/Processed/NAICS6/2013.

In [340]:
# Define the directory containing the parquet files
input_dir = "/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/Industry Employment/Processed/"  # 🔄 Update with your actual path

# Define the years to process
years = range(2004, 2010)  # 2004-2009

# Define the columns to keep
columns_to_keep = ['area_fips', 'qtr', 'year', 'LNAICSShare', 'NAICS3', 'NatShare']

# Initialize an empty DataFrame for incremental merging
df_merged = None

# Loop through each year and process files one by one
for year in years:
    file_path = os.path.join(input_dir, f"{year}.parquet")

    if not os.path.exists(file_path):
        print(f"⚠️ File not found: {file_path}, skipping...")
        continue

    print(f"🔄 Processing {year}...")

    # Read only the necessary columns
    df_temp = pd.read_parquet(file_path, columns=columns_to_keep)

    # Rename the share columns to include the year suffix
    year_suffix = str(year)[-2:]  # Get last two digits of the year
    df_temp.rename(columns={
        'LNAICSShare': f"{year_suffix}LNAICSShare",
        'NatShare': f"{year_suffix}NatShare"
    }, inplace=True)

    # Drop the 'year' column as it's redundant after renaming
    df_temp.drop(columns=['year'], inplace=True)

    # Merge with existing dataframe
    if df_merged is None:
        df_merged = df_temp  # First file, initialize df_merged
    else:
        df_merged = df_merged.merge(df_temp, on=['area_fips', 'qtr', 'NAICS3'], how='outer')

    # Free memory
    del df_temp
    print(f"✅ Merged {year} into df_merged")




🔄 Processing 2004...
✅ Merged 2004 into df_merged
🔄 Processing 2005...
✅ Merged 2005 into df_merged
🔄 Processing 2006...
✅ Merged 2006 into df_merged
🔄 Processing 2007...
✅ Merged 2007 into df_merged
🔄 Processing 2008...
✅ Merged 2008 into df_merged
🔄 Processing 2009...
✅ Merged 2009 into df_merged


In [341]:
df_merged.NAICS3.unique()

array(['111', '112', '113', '114', '115', '211', '212', '213', '221',
       '236', '237', '238', '311', '312', '313', '314', '315', '316',
       '321', '322', '323', '324', '325', '326', '327', '331', '332',
       '333', '334', '335', '336', '337', '339', '423', '424', '425',
       '441', '442', '443', '444', '445', '446', '447', '448', '449',
       '451', '452', '453', '454', '455', '456', '457', '458', '459',
       '481', '482', '483', '484', '485', '486', '487', '488', '491',
       '492', '493', '511', '512', '513', '515', '516', '517', '518',
       '519', '521', '522', '523', '524', '525', '531', '532', '533',
       '541', '551', '561', '562', '611', '621', '622', '623', '624',
       '711', '712', '713', '721', '722', '811', '812', '813', '814',
       '921', '922', '923', '924', '925', '926', '927', '928'],
      dtype=object)

In [342]:
df_merged.head()

,area_fips,qtr,04LNAICSShare,NAICS3,04NatShare,05LNAICSShare,05NatShare,06LNAICSShare,06NatShare,07LNAICSShare,07NatShare,08LNAICSShare,08NatShare,09LNAICSShare,09NatShare
0,01000,1,0.178312,111,0.337142,0.182754,0.328630,0.171571,0.322467,0.150298,0.319748,0.133495,0.314700,0.122538,0.324780
1,01000,2,0.230538,111,0.440721,0.221284,0.422697,0.209612,0.393938,0.174099,0.398199,0.161771,0.390873,0.156348,0.400576
2,01000,3,0.220608,111,0.505813,0.214846,0.492440,0.198711,0.480268,0.181256,0.468994,0.175825,0.468330,0.167295,0.504792
3,01000,4,0.209209,111,0.467799,0.197638,0.457059,0.187316,0.446733,0.167953,0.436790,0.158716,0.450568,0.144894,0.468638
4,01001,1,0.000000,111,0.337142,0.000000,0.328630,0.000000,0.322467,0.000000,0.319748,0.000000,0.314700,0.000000,0.324780


In [353]:
# Compute φ (industry employment share)
df_merged['phi_mk'] = df_merged[['04LNAICSShare', '05LNAICSShare']].mean(axis=1)
df_merged['0907LShare'] = df_merged[['07LNAICSShare', '09LNAICSShare']].mean(axis=1)

In [344]:
# Step 1: Calculate log differences for national employment shares
df_merged['log_diff_2009'] = np.log(df_merged['09NatShare']) - np.log(df_merged['08NatShare'])
df_merged['log_diff_2006'] = np.log(df_merged['06NatShare']) - np.log(df_merged['05NatShare'])
# Step 2: Calculate projected employment growth for 2009 and 2006
df_merged['pgrowth09'] = df_merged['phi_mk'] * df_merged['log_diff_2009']
df_merged['pgrowth06'] = df_merged['phi_mk'] * df_merged['log_diff_2006']


In [345]:
# Compute the Bartik shock for each (MSA, Industry)
df_merged['BShareimq'] = df_merged['pgrowth09'] - df_merged['pgrowth06']


In [354]:
df_merged.head()

,area_fips,qtr,04LNAICSShare,NAICS3,04NatShare,05LNAICSShare,05NatShare,06LNAICSShare,06NatShare,07LNAICSShare,07NatShare,08LNAICSShare,08NatShare,09LNAICSShare,09NatShare,phi_mk,log_diff_2009,log_diff_2006,pgrowth09,pgrowth06,BShareimq,0907LShare
0,01000,1,0.178312,111,0.337142,0.182754,0.328630,0.171571,0.322467,0.150298,0.319748,0.133495,0.314700,0.122538,0.324780,0.180533,0.031527,-0.018930,0.005692,-0.003417,0.009109,0.136418
1,01000,2,0.230538,111,0.440721,0.221284,0.422697,0.209612,0.393938,0.174099,0.398199,0.161771,0.390873,0.156348,0.400576,0.225911,0.024520,-0.070464,0.005539,-0.015919,0.021458,0.165223
2,01000,3,0.220608,111,0.505813,0.214846,0.492440,0.198711,0.480268,0.181256,0.468994,0.175825,0.468330,0.167295,0.504792,0.217727,0.074975,-0.025029,0.016324,-0.005449,0.021773,0.174276
3,01000,4,0.209209,111,0.467799,0.197638,0.457059,0.187316,0.446733,0.167953,0.436790,0.158716,0.450568,0.144894,0.468638,0.203424,0.039323,-0.022851,0.007999,-0.004648,0.012648,0.156424
4,01001,1,0.000000,111,0.337142,0.000000,0.328630,0.000000,0.322467,0.000000,0.319748,0.000000,0.314700,0.000000,0.324780,0.000000,0.031527,-0.018930,0.000000,-0.000000,0.000000,0.000000


In [ ]:
df_aux=df_merged

In [355]:
# Sum these projected growths by MSA to get total projected growth
df_msa_growth = df_merged.groupby(['area_fips', 'qtr']).agg({
    'pgrowth09': 'sum',
    'pgrowth06': 'sum'
}).reset_index()

# Calculate the Bartik shock
df_msa_growth['BShockHK'] = df_msa_growth['pgrowth09'] - df_msa_growth['pgrowth06']



In [356]:
df_final=pd.merge(df_merged, df_msa_growth.drop(columns=['pgrowth09','pgrowth06'], errors='ignore'), on=['area_fips', 'qtr'], how='left')

In [360]:
df_final=df_final[['area_fips', 'qtr', 'NAICS3','07LNAICSShare','0907LShare','09LNAICSShare','BShareimq', 'BShockHK']]

In [362]:
# Save final merged dataframe
output_file = os.path.join(input_dir, "SharesNAICS.parquet")
df_final.to_parquet(output_file, compression="zstd", engine="pyarrow", index=False)

print(f"✅ Final merged data saved to {output_file}")

✅ Final merged data saved to /Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/Industry Employment/Processed/SharesNAICS.parquet


In [367]:
directory='/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/Industry Employment/Processed/'
# Define years to process
years = list(range(2009, 2022))  # From 2009 to 2021

# Load files only if they exist
file_paths = {year: os.path.join(directory, f"{year}.parquet") for year in years}
df_years = {year: pd.read_parquet(file_paths[year]) for year in years if os.path.exists(file_paths[year])}

# Define the column for employment share
employment_col = "NAICSShare"  # Column for Leave-One-Out (LOO) national employment

# Compute Shift_k,t (log difference of national employment)
for year in years[1:]:  # Start from 2010 to compare with 2009
    prev_year = year - 1

    if prev_year in df_years and year in df_years:
        print(f"🔄 Processing {year}...")

        # Merge current year with previous year data
        df_merged = pd.merge(
            df_years[year],
            df_years[prev_year][['area_fips', 'qtr', 'NAICS3', employment_col]],
            on=['area_fips', 'qtr', 'NAICS3'],
            how='left',
            suffixes=('', '_prev')
        )

        # Prevent log(0) error by replacing zeros with a small value
        df_merged[employment_col] = df_merged[employment_col].replace(0, np.finfo(float).eps)
        df_merged[f"{employment_col}_prev"] = df_merged[f"{employment_col}_prev"].replace(0, np.finfo(float).eps)

        # Compute the Shift component: log difference
        df_merged["shiftNAICS"] = np.log(df_merged[employment_col]) - np.log(df_merged[f"{employment_col}_prev"])

        # Drop auxiliary columns safely
        df_merged.drop(columns=[f"{employment_col}_prev", 'own_code'], errors='ignore', inplace=True)

        # Merge with df_final, ensuring correct merging keys
        df_merged['area_fips'] = df_merged['area_fips'].astype(str).str.zfill(5)
        df_merged = pd.merge(
            df_merged,
            df_final.drop_duplicates(subset=['area_fips', 'qtr', 'NAICS3'], keep='first'),
            on=['area_fips', 'qtr', 'NAICS3'],
            how='left'
        )

        # Extract FIPS, FIPSState, FIPSCounty
        df_merged['FIPS'] = df_merged['area_fips']

        # Check if the first two characters are numeric
        is_numeric = df_merged['FIPS'].str[:2].str.isnumeric()

        # Assign values conditionally
        df_merged.loc[is_numeric, 'FIPSState'] = df_merged.loc[is_numeric, 'FIPS'].str[:2].astype(int)
        df_merged.loc[is_numeric, 'FIPSCounty'] = df_merged.loc[is_numeric, 'FIPS'].str[-3:].astype(int)
        df_merged.loc[~is_numeric, ['FIPSState', 'FIPSCounty', 'FIPS']] = -999

        # Replace FIPSCounty = 0 with -999
        df_merged.loc[df_merged['FIPSCounty'] == 0, 'FIPSCounty'] = -999

        # Convert to integer safely
        df_merged[['FIPS', 'FIPSState', 'FIPSCounty']] = df_merged[['FIPS', 'FIPSState', 'FIPSCounty']].astype(int, errors='ignore')

        # Drop rows where area_fips starts with "CS"
        df_merged = df_merged[~df_merged['area_fips'].astype(str).str.startswith('CS')]

        # Assign BestFitMSA4
        mask = df_merged['area_fips'].astype(str).str.match(r'^C\d{4}$')
        df_merged.loc[mask, 'BestFitMSA4'] = df_merged.loc[mask, 'area_fips'].str[1:].astype(int)
        df_merged.loc[~mask, 'BestFitMSA4'] = -999

        # Ensure BestFitMSA4 is an integer
        df_merged['BestFitMSA4'] = df_merged['BestFitMSA4'].astype(int)

        # Rename columns
        df_merged.rename(columns={'year': 'Year', 'avg_wkly_wage': 'wkWageNAICS'}, inplace=True)

        # Keep relevant columns
        columns_to_keep = [
            'Year', 'qtr', 'wkWageNAICS', 'NAICS3', 'shiftNAICS',
            '07LNAICSShare', '09LNAICSShare','0907LShare', 'BShareimq', 'BShockHK', 'FIPS', 'FIPSState',
            'FIPSCounty', 'BestFitMSA4'
        ]
        df_merged = df_merged[[col for col in columns_to_keep if col in df_merged.columns]]
        df_merged = df_merged[~((df_merged['FIPSCounty'] == -999) & (df_merged['BestFitMSA4'] == -999))]
        df_merged['bartik07']=df_merged['shiftNAICS']*df_merged['07LNAICSShare']
        df_merged['bartik0709']=df_merged['shiftNAICS']*df_merged['0907LShare']
        df_merged['bartik09']=df_merged['shiftNAICS']*df_merged['09LNAICSShare']
        # Save updated file
        output_file = os.path.join(directory,'With Bartik', f"{year}.parquet")
        df_merged.to_parquet(output_file, compression='zstd', engine='pyarrow', index=False)

        print(f"✅ Computed and saved Shift_k_t for {year}")

print("🎉 All years processed successfully!")

🔄 Processing 2010...
✅ Computed and saved Shift_k_t for 2010
🔄 Processing 2011...
✅ Computed and saved Shift_k_t for 2011
🔄 Processing 2012...
✅ Computed and saved Shift_k_t for 2012
🔄 Processing 2013...
✅ Computed and saved Shift_k_t for 2013
🔄 Processing 2014...
✅ Computed and saved Shift_k_t for 2014
🔄 Processing 2015...
✅ Computed and saved Shift_k_t for 2015
🔄 Processing 2016...
✅ Computed and saved Shift_k_t for 2016
🔄 Processing 2017...
✅ Computed and saved Shift_k_t for 2017
🔄 Processing 2018...
✅ Computed and saved Shift_k_t for 2018
🔄 Processing 2019...
✅ Computed and saved Shift_k_t for 2019
🔄 Processing 2020...
✅ Computed and saved Shift_k_t for 2020
🔄 Processing 2021...
✅ Computed and saved Shift_k_t for 2021
🎉 All years processed successfully!


In [366]:
df_merged.head()

,Year,qtr,wkWageNAICS,NAICS3,shiftNAICS,07LNAICSShare,09LNAICSShare,0907LShare,BShareimq,BShockHK,FIPS,FIPSState,FIPSCounty,BestFitMSA4,bartik
4,2021,1,NaN,111,-0.034600,0.0,0.0,0.0,0.0,-1.106444,1001,1,1,-999,-0.0
5,2021,2,NaN,111,0.014147,0.0,0.0,0.0,0.0,-0.372860,1001,1,1,-999,0.0
6,2021,3,NaN,111,-0.022389,0.0,0.0,0.0,0.0,0.072970,1001,1,1,-999,-0.0
7,2021,4,NaN,111,-0.020221,0.0,0.0,0.0,0.0,-0.185531,1001,1,1,-999,-0.0
8,2021,1,576.5,111,-0.034595,0.0,0.0,0.0,0.0,-1.869909,1003,1,3,-999,-0.0


### Crosswalk + turn NAICS6 to NAICS3

In [95]:
df_naics=pd.read_excel('/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/Crosswalks/2022_to_2017_NAICS.xlsx', skiprows=2) 
df_naics = df_naics.loc[:, ~df_naics.columns.str.startswith('Unnamed')]


In [96]:
df_naics['2022NAICS3'] = df_naics['2022 NAICS Code'].astype(str).str[:3]
df_naics['2017NAICS3'] = df_naics['2017 NAICS Code'].astype(str).str[:3]

In [98]:
df_naics['NAICS3_change'] = np.where(df_naics['2022NAICS3'] == df_naics['2017NAICS3'], 'No change', 'Changed')


In [105]:
naics_change= df_naics[df_naics.NAICS3_change=='Changed']
naics_change['2022NAICS3'].unique()

array(['441', '444', '445', '449', '455', '456', '457', '458', '459',
       '513', '516'], dtype=object)

In [106]:
naics_change['2017NAICS3'].unique()

array(['454', '442', '443', '452', '453', '446', '447', '448', '451',
       '511', '519', '515'], dtype=object)

In [107]:
unique2017=df_naics['2017NAICS3'].unique()
unique2022=df_naics['2022NAICS3'].unique()


In [110]:
# Convert unique values to sets
set_22 = set(unique2022)
set_17 = set(unique2017)

# Find values only in NAICS22 but not in NAICS17
only22 = set_22 - set_17



In [114]:
# Filter rows where '2022NAICS3' is in only22
df_new = df_naics[df_naics['2022NAICS3'].isin(only22)]


In [117]:
df_new.rename(columns={'2022NAICS3':'NAICS3'}, inplace=True, errors='ignore')
df_naics.rename(columns={'2017NAICS3':'NAICS3'}, inplace=True, errors='ignore')
df_naics=pd.concat([df_naics, df_new], ignore_index=True)

In [119]:
df_naics.rename(columns={'2022 NAICS Code':'2022NAICS', '2017 NAICS Code':'2017NAICS' }, inplace=True, errors='ignore')
df_naics=df_naics[['NAICS3', '2017NAICS','2022NAICS']]

In [121]:
df_naics=df_naics.drop_duplicates()

In [122]:
df_naics.head()

,NAICS3,2017NAICS,2022NAICS
0,111,111110,111110
1,111,111120,111120
2,111,111130,111130
3,111,111140,111140
4,111,111150,111150


In [124]:
df_naics2=pd.read_excel('/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/Crosswalks/2012_to_2017_NAICS.xlsx', skiprows=2)
df_naics2 = df_naics2.loc[:, ~df_naics2.columns.str.startswith('Unnamed')]


In [130]:
df_naics2['2012NAICS3'] = df_naics2['2012 NAICS Code'].astype(str).str[:3]
df_naics2['2017NAICS3'] = df_naics2['2017 NAICS Code'].astype(str).str[:3]

unique2017=df_naics2['2017NAICS3'].unique()
unique2012=df_naics2['2012NAICS3'].unique()

In [132]:
# Convert unique values to sets
set_12 = set(unique2012)
set_17 = set(unique2017)

# Find values only in NAICS22 but not in NAICS17
only12 = set_12 - set_17
# Filter rows where '2022NAICS3' is in only22
df_new = df_naics2[df_naics2['2012NAICS3'].isin(only12)]

In [135]:
df_naics2.rename(columns={'2017NAICS3':'NAICS3'}, inplace=True, errors='ignore')

In [136]:
df_naics2.rename(columns={'2012 NAICS Code':'2012NAICS', '2017 NAICS Code':'2017NAICS' }, inplace=True, errors='ignore')
df_naics2=df_naics2[['NAICS3', '2017NAICS','2012NAICS']]

In [145]:
df_naics=pd.merge(df_naics, df_naics2.drop(columns=['NAICS3']), how='outer', on=['2017NAICS'])

In [147]:
df_naics2=pd.read_excel('/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/Crosswalks/2007_to_2012_NAICS.xls', skiprows=2)
df_naics2 = df_naics2.loc[:, ~df_naics2.columns.str.startswith('Unnamed')]


In [148]:
df_naics2['2012NAICS3'] = df_naics2['2012 NAICS Code'].astype(str).str[:3]
df_naics2['2007NAICS3'] = df_naics2['2007 NAICS Code'].astype(str).str[:3]

unique2007=df_naics2['2007NAICS3'].unique()
unique2012=df_naics2['2012NAICS3'].unique()

In [155]:
# Convert unique values to sets
set_12 = set(unique2012)
set_07 = set(unique2007)

# Find values only in NAICS22 but not in NAICS17
only07 = set_07 - set_12
# Filter rows where '2022NAICS3' is in only22
df_new = df_naics2[df_naics2['2007NAICS3'].isin(only07)]

In [157]:
df_naics2.rename(columns={'2012 NAICS Code':'2012NAICS', '2007 NAICS Code':'2007NAICS' }, inplace=True, errors='ignore')
df_naics2=df_naics2[['2007NAICS','2012NAICS']]

In [162]:
df_naics=pd.merge(df_naics, df_naics2, how='outer', on=['2012NAICS'])

In [161]:
df_test['_merge'].value_counts()

both          1551
left_only        0
right_only       0
Name: _merge, dtype: int64

In [179]:
df_naics2=pd.read_excel('/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/Crosswalks/2002_to_2007_NAICS.xls', skiprows=2)
df_naics2 = df_naics2.loc[:, ~df_naics2.columns.str.startswith('Unnamed')]

In [180]:
df_naics2['2002NAICS3'] = df_naics2['2002 NAICS Code'].astype(str).str[:3]
df_naics2['2007NAICS3'] = df_naics2['2007 NAICS Code'].astype(str).str[:3]

unique2007=df_naics2['2007NAICS3'].unique()
unique2002=df_naics2['2002NAICS3'].unique()

# Convert unique values to sets
set_02 = set(unique2002)
set_07 = set(unique2007)

# Find values only in NAICS22 but not in NAICS17
only02 = set_02 - set_07
# Filter rows where '2022NAICS3' is in only22
df_new = df_naics2[df_naics2['2002NAICS3'].isin(only02)]

In [170]:
df_naics2.rename(columns={'2002 NAICS Code':'2002NAICS', '2007 NAICS Code':'2007NAICS' }, inplace=True, errors='ignore')
df_naics2=df_naics2[['2007NAICS','2002NAICS']]

In [173]:
df_naics=pd.merge(df_naics, df_naics2, how='outer', on=['2007NAICS'])

In [172]:
df_test['_merge'].value_counts()

both          1591
left_only        0
right_only       0
Name: _merge, dtype: int64

In [190]:
df_naics['2007NAICS3'] = df_naics['2007NAICS'].astype(str).str[:3]
unique2002=df_naics['2002NAICS3'].unique()
unique2007=df_naics['2007NAICS3'].unique()

# Convert unique values to sets
set_02 = set(unique2002)
set_07 = set(unique2007)

In [191]:
# Find values only in NAICS22 but not in NAICS17
only02 = set_02 - set_07

In [22]:
df_naics=pd.read_parquet('/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/Industry Employment/Processed/cross2017base.parquet')


In [306]:
df=pd.read_parquet('/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/Industry Employment/Processed/NAICS6/2021.parquet')



In [307]:
df.head()

,area_fips,year,qtr,month1_emplvl,taxable_qtrly_wages,qtrly_contributions,avg_wkly_wage,LNAICSShare,NatShare,NAICS6,NAICSShare
0,01000,2021,1,8.0,55007.0,632.0,542.0,0.000451,0.003142,111110,0.000769
1,01000,2021,2,11.0,35984.0,538.0,843.0,0.000612,0.003472,111110,0.000872
2,01000,2021,3,13.0,10657.0,122.0,598.0,0.000714,0.003592,111110,0.000913
3,01000,2021,4,14.0,6013.0,70.0,748.0,0.000763,0.003868,111110,0.001007
4,01000,2021,1,0.0,0.0,0.0,0.0,0.000000,0.000168,111120,0.000041


In [308]:
df_merge=pd.merge(df_naics[['NAICS3', '2017NAICS']].drop_duplicates(), df, left_on='2017NAICS', right_on='NAICS6', how='right', indicator=True)

df_merge['_merge'].value_counts()

both          16375720
right_only      579540
left_only            0
Name: _merge, dtype: int64

In [309]:
no_merge=df_merge[df_merge['_merge']=='right_only']

In [310]:
no_merge.NAICS6.unique()

array([238111, 238112, 238121, 238122, 238131, 238132, 238141, 238142,
       238151, 238152, 238161, 238162, 238171, 238172, 238191, 238192,
       238211, 238212, 238221, 238222, 238291, 238292, 238311, 238312,
       238321, 238322, 238331, 238332, 238341, 238342, 238351, 238352,
       238391, 238392, 238911, 238912, 238991, 238992, 999999])

In [311]:
no_merge['NAICS5'] = (no_merge['NAICS6'].astype(str).str[:5] + '0').astype(int)
no_merge.drop(columns=['NAICS3','2017NAICS', '_merge'], errors='ignore', inplace=True)

df_merge2=pd.merge(df_naics[['NAICS3', '2017NAICS']].drop_duplicates(), no_merge, left_on='2017NAICS', right_on='NAICS5', how='right', indicator=True)
df_merge2['_merge'].value_counts()

/opt/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  """Entry point for launching an IPython kernel.
/opt/anaconda3/lib/python3.7/site-packages/pandas/core/frame.py:4913: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  errors=errors,


both          564680
right_only     14860
left_only          0
Name: _merge, dtype: int64

In [312]:
nomerge=df_merge2[df_merge2['_merge']=='right_only']
nomerge.NAICS6.unique()

array([999999])

In [313]:
df_merge=pd.concat([df_merge[df_merge['_merge']=='both'].drop(columns=['_merge','2017NAICS']), df_merge2.drop(columns=['_merge','2017NAICS', 'NAICS5'])], ignore_index=True)



In [314]:
len(df_merge)

16955260

In [315]:

# Replace 0 values in avg_wkly_wage with NaN before computing mean
df_merge['avg_wkly_wage'] = df_merge['avg_wkly_wage'].replace(0, np.nan)

# List of columns to sum (all except 'avg_wkly_wage')
cols_to_sum = [col for col in df_merge.columns if col not in ['avg_wkly_wage',  'area_fips', 'year', 'qtr', 'NAICS6']]

# Groupby and compute sum for selected columns
df_sum = df_merge.groupby(['NAICS3', 'area_fips', 'year', 'qtr'])[cols_to_sum].sum()

# Compute mean for avg_wkly_wage separately (ignoring NaNs)
df_mean = df_merge.groupby(['NAICS3', 'area_fips', 'year', 'qtr'])['avg_wkly_wage'].mean()

# Merge both aggregated results
df_grouped = df_sum.join(df_mean).reset_index()


In [316]:
df_grouped[0:10]

,NAICS3,area_fips,year,qtr,month1_emplvl,taxable_qtrly_wages,qtrly_contributions,LNAICSShare,NatShare,NAICSShare,avg_wkly_wage
0,111,01000,2021,1,2607.0,17456362.0,381798.0,0.147123,0.325765,0.079395,618.181818
1,111,01000,2021,2,2919.0,5821434.0,141982.0,0.162302,0.382875,0.095907,808.818182
2,111,01000,2021,3,2864.0,2376369.0,55405.0,0.157312,0.423718,0.107483,743.818182
3,111,01000,2021,4,2958.0,2610523.0,52363.0,0.161196,0.411379,0.106833,850.909091
4,111,01001,2021,1,0.0,0.0,0.0,0.000000,0.325765,0.078849,NaN
5,111,01001,2021,2,0.0,0.0,0.0,0.000000,0.382875,0.095206,NaN
6,111,01001,2021,3,0.0,0.0,0.0,0.000000,0.423718,0.106627,NaN
7,111,01001,2021,4,0.0,0.0,0.0,0.000000,0.411379,0.106001,NaN
8,111,01003,2021,1,33.0,180784.0,2088.0,0.066611,0.325765,0.078870,576.500000
9,111,01003,2021,2,19.0,40727.0,468.0,0.035918,0.382875,0.095236,623.000000


In [317]:
df_grouped.NAICS3.unique()

array(['111', '112', '113', '114', '115', '211', '212', '213', '221',
       '236', '237', '238', '311', '312', '313', '314', '315', '316',
       '321', '322', '323', '324', '325', '326', '327', '331', '332',
       '333', '334', '335', '336', '337', '339', '423', '424', '425',
       '441', '442', '443', '444', '445', '446', '447', '448', '449',
       '451', '452', '453', '454', '455', '456', '457', '458', '459',
       '481', '482', '483', '484', '485', '486', '487', '488', '491',
       '492', '493', '511', '512', '513', '515', '516', '517', '518',
       '519', '521', '522', '523', '524', '525', '531', '532', '533',
       '541', '551', '561', '562', '611', '621', '622', '623', '624',
       '711', '712', '713', '721', '722', '811', '812', '813', '814',
       '921', '922', '923', '924', '925', '926', '927', '928'],
      dtype=object)

In [318]:
len(df_grouped.NAICS3.unique())

107

In [319]:
df_grouped.to_parquet('/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/Industry Employment/Processed/2021.parquet',compression='zstd', engine='pyarrow', index=False)


In [320]:
# Group by area_fips and sum LNAICSShare
df_share_check = df.groupby(['area_fips', 'qtr'])['LNAICSShare'].sum().reset_index()

df_share_check[0:10]


,area_fips,qtr,LNAICSShare
0,01000,1,100.0
1,01000,2,100.0
2,01000,3,100.0
3,01000,4,100.0
4,01001,1,100.0
5,01001,2,100.0
6,01001,3,100.0
7,01001,4,100.0
8,01003,1,100.0
9,01003,2,100.0


In [329]:
df=pd.read_parquet('/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/Industry Employment/Processed/2019.parquet')


In [322]:
df2=pd.read_parquet('/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Other data/Industry Employment/Processed/2020.parquet')


In [330]:
unique2021=df['area_fips'].unique()
unique2020=df2['area_fips'].unique()

# Convert unique values to sets
set_02 = set(unique2021)
set_07 = set(unique2020)

In [335]:
only07 = set_02 - set_07
